# TFSC v21.1 — MAIN notebook

Use this notebook for ordinary development and for the three principal conditional transition scenarios. It contains the complete SD–ABM model, essential accounting checks, the paired Monte Carlo experiment, paper figures, and the two main outcome tables.

`QUICK_MODE = True` runs five paired draws for code checks. Change only that switch to `False` when intentionally generating the definitive 500-run principal analysis.

Version 21 fixes the causal and accounting boundaries of the transition model: technology-neutral closed-loop eligibility, independent new-blade demand, harmonized pyrolysis costs, symmetric gate-fee accounting, realized recycler profit, prospective first entry, and delivery-based supply reliability. The model remains a stylized conditional-transition experiment rather than an industrial point forecast.

In [ ]:
# ============================================================
# 1. Imports, simulation horizon, pathways, and scenarios
# ============================================================

from dataclasses import asdict, dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


MODEL_VERSION = "v21.1"
MODEL_REVISION = (
    "Market-offtake separation and parsimonious policy-channel "
    "specification built on the v21 causal boundary"
)


# ============================================================
# Simulation horizon
# ============================================================

START_YEAR = 2026
END_YEAR = 2050

YEARS = np.arange(START_YEAR, END_YEAR + 1, dtype=int)
N_YEARS = len(YEARS)


# ============================================================
# Active post-2025 wind turbine blade EoL pathways
# ============================================================

PATHWAYS = (
    "reuse",
    "repurposing",
    "mechanical_recycling",
    "pyrolysis",
    "recovery",
    "solvolysis",
)

# Pathways that retain the blade, blade component, or structure
# in a direct secondary-use application.
DIRECT_REUSE_PATHWAYS = (
    "reuse",
    "repurposing",
)

# Established material-recycling pathways that predominantly
# supply open-loop applications.
INCUMBENT_OPEN_LOOP_MATERIAL_PATHWAYS = (
    "mechanical_recycling",
    "pyrolysis",
)

# Energy recovery treats blade waste but does not generate
# recovered material for recirculation.
ENERGY_RECOVERY_PATHWAYS = (
    "recovery",
)

# Established treatment pathways competing with emerging solvolysis.
INCUMBENT_TREATMENT_PATHWAYS = (
    "mechanical_recycling",
    "pyrolysis",
    "recovery",
)

# Solvolysis produces high-quality recovered material with the
# technical potential for closed-loop use. Actual closed-loop
# utilization additionally depends on manufacturer adoption,
# demand formation, coordination, and material substitution.
HIGH_QUALITY_RECOVERY_PATHWAYS = (
    "solvolysis",
)


# ============================================================
# Scenario configuration
# ============================================================

@dataclass(frozen=True)
class ScenarioConfig:
    """
    Policy, technology, market, and coordination conditions defining
    one transition scenario.

    Parameters expressed as intensities are bounded between 0 and 1.
    """

    name: str
    description: str

    # Regulatory feasibility
    landfill_allowed: bool

    # Technology development
    solvolysis_trl_acceleration_per_year: float
    technology_support: float

    # Market formation
    demand_pull: float
    recycled_content_mandate: float
    closed_loop_demand_growth: float

    # Producer responsibility
    epr_strength: float

    # Behavioural and coordination effects
    coordination_strength: float

    def __post_init__(self) -> None:
        """Validate scenario parameters after initialization."""

        bounded_parameters = {
            "technology_support": self.technology_support,
            "demand_pull": self.demand_pull,
            "recycled_content_mandate": self.recycled_content_mandate,
            "epr_strength": self.epr_strength,
            "coordination_strength": self.coordination_strength,
        }

        for parameter_name, value in bounded_parameters.items():
            if not 0.0 <= value <= 1.0:
                raise ValueError(
                    f"{parameter_name} must lie between 0 and 1; "
                    f"received {value}."
                )

        non_negative_parameters = {
            "solvolysis_trl_acceleration_per_year":
                self.solvolysis_trl_acceleration_per_year,
            "closed_loop_demand_growth":
                self.closed_loop_demand_growth,
        }

        for parameter_name, value in non_negative_parameters.items():
            if value < 0.0:
                raise ValueError(
                    f"{parameter_name} cannot be negative; "
                    f"received {value}."
                )


# ============================================================
# Transition scenarios
# ============================================================

SCENARIOS = {
    "post_2025_baseline": ScenarioConfig(
        name="post_2025_baseline",
        description=(
            "Post-2025 reference scenario with landfill excluded, "
            "no additional technology or market-support instruments, "
            "weak autonomous growth in closed-loop demand, and "
            "reference solvolysis maturity."
        ),
        landfill_allowed=False,
        solvolysis_trl_acceleration_per_year=0.000,
        technology_support=0.00,
        demand_pull=0.00,
        recycled_content_mandate=0.00,
        closed_loop_demand_growth=0.01,
        epr_strength=0.00,
        coordination_strength=0.00,
    ),

    "demand_pull_circularity": ScenarioConfig(
        name="demand_pull_circularity",
        description=(
            "Predominantly demand-oriented transition scenario combining "
            "recovered-material demand incentives, a moderate recycled-content "
            "mandate and limited "
            "value-chain coordination. Direct technology support and EPR "
            "intervention are absent."
        ),
        landfill_allowed=False,
        solvolysis_trl_acceleration_per_year=0.000,
        technology_support=0.00,
        demand_pull=0.40,
        recycled_content_mandate=0.25,
        closed_loop_demand_growth=0.04,
        epr_strength=0.00,
        coordination_strength=0.10,
    ),

    "coordinated_transition": ScenarioConfig(
        name="coordinated_transition",
        description=(
            "Coordinated transition scenario combining solvolysis technology "
            "support, accelerated technology maturity, producer responsibility, "
            "recycled-content mandates, demand-side incentives, and strong "
            "value-chain coordination."
        ),
        landfill_allowed=False,
        solvolysis_trl_acceleration_per_year=0.043,
        technology_support=0.40,
        demand_pull=0.50,
        recycled_content_mandate=0.35,
        closed_loop_demand_growth=0.06,
        epr_strength=0.40,
        coordination_strength=0.60,
    ),
}


# ============================================================
# Scenario ordering and labels for results
# ============================================================

SCENARIO_ORDER = (
    "post_2025_baseline",
    "demand_pull_circularity",
    "coordinated_transition",
)

SCENARIO_LABELS = {
    "post_2025_baseline": "Post-2025 baseline",
    "demand_pull_circularity": "Demand-pull circularity",
    "coordinated_transition": "Coordinated transition",
}

SCENARIO_LABELS_SHORT = {
    "post_2025_baseline": "Baseline",
    "demand_pull_circularity": "Demand-pull",
    "coordinated_transition": "Coordinated",
}


# ============================================================
# Internal consistency checks
# ============================================================

if tuple(SCENARIOS.keys()) != SCENARIO_ORDER:
    raise ValueError(
        "The order of SCENARIOS must match SCENARIO_ORDER."
    )

for scenario_key, scenario_config in SCENARIOS.items():
    if scenario_key != scenario_config.name:
        raise ValueError(
            f"Scenario key '{scenario_key}' does not match "
            f"ScenarioConfig.name '{scenario_config.name}'."
        )


# ============================================================
# Scenario summary table
# ============================================================

scenario_table = pd.DataFrame(
    [
        {
            "scenario": scenario_key,
            "scenario_label": SCENARIO_LABELS[scenario_key],
            **asdict(SCENARIOS[scenario_key]),
        }
        for scenario_key in SCENARIO_ORDER
    ]
)

scenario_table


In [ ]:
# ============================================================
# 2. System, technology, and market parameters
# ============================================================

# WTB inflow, treatment, and capacity are expressed in normalized
# whole-blade mass units. Recovered-material generation, inventory,
# demand, and utilization are expressed in normalized secondary-material
# mass units. Explicit conversion factors connect the two ledgers.
#
# The model explores conditional transition pathways rather than
# providing an empirically calibrated forecast of absolute European
# WTB waste tonnage.


# ============================================================
# 2.1 WTB end-of-life inflow
# ============================================================

# Normalised quantity of WTB material reaching end of life in 2026.
BASE_DECOMMISSIONED_WTB = 100.0

# Annual growth rate of the WTB end-of-life inflow.
ANNUAL_EOL_GROWTH = 0.035


# ============================================================
# 2.2 Initial pathway allocation
# ============================================================

# Desired allocation of the 2026 WTB end-of-life inflow.
#
# These values describe the initial configuration of the system.
# They do not represent treatment capacities. Their influence on
# operator decisions is introduced later through a decaying
# pathway-inertia mechanism.
INITIAL_PATHWAY_SHARES = {
    "reuse": 0.01,
    "repurposing": 0.04,
    "mechanical_recycling": 0.38,
    "pyrolysis": 0.02,
    "recovery": 0.55,
    "solvolysis": 0.00,
}


# ============================================================
# 2.3 Pathway functional groups
# ============================================================

# Pathways that generate secondary-material stocks.
MATERIAL_RECOVERY_PATHWAYS = (
    "mechanical_recycling",
    "pyrolysis",
    "solvolysis",
)

# Lower-value resource recovery that does not create a conventional
# secondary-material inventory.
LOW_VALUE_RECOVERY_PATHWAYS = (
    "recovery",
)

# All pathways producing recovered material or an equivalent
# resource-recovery benefit.
RESOURCE_RECOVERY_PATHWAYS = (
    MATERIAL_RECOVERY_PATHWAYS
    + LOW_VALUE_RECOVERY_PATHWAYS
)


# ============================================================
# 2.4 Technical performance
# ============================================================

# Fraction of accepted whole-blade mass forming the pathway-specific
# recoverable feedstock basis. Mechanical recycling, pyrolysis, and energy
# recovery retain the previous whole-mass basis. For solvolysis, only the
# glass-fibre mass fraction is eligible for high-quality recovery.
#
# The solvolysis value is calibrated to a representative 48.7 m, 9 t blade
# containing 5,283 kg of glass fibre in a 9,000 kg blade, i.e.
# a 0.587 glass-fibre mass fraction (Sproul et al., 2023).
RECOVERABLE_FEEDSTOCK_FRACTION = {
    "mechanical_recycling": 1.00,
    "pyrolysis": 0.587,
    "recovery": 1.00,
    "solvolysis": 0.587,
}

# Fraction of the pathway-specific recoverable feedstock converted into
# usable secondary material or an equivalent resource-recovery output.
# Solvolysis glass-fibre recovery is fixed at 0.85 as a conservative
# modelling assumption. It is not sampled in Monte Carlo.
RECOVERY_EFFICIENCY = {
    "mechanical_recycling": 0.55,
    "pyrolysis": 0.65,
    "recovery": 0.30,
    "solvolysis": 0.85,
}

# Stylised recovered-material or resource-quality index in [0, 1].
MATERIAL_QUALITY = {
    "mechanical_recycling": 0.35,
    "pyrolysis": 0.55,
    "recovery": 0.20,
    "solvolysis": 0.85,
}

# Fraction of virgin-material demand displaced by one unit of
# recovered material or equivalent resource-recovery output.
SUBSTITUTION_FACTOR = {
    "mechanical_recycling": 0.30,
    "pyrolysis": 0.50,
    "recovery": 0.15,
    "solvolysis": 0.80,
}


# ============================================================
# 2.5 Processing costs and technological learning
# ============================================================

# Pyrolysis and solvolysis cost components in 2019 USD per tonne of accepted WTB material,
# adapted from Liu et al. (2022). Transport is represented separately in
# BASE_LOGISTICS_COST; the remaining components form the initial treatment
# cost used by the learning curve and operator decision rule.
PYROLYSIS_COST_COMPONENTS = {
    "disassembly": 25.9,
    "cutting": 38.2,
    "transport": 96.7,
    "recycling_process": 207.3,
}

PYROLYSIS_NON_LOGISTICS_TREATMENT_COST = (
    PYROLYSIS_COST_COMPONENTS["disassembly"]
    + PYROLYSIS_COST_COMPONENTS["cutting"]
    + PYROLYSIS_COST_COMPONENTS["recycling_process"]
)

PYROLYSIS_COMPONENT_SUM_COST = sum(
    PYROLYSIS_COST_COMPONENTS.values()
)

SOLVOLYSIS_COST_COMPONENTS = {
    "disassembly": 25.9,
    "cutting": 38.2,
    "transport": 96.7,
    "recycling_process": 682.6,
}

SOLVOLYSIS_NON_LOGISTICS_TREATMENT_COST = (
    SOLVOLYSIS_COST_COMPONENTS["disassembly"]
    + SOLVOLYSIS_COST_COMPONENTS["cutting"]
    + SOLVOLYSIS_COST_COMPONENTS["recycling_process"]
)

SOLVOLYSIS_COMPONENT_SUM_COST = sum(
    SOLVOLYSIS_COST_COMPONENTS.values()
)

# All pathway costs and recovered-material prices are interpreted on a
# common 2019 USD per treated WTB mass-unit basis. Non-solvolysis values
# remain stylised; solvolysis uses the empirical component mapping above.
# Rounded components sum to 843.4 2019 USD/t; Liu et al. report 843.3
# 2019 USD/t because the published components are rounded.
INITIAL_PROCESSING_COST = {
    "reuse": 250.0,
    "repurposing": 350.0,
    "mechanical_recycling": 500.0,
    "pyrolysis": PYROLYSIS_NON_LOGISTICS_TREATMENT_COST,
    "recovery": 400.0,
    "solvolysis": SOLVOLYSIS_NON_LOGISTICS_TREATMENT_COST,
}

# Reference cumulative throughput at which the initial processing
# cost is observed.
REFERENCE_CUMULATIVE_THROUGHPUT = {
    "mechanical_recycling": 100.0,
    "pyrolysis": 50.0,
    "recovery": 100.0,
    "solvolysis": 20.0,
}

# Learning-curve exponents governing cost reductions through
# cumulative processing experience.
LEARNING_EXPONENT = {
    "mechanical_recycling": 0.06,
    "pyrolysis": 0.08,
    "recovery": 0.03,
    "solvolysis": 0.14,
}

# Initial cumulative processing experience.
#
# The positive value assigned to solvolysis represents pilot-scale
# and demonstration experience rather than commercial capacity.
INITIAL_CUMULATIVE_THROUGHPUT = {
    "mechanical_recycling": 100.0,
    "pyrolysis": 50.0,
    "recovery": 100.0,
    "solvolysis": 20.0,
}


# ============================================================
# 2.6 Recovered-material prices
# ============================================================

# Stylised market value of recovered material or equivalent
# resource-recovery output.
RECOVERED_MATERIAL_PRICE = {
    "mechanical_recycling": 300.0,
    "pyrolysis": 520.0,
    "recovery": 180.0,
    "solvolysis": 950.0,
}

# Reference price of virgin material displaced by recovered output.
VIRGIN_MATERIAL_PRICE = 900.0


# ============================================================
# 2.7 Exploratory environmental proxy (not reported)
# ============================================================

# Exploratory normalised environmental-intensity proxy per unit of
# recovered output. This is retained only as an internal diagnostic;
# it is not a physical GHG inventory and is not reported or exported.
ENVIRONMENTAL_INTENSITY_RECOVERED = {
    "mechanical_recycling": 0.55,
    "pyrolysis": 0.75,
    "recovery": 0.40,
    "solvolysis": 0.65,
}

# Exploratory reference intensity for the displaced virgin-material
# alternative; internal diagnostic only.
ENVIRONMENTAL_INTENSITY_VIRGIN = {
    pathway: 1.80
    for pathway in RESOURCE_RECOVERY_PATHWAYS
}


# ============================================================
# 2.8 Solvolysis technological maturity
# ============================================================

# Initial technology-readiness level.
INITIAL_TRL_SOLVOLYSIS = 5.0

# Minimum TRL required for commercial deployment.
TRL_THRESHOLD_SOLVOLYSIS = 7.0

# Commercialisation year under baseline maturity progression.
REFERENCE_SOLVOLYSIS_COMMERCIAL_YEAR = 2037

# Baseline annual TRL growth.
#
# The first technology-maturity update occurs at the end of 2026,
# moving the technology state towards 2027.
BASE_TRL_GROWTH_SOLVOLYSIS = (
    TRL_THRESHOLD_SOLVOLYSIS - INITIAL_TRL_SOLVOLYSIS
) / (
    REFERENCE_SOLVOLYSIS_COMMERCIAL_YEAR - START_YEAR
)


# ============================================================
# 2.9 Downstream material demand
# ============================================================

# Initial demand for recovered material in open-loop applications.
BASE_OPEN_LOOP_DEMAND = 80.0

# Initial latent or committed demand for high-quality recovered
# material in closed-loop applications.
#
# This is equivalent to 12.5% of initial open-loop demand.
BASE_CLOSED_LOOP_DEMAND = 10.0

# Independent normalized glass-fibre demand associated with new-blade
# manufacturing. This demand driver is deliberately separate from
# the quantity of blades reaching end of life.
BASE_NEW_BLADE_GLASS_FIBRE_DEMAND = 60.0
NEW_BLADE_GLASS_FIBRE_DEMAND_GROWTH = 0.025

# Common closed-loop eligibility rule. A recovered output must pass
# both the material-quality and certification thresholds; technology
# identity alone never determines closed-loop use.
CLOSED_LOOP_MIN_MATERIAL_QUALITY = 0.75
CLOSED_LOOP_MIN_CERTIFICATION_SCORE = 0.50

# Autonomous annual growth of open-loop demand.
OPEN_LOOP_DEMAND_GROWTH = 0.02

# Closed-loop demand growth is not defined here because it is
# scenario-specific through:
#
# scenario.closed_loop_demand_growth


# ============================================================
# 2.10 Initial treatment capacity
# ============================================================

# Initial annual treatment capacity by pathway.
INITIAL_CAPACITY = {
    "reuse": 5.0,
    "repurposing": 10.0,
    "mechanical_recycling": 60.0,
    "pyrolysis": 10.0,
    "recovery": 80.0,
    "solvolysis": 0.0,
}

# Maximum permitted annual capacity-expansion rates.
#
# Reuse and repurposing capacity are exogenous in this model and
# therefore have zero endogenous expansion. Recycler-controlled
# pathways expand only through the ABM investment mechanism.
#
# These values impose upper bounds only. Actual expansion decisions
# are determined later by realised utilisation, profitability,
# market absorption, investment decisions, and technological
# feasibility.
MAX_ANNUAL_CAPACITY_EXPANSION = {
    "reuse": 0.00,
    "repurposing": 0.00,
    "mechanical_recycling": 0.05,
    "pyrolysis": 0.08,
    "recovery": 0.03,
    "solvolysis": 0.20,
}


# ============================================================
# 2.11 System-parameter consistency checks
# ============================================================

# Initial allocation
assert set(INITIAL_PATHWAY_SHARES) == set(PATHWAYS)

assert all(
    0.0 <= value <= 1.0
    for value in INITIAL_PATHWAY_SHARES.values()
)

assert np.isclose(
    sum(INITIAL_PATHWAY_SHARES.values()),
    1.0,
)

assert np.isclose(PYROLYSIS_NON_LOGISTICS_TREATMENT_COST, 271.4)
assert np.isclose(PYROLYSIS_COMPONENT_SUM_COST, 368.1)
assert np.isclose(SOLVOLYSIS_COMPONENT_SUM_COST, 843.4)

# Pathway-level system parameters
assert set(INITIAL_PROCESSING_COST) == set(PATHWAYS)
assert set(INITIAL_CAPACITY) == set(PATHWAYS)
assert set(MAX_ANNUAL_CAPACITY_EXPANSION) == set(PATHWAYS)

# Resource-recovery parameters
assert set(RECOVERABLE_FEEDSTOCK_FRACTION) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(RECOVERY_EFFICIENCY) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(MATERIAL_QUALITY) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(SUBSTITUTION_FACTOR) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(REFERENCE_CUMULATIVE_THROUGHPUT) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(LEARNING_EXPONENT) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(INITIAL_CUMULATIVE_THROUGHPUT) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(RECOVERED_MATERIAL_PRICE) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(ENVIRONMENTAL_INTENSITY_RECOVERED) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(ENVIRONMENTAL_INTENSITY_VIRGIN) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

# Technical performance bounds
assert all(
    0.0 <= value <= 1.0
    for value in RECOVERABLE_FEEDSTOCK_FRACTION.values()
)

assert all(
    0.0 <= value <= 1.0
    for value in RECOVERY_EFFICIENCY.values()
)

assert np.isclose(
    RECOVERABLE_FEEDSTOCK_FRACTION["solvolysis"],
    0.587,
)

assert np.isclose(
    RECOVERY_EFFICIENCY["solvolysis"],
    0.85,
)

assert np.isclose(
    (
        RECOVERABLE_FEEDSTOCK_FRACTION["solvolysis"]
        * RECOVERY_EFFICIENCY["solvolysis"]
    ),
    0.49895,
)

assert np.isclose(
    SOLVOLYSIS_COMPONENT_SUM_COST,
    843.4,
)

assert all(
    0.0 <= value <= 1.0
    for value in MATERIAL_QUALITY.values()
)

assert all(
    0.0 <= value <= 1.0
    for value in SUBSTITUTION_FACTOR.values()
)

# Non-negative economic and capacity parameters
assert all(
    value >= 0.0
    for value in INITIAL_PROCESSING_COST.values()
)

assert all(
    value >= 0.0
    for value in REFERENCE_CUMULATIVE_THROUGHPUT.values()
)

assert all(
    value >= 0.0
    for value in INITIAL_CUMULATIVE_THROUGHPUT.values()
)

assert all(
    value >= 0.0
    for value in LEARNING_EXPONENT.values()
)

assert all(
    value >= 0.0
    for value in RECOVERED_MATERIAL_PRICE.values()
)

assert VIRGIN_MATERIAL_PRICE >= 0.0

assert all(
    value >= 0.0
    for value in INITIAL_CAPACITY.values()
)

assert all(
    0.0 <= value <= 1.0
    for value in MAX_ANNUAL_CAPACITY_EXPANSION.values()
)

# End-of-life inflow and demand
assert BASE_DECOMMISSIONED_WTB > 0.0
assert ANNUAL_EOL_GROWTH >= 0.0

assert BASE_OPEN_LOOP_DEMAND >= 0.0
assert BASE_CLOSED_LOOP_DEMAND >= 0.0
assert OPEN_LOOP_DEMAND_GROWTH >= 0.0

# Solvolysis maturity
assert INITIAL_TRL_SOLVOLYSIS >= 1.0
assert TRL_THRESHOLD_SOLVOLYSIS <= 9.0
assert INITIAL_TRL_SOLVOLYSIS < TRL_THRESHOLD_SOLVOLYSIS

assert (
    REFERENCE_SOLVOLYSIS_COMMERCIAL_YEAR
    > START_YEAR
)

assert BASE_TRL_GROWTH_SOLVOLYSIS > 0.0


# ============================================================
# 2.12 Parameter inspection tables
# ============================================================

pathway_parameter_table = pd.DataFrame(
    {
        "pathway": PATHWAYS,
        "initial_share": [
            INITIAL_PATHWAY_SHARES[pathway]
            for pathway in PATHWAYS
        ],
        "initial_processing_cost": [
            INITIAL_PROCESSING_COST[pathway]
            for pathway in PATHWAYS
        ],
        "initial_capacity": [
            INITIAL_CAPACITY[pathway]
            for pathway in PATHWAYS
        ],
        "maximum_capacity_expansion": [
            MAX_ANNUAL_CAPACITY_EXPANSION[pathway]
            for pathway in PATHWAYS
        ],
    }
)

resource_recovery_parameter_table = pd.DataFrame(
    {
        "pathway": RESOURCE_RECOVERY_PATHWAYS,
        "recoverable_feedstock_fraction": [
            RECOVERABLE_FEEDSTOCK_FRACTION[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "recovery_efficiency": [
            RECOVERY_EFFICIENCY[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "whole_wtb_output_yield": [
            (
                RECOVERABLE_FEEDSTOCK_FRACTION[pathway]
                * RECOVERY_EFFICIENCY[pathway]
            )
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "material_quality": [
            MATERIAL_QUALITY[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "substitution_factor": [
            SUBSTITUTION_FACTOR[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "recovered_material_price": [
            RECOVERED_MATERIAL_PRICE[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "learning_exponent": [
            LEARNING_EXPONENT[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "reference_cumulative_throughput": [
            REFERENCE_CUMULATIVE_THROUGHPUT[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "initial_cumulative_throughput": [
            INITIAL_CUMULATIVE_THROUGHPUT[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
    }
)

display(pathway_parameter_table)
display(resource_recovery_parameter_table)

In [ ]:
# ============================================================
# 3. Behavioural, economic, and transition-mechanism parameters
# ============================================================

# This cell defines agent behaviour, recycler economics,
# market-formation mechanisms, investment rules, pathway switching,
# capacity adjustment, material rerouting, and numerical tolerances.
#
# Physical, technological, and baseline market parameters are defined
# separately in Cell 2.


# ============================================================
# 3.1 Operator behaviour
# ============================================================

# Mean relative importance assigned by wind-farm operators to
# treatment cost and an internal environmental diagnostic.
MEAN_OPERATOR_COST_WEIGHT = 0.65
MEAN_OPERATOR_ENV_WEIGHT = 0.35

# Sensitivity of operator pathway choice to differences in
# pathway attractiveness.
MEAN_OPERATOR_CHOICE_SENSITIVITY = 2.50

# Stylised environmental-performance score used in operator
# pathway-selection decisions.
PATHWAY_ENVIRONMENTAL_SCORE = {
    "reuse": 0.90,
    "repurposing": 0.75,
    "mechanical_recycling": 0.45,
    "pyrolysis": 0.55,
    "recovery": 0.25,
    "solvolysis": 0.85,
}


# ============================================================
# 3.2 Recycler operating economics
# ============================================================

# Gate fee received per unit of accepted WTB material.
BASE_GATE_FEE = {
    "mechanical_recycling": 620.0,
    "pyrolysis": 760.0,
    "recovery": 520.0,
    "solvolysis": 820.0,
}

# Logistics cost per unit of accepted WTB material.
BASE_LOGISTICS_COST = {
    "mechanical_recycling": 70.0,
    "pyrolysis": PYROLYSIS_COST_COMPONENTS["transport"],
    "recovery": 65.0,
    "solvolysis": SOLVOLYSIS_COST_COMPONENTS["transport"],
}

# Investment cost associated with additional treatment capacity.
RECYCLER_INVESTMENT_COST = {
    "mechanical_recycling": 500.0,
    "pyrolysis": 700.0,
    "recovery": 450.0,
    "solvolysis": 1300.0,
}

# Pathway-switching cost faced by an incumbent recycler.
RECYCLER_SWITCHING_COST = {
    "mechanical_recycling": 100.0,
    "pyrolysis": 140.0,
    "recovery": 90.0,
    "solvolysis": 380.0,
}

# Annual fixed cost associated with installed treatment capacity.
# These costs discourage persistent idle and excess capacity.
ANNUAL_FIXED_CAPACITY_COST = {
    "mechanical_recycling": 55.0,
    "pyrolysis": 75.0,
    "recovery": 45.0,
    "solvolysis": 110.0,
}

# Number of years considered when recyclers assess expected
# investment profitability.
RECYCLER_PROFIT_HORIZON = 12

# Scaling parameter used when converting expected profit or NPV
# into a bounded behavioural decision signal.
RECYCLER_PROFIT_SCALE = 650.0


# ============================================================
# 3.3 Recycler heterogeneity
# ============================================================

# Economic switching threshold used by the active ABM recycler
# population. It is drawn in expected-profit/NPV units and bounded
# below by ABM_RECYCLER_SWITCHING_THRESHOLD_MIN.
ABM_RECYCLER_SWITCHING_THRESHOLD_MEAN = 0.50
ABM_RECYCLER_SWITCHING_THRESHOLD_STD = 12.0
ABM_RECYCLER_SWITCHING_THRESHOLD_MIN = 5.0

# Recycler discount-rate distribution.
MEAN_RECYCLER_DISCOUNT_RATE = 0.08
STD_RECYCLER_DISCOUNT_RATE = 0.02

# Exit-threshold distribution.
MEAN_RECYCLER_EXIT_THRESHOLD = 0.15
STD_RECYCLER_EXIT_THRESHOLD = 0.04

# Exponential moving-average smoothing factor applied to recycler
# utilisation, profit, and material-sales signals.
RECYCLER_EMA_SMOOTHING = 0.50


# ============================================================
# 3.4 Recycler contraction and exit
# ============================================================

# Number of consecutive negative-profit years required before
# capacity contraction may occur.
RECYCLER_NEGATIVE_PROFIT_YEARS_FOR_CONTRACTION = 2

# Fraction of pathway capacity removed following a contraction decision.
RECYCLER_CAPACITY_CONTRACTION_RATE = 0.08

# Minimum probabilistic adoption signal required for solvolysis
# adoption or investment consideration.
SOLVOLYSIS_ADOPTION_THRESHOLD_PROB = 0.50


# ============================================================
# 3.5 Scenario effects on recycler economics
# ============================================================

# Maximum proportional gate-fee effect generated by EPR policy.
EPR_GATE_FEE_MULTIPLIER = 0.40

# Maximum proportional investment-cost reduction generated by
# direct technology support.
TECH_SUPPORT_INVESTMENT_REDUCTION = 0.35


# ============================================================
# 3.6 Manufacturer behaviour
# ============================================================

# Mean adoption threshold for manufacturers considering recovered
# high-quality material.
MEAN_MANUFACTURER_ADOPTION_THRESHOLD = 0.50

# Sensitivity of manufacturer adoption probability to differences
# between perceived attractiveness and adoption thresholds.
MEAN_MANUFACTURER_ADOPTION_SENSITIVITY = 5.0

# Mean perceived technical risk associated with solvolysis-derived
# recovered material.
MEAN_PERCEIVED_TECHNICAL_RISK = 0.35

# Baseline perceived reliability of material supply from an active
# recovered-material supplier.
#
# Realised adoption remains impossible when no commercial solvolysis
# capacity or qualified supply is available.
INITIAL_OBSERVED_SUPPLY_RELIABILITY = 0.00
SUPPLY_RELIABILITY_SMOOTHING = 0.50

# Sensitivity of manufacturer adoption to certification and
# technology-readiness conditions.
CERTIFICATION_SENSITIVITY = 2.0

# No artificial minimum manufacturer adoption is imposed.
# Closed-loop demand therefore responds directly to realised adoption.
CLOSED_LOOP_DEMAND_ADOPTION_FLOOR = 0.00

# Maximum mandate-activated closed-loop demand represented by a
# fully implemented recycled-content mandate.
#
# A value of 0.15 means that a fully implemented mandate activates
# compulsory recovered-fibre demand equal to 15% of the independent
# new-blade glass-fibre demand series.
CLOSED_LOOP_MANDATE_DEMAND_FACTOR = 0.15

# Fraction of technically matched eligible supply and demand that
# becomes realized blade-manufacturer offtake. The remainder may
# enter an open-loop market or recovered-material inventory.
CLOSED_LOOP_OFFTAKE_REALIZATION_RATE = 0.85

# Fraction of manufacturers reconsidering their adoption decision
# in each year once solvolysis becomes commercially available.
MANUFACTURER_ANNUAL_REVIEW_RATE = 0.35

# Baseline annual probability that an adopting manufacturer abandons
# recovered material in the absence of sufficiently strong performance.
MANUFACTURER_BASE_ABANDONMENT_RATE = 0.02


# ============================================================
# 3.7 Initial allocation inertia
# ============================================================

# Initial pathway shares influence early operator decisions through
# a temporary inertia term.
INITIAL_ALLOCATION_INERTIA = 0.80

# Annual decay rate of the initial allocation-inertia effect.
INITIAL_ALLOCATION_INERTIA_DECAY = 0.12


# ============================================================
# 3.8 Flow rerouting
# ============================================================

# Maximum fraction of flow rejected from its initially preferred
# pathway that may be redirected to alternative pathways.
MAX_REROUTING_SHARE = {
    "reuse": 0.40,
    "repurposing": 0.45,
    "mechanical_recycling": 0.35,
    "pyrolysis": 0.35,
    "recovery": 0.25,
    "solvolysis": 0.30,
}

# Technical and organisational compatibility between an initially
# preferred pathway and each alternative pathway.
#
# Values are stylised structural assumptions in [0, 1] and should
# be examined through sensitivity analysis.
REROUTING_COMPATIBILITY = {
    "reuse": {
        "repurposing": 0.80,
        "mechanical_recycling": 0.35,
        "pyrolysis": 0.15,
        "recovery": 0.25,
        "solvolysis": 0.30,
    },
    "repurposing": {
        "reuse": 0.15,
        "mechanical_recycling": 0.55,
        "pyrolysis": 0.20,
        "recovery": 0.40,
        "solvolysis": 0.35,
    },
    "mechanical_recycling": {
        "reuse": 0.00,
        "repurposing": 0.00,
        "pyrolysis": 0.30,
        "recovery": 0.55,
        "solvolysis": 0.30,
    },
    "pyrolysis": {
        "reuse": 0.00,
        "repurposing": 0.00,
        "mechanical_recycling": 0.25,
        "recovery": 0.50,
        "solvolysis": 0.35,
    },
    "recovery": {
        "reuse": 0.00,
        "repurposing": 0.00,
        "mechanical_recycling": 0.20,
        "pyrolysis": 0.15,
        "solvolysis": 0.20,
    },
    "solvolysis": {
        "reuse": 0.00,
        "repurposing": 0.00,
        "mechanical_recycling": 0.20,
        "pyrolysis": 0.20,
        "recovery": 0.35,
    },
}


# ============================================================
# 3.9 Recovered-material sales and inventories
# ============================================================

# Initial fraction of recovered output absorbed by downstream markets.
#
# The solvolysis sales ratio is initialised later using scenario
# parameters because initial market support differs across scenarios.
INITIAL_MATERIAL_SALES_RATIO = {
    "mechanical_recycling": 0.90,
    "pyrolysis": 0.85,
    "recovery": 1.00,
    "solvolysis": 0.00,
}

# Smoothing factor applied to the realised material-sales ratio.
MATERIAL_SALES_RATIO_SMOOTHING = 0.40

# Annual cost of carrying unsold recovered-material inventory,
# expressed as a fraction of inventory value.
INVENTORY_HOLDING_COST_RATE = 0.08

# Annual physical deterioration, obsolescence, or loss rate of
# recovered-material inventories.
RECOVERED_STOCK_DECAY_RATE = {
    "mechanical_recycling": 0.05,
    "pyrolysis": 0.04,
    "solvolysis": 0.03,
}


# ============================================================
# 3.10 Capacity expansion
# ============================================================

# Minimum realised capacity-utilisation ratio required before an
# expansion decision may be considered.
EXPANSION_UTILIZATION_THRESHOLD = 0.78

# Minimum realised material-sales ratio required before expansion.
EXPANSION_MIN_SALES_RATIO = 0.65

# Number of consecutive favourable years required before expansion.
EXPANSION_SIGNAL_YEARS = 2

# Minimum waiting period between successive expansion decisions.
EXPANSION_COOLDOWN_YEARS = 3

# Upper bound on the annual probability of an expansion decision.
EXPANSION_MAX_DECISION_PROBABILITY = 0.60

# Multiplier applied to capacity-proportional expansion CAPEX.
EXPANSION_CAPEX_MULTIPLIER = 1.00


# ============================================================
# 3.11 Incumbent pathway switching to solvolysis
# ============================================================

# Number of consecutive favourable years required before switching.
SWITCH_SIGNAL_YEARS = 2

# Minimum waiting period between successive pathway-switching decisions.
SWITCH_COOLDOWN_YEARS = 5

# Maximum fraction of incumbent capacity that may switch to
# solvolysis in one year.
MAX_ANNUAL_SWITCH_SHARE = 0.15

# Minimum closed-loop demand pressure required before switching.
MIN_SOLVOLYSIS_DEMAND_PRESSURE = 0.15

# Minimum technical compatibility required for an incumbent plant
# to be considered for retrofit.
MIN_SOLVOLYSIS_RETROFIT_COMPATIBILITY = 0.50

# Technical compatibility between each incumbent treatment pathway
# and solvolysis conversion.
SOLVOLYSIS_RETROFIT_COMPATIBILITY = {
    "mechanical_recycling": 0.75,
    "pyrolysis": 0.65,
    "recovery": 0.25,
    "solvolysis": 1.00,
}

# Baseline share of compatible incumbent capacity converted when a
# switching decision is realised.
SOLVOLYSIS_CONVERSION_BASE = 0.35

# Additional conversion share generated by retrofit compatibility.
SOLVOLYSIS_CONVERSION_COMPATIBILITY_EFFECT = 0.30

# Fixed and capacity-proportional retrofit costs.
SOLVOLYSIS_RETROFIT_FIXED_COST = 250.0
SOLVOLYSIS_RETROFIT_CAPEX_PER_CAPACITY = 500.0


# ============================================================
# 3.12 Greenfield solvolysis entry
# ============================================================

# Standard capacity added by a new solvolysis entrant.
NEW_SOLVOLYSIS_ENTRY_CAPACITY = 2.50

# Fixed cost associated with new solvolysis market entry.
NEW_SOLVOLYSIS_ENTRY_FIXED_COST = 500.0

# Number of consecutive favourable years required before entry.
SOLVOLYSIS_ENTRY_SIGNAL_YEARS = 2

# Upper bound on the probability of greenfield entry in one year.
SOLVOLYSIS_ENTRY_MAX_PROBABILITY = 0.75


# ============================================================
# 3.13 Untreated-stock reallocation
# ============================================================

# Fraction of accumulated untreated stock reconsidered for treatment
# in each simulation year.
BASE_BACKLOG_REALLOCATION_RATE = 0.10


# ============================================================
# 3.14 Numerical tolerances
# ============================================================

TRL_NUMERICAL_TOLERANCE = 1e-10
CAPACITY_SYNC_TOLERANCE = 1e-8


# ============================================================
# 3.15 Behavioural-parameter consistency checks
# ============================================================

# Operator behaviour
assert np.isclose(
    MEAN_OPERATOR_COST_WEIGHT + MEAN_OPERATOR_ENV_WEIGHT,
    1.0,
)

assert 0.0 <= MEAN_OPERATOR_COST_WEIGHT <= 1.0
assert 0.0 <= MEAN_OPERATOR_ENV_WEIGHT <= 1.0
assert MEAN_OPERATOR_CHOICE_SENSITIVITY > 0.0

assert set(PATHWAY_ENVIRONMENTAL_SCORE) == set(PATHWAYS)

assert all(
    0.0 <= value <= 1.0
    for value in PATHWAY_ENVIRONMENTAL_SCORE.values()
)

# Recycler parameter dictionaries
assert set(BASE_GATE_FEE) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(BASE_LOGISTICS_COST) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(RECYCLER_INVESTMENT_COST) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(RECYCLER_SWITCHING_COST) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert set(ANNUAL_FIXED_CAPACITY_COST) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert all(
    value >= 0.0
    for value in BASE_GATE_FEE.values()
)

assert all(
    value >= 0.0
    for value in BASE_LOGISTICS_COST.values()
)

assert all(
    value >= 0.0
    for value in RECYCLER_INVESTMENT_COST.values()
)

assert all(
    value >= 0.0
    for value in RECYCLER_SWITCHING_COST.values()
)

assert all(
    value >= 0.0
    for value in ANNUAL_FIXED_CAPACITY_COST.values()
)

# Recycler heterogeneity
assert ABM_RECYCLER_SWITCHING_THRESHOLD_MEAN >= 0.0
assert ABM_RECYCLER_SWITCHING_THRESHOLD_STD >= 0.0
assert ABM_RECYCLER_SWITCHING_THRESHOLD_MIN >= 0.0

assert 0.0 <= MEAN_RECYCLER_EXIT_THRESHOLD <= 1.0
assert STD_RECYCLER_EXIT_THRESHOLD >= 0.0

assert 0.0 <= MEAN_RECYCLER_DISCOUNT_RATE <= 1.0
assert STD_RECYCLER_DISCOUNT_RATE >= 0.0

assert 0.0 <= RECYCLER_EMA_SMOOTHING <= 1.0

# Recycler contraction
assert (
    RECYCLER_NEGATIVE_PROFIT_YEARS_FOR_CONTRACTION
    >= 1
)

assert (
    0.0
    <= RECYCLER_CAPACITY_CONTRACTION_RATE
    <= 1.0
)

assert 0.0 <= SOLVOLYSIS_ADOPTION_THRESHOLD_PROB <= 1.0

# Scenario effects
assert 0.0 <= EPR_GATE_FEE_MULTIPLIER <= 1.0
assert 0.0 <= TECH_SUPPORT_INVESTMENT_REDUCTION <= 1.0

# Manufacturer behaviour
assert 0.0 <= MEAN_MANUFACTURER_ADOPTION_THRESHOLD <= 1.0
assert MEAN_MANUFACTURER_ADOPTION_SENSITIVITY > 0.0
assert 0.0 <= MEAN_PERCEIVED_TECHNICAL_RISK <= 1.0
assert 0.0 <= INITIAL_OBSERVED_SUPPLY_RELIABILITY <= 1.0
assert CERTIFICATION_SENSITIVITY >= 0.0

assert 0.0 <= CLOSED_LOOP_DEMAND_ADOPTION_FLOOR <= 1.0

assert (
    0.0
    <= CLOSED_LOOP_MANDATE_DEMAND_FACTOR
    <= 1.0
)

assert 0.0 <= CLOSED_LOOP_OFFTAKE_REALIZATION_RATE <= 1.0
assert 0.0 <= MANUFACTURER_ANNUAL_REVIEW_RATE <= 1.0
assert 0.0 <= MANUFACTURER_BASE_ABANDONMENT_RATE <= 1.0

# Allocation inertia
assert 0.0 <= INITIAL_ALLOCATION_INERTIA <= 1.0
assert 0.0 <= INITIAL_ALLOCATION_INERTIA_DECAY <= 1.0

# Rerouting
assert set(MAX_REROUTING_SHARE) == set(PATHWAYS)
assert set(REROUTING_COMPATIBILITY) == set(PATHWAYS)

assert all(
    0.0 <= value <= 1.0
    for value in MAX_REROUTING_SHARE.values()
)

for origin_pathway, destination_values in (
    REROUTING_COMPATIBILITY.items()
):
    expected_destinations = set(PATHWAYS) - {origin_pathway}

    assert set(destination_values) == expected_destinations

    assert all(
        0.0 <= value <= 1.0
        for value in destination_values.values()
    )

# Material sales and inventories
assert set(INITIAL_MATERIAL_SALES_RATIO) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert all(
    0.0 <= value <= 1.0
    for value in INITIAL_MATERIAL_SALES_RATIO.values()
)

assert 0.0 <= MATERIAL_SALES_RATIO_SMOOTHING <= 1.0
assert INVENTORY_HOLDING_COST_RATE >= 0.0

assert set(RECOVERED_STOCK_DECAY_RATE) == set(
    MATERIAL_RECOVERY_PATHWAYS
)

assert all(
    0.0 <= value <= 1.0
    for value in RECOVERED_STOCK_DECAY_RATE.values()
)

# Capacity expansion
assert 0.0 <= EXPANSION_UTILIZATION_THRESHOLD <= 1.0
assert 0.0 <= EXPANSION_MIN_SALES_RATIO <= 1.0
assert EXPANSION_SIGNAL_YEARS >= 1
assert EXPANSION_COOLDOWN_YEARS >= 0

assert (
    0.0
    <= EXPANSION_MAX_DECISION_PROBABILITY
    <= 1.0
)

assert EXPANSION_CAPEX_MULTIPLIER >= 0.0

# Pathway switching
assert SWITCH_SIGNAL_YEARS >= 1
assert SWITCH_COOLDOWN_YEARS >= 0
assert 0.0 <= MAX_ANNUAL_SWITCH_SHARE <= 1.0
assert 0.0 <= MIN_SOLVOLYSIS_DEMAND_PRESSURE <= 1.0

assert (
    0.0
    <= MIN_SOLVOLYSIS_RETROFIT_COMPATIBILITY
    <= 1.0
)

assert set(SOLVOLYSIS_RETROFIT_COMPATIBILITY) == set(
    RESOURCE_RECOVERY_PATHWAYS
)

assert all(
    0.0 <= value <= 1.0
    for value in SOLVOLYSIS_RETROFIT_COMPATIBILITY.values()
)

assert 0.0 <= SOLVOLYSIS_CONVERSION_BASE <= 1.0

assert (
    0.0
    <= SOLVOLYSIS_CONVERSION_COMPATIBILITY_EFFECT
    <= 1.0
)

assert SOLVOLYSIS_RETROFIT_FIXED_COST >= 0.0
assert SOLVOLYSIS_RETROFIT_CAPEX_PER_CAPACITY >= 0.0

# Greenfield entry
assert NEW_SOLVOLYSIS_ENTRY_CAPACITY > 0.0
assert NEW_SOLVOLYSIS_ENTRY_FIXED_COST >= 0.0
assert SOLVOLYSIS_ENTRY_SIGNAL_YEARS >= 1

assert (
    0.0
    <= SOLVOLYSIS_ENTRY_MAX_PROBABILITY
    <= 1.0
)

# Untreated-stock reallocation and tolerances
assert 0.0 <= BASE_BACKLOG_REALLOCATION_RATE <= 1.0
assert TRL_NUMERICAL_TOLERANCE > 0.0
assert CAPACITY_SYNC_TOLERANCE > 0.0


# ============================================================
# 3.16 Parameter inspection tables
# ============================================================

agent_behaviour_parameter_table = pd.DataFrame(
    [
        {
            "agent": "operator",
            "parameter": "cost_weight",
            "value": MEAN_OPERATOR_COST_WEIGHT,
        },
        {
            "agent": "operator",
            "parameter": "environmental_weight",
            "value": MEAN_OPERATOR_ENV_WEIGHT,
        },
        {
            "agent": "operator",
            "parameter": "choice_sensitivity",
            "value": MEAN_OPERATOR_CHOICE_SENSITIVITY,
        },
        {
            "agent": "recycler",
            "parameter": "discount_rate",
            "value": MEAN_RECYCLER_DISCOUNT_RATE,
        },
        {
            "agent": "recycler",
            "parameter": "economic_switching_threshold_location",
            "value": ABM_RECYCLER_SWITCHING_THRESHOLD_MEAN,
        },
        {
            "agent": "recycler",
            "parameter": "economic_switching_threshold_std",
            "value": ABM_RECYCLER_SWITCHING_THRESHOLD_STD,
        },
        {
            "agent": "recycler",
            "parameter": "economic_switching_threshold_minimum",
            "value": ABM_RECYCLER_SWITCHING_THRESHOLD_MIN,
        },
        {
            "agent": "manufacturer",
            "parameter": "adoption_threshold",
            "value": MEAN_MANUFACTURER_ADOPTION_THRESHOLD,
        },
        {
            "agent": "manufacturer",
            "parameter": "adoption_sensitivity",
            "value": MEAN_MANUFACTURER_ADOPTION_SENSITIVITY,
        },
        {
            "agent": "manufacturer",
            "parameter": "perceived_technical_risk",
            "value": MEAN_PERCEIVED_TECHNICAL_RISK,
        },
        {
            "agent": "manufacturer",
            "parameter": "supply_reliability",
            "value": INITIAL_OBSERVED_SUPPLY_RELIABILITY,
        },
    ]
)

recycler_economic_parameter_table = pd.DataFrame(
    {
        "pathway": RESOURCE_RECOVERY_PATHWAYS,
        "gate_fee": [
            BASE_GATE_FEE[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "logistics_cost": [
            BASE_LOGISTICS_COST[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "investment_cost": [
            RECYCLER_INVESTMENT_COST[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "switching_cost": [
            RECYCLER_SWITCHING_COST[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
        "annual_fixed_capacity_cost": [
            ANNUAL_FIXED_CAPACITY_COST[pathway]
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        ],
    }
)

transition_mechanism_parameter_table = pd.DataFrame(
    [
        {
            "mechanism": "capacity_expansion",
            "parameter": "utilization_threshold",
            "value": EXPANSION_UTILIZATION_THRESHOLD,
        },
        {
            "mechanism": "capacity_expansion",
            "parameter": "minimum_sales_ratio",
            "value": EXPANSION_MIN_SALES_RATIO,
        },
        {
            "mechanism": "capacity_expansion",
            "parameter": "signal_years",
            "value": EXPANSION_SIGNAL_YEARS,
        },
        {
            "mechanism": "switching",
            "parameter": "maximum_annual_switch_share",
            "value": MAX_ANNUAL_SWITCH_SHARE,
        },
        {
            "mechanism": "switching",
            "parameter": "minimum_demand_pressure",
            "value": MIN_SOLVOLYSIS_DEMAND_PRESSURE,
        },
        {
            "mechanism": "greenfield_entry",
            "parameter": "entry_capacity",
            "value": NEW_SOLVOLYSIS_ENTRY_CAPACITY,
        },
        {
            "mechanism": "manufacturer_adoption",
            "parameter": "annual_review_rate",
            "value": MANUFACTURER_ANNUAL_REVIEW_RATE,
        },
        {
            "mechanism": "backlog_reallocation",
            "parameter": "annual_reallocation_rate",
            "value": BASE_BACKLOG_REALLOCATION_RATE,
        },
    ]
)

display(agent_behaviour_parameter_table)
display(recycler_economic_parameter_table)
display(transition_mechanism_parameter_table)


In [ ]:
# ============================================================
# 4. Core mathematical and allocation functions
# ============================================================


# ============================================================
# Numerical stability
# ============================================================

NUMERICAL_EPSILON = 1e-12


# ============================================================
# Market-allocation parameters used by this cell
# ============================================================

# CLOSED_LOOP_MANDATE_DEMAND_FACTOR is defined once in the
# behavioural-parameter cell and is sampled explicitly in the
# Monte Carlo analysis. No hidden fallback is permitted here.
if "CLOSED_LOOP_MANDATE_DEMAND_FACTOR" not in globals():
    raise NameError(
        "CLOSED_LOOP_MANDATE_DEMAND_FACTOR must be defined "
        "before the market-allocation functions."
    )

OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY = dict(
    globals().get(
        "OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY",
        {
            "mechanical_recycling": 0.45,
            "pyrolysis": 0.20,
            "solvolysis": 0.10,
        },
    )
)


def logistic(x):
    """
    Transform a scalar or NumPy array into values between 0 and 1.

    The input is clipped to avoid numerical overflow when evaluating
    the exponential function.
    """

    values = np.asarray(x, dtype=float)
    values = np.clip(values, -60.0, 60.0)

    probabilities = 1.0 / (1.0 + np.exp(-values))

    if probabilities.ndim == 0:
        return float(probabilities)

    return probabilities


def bounded(value, lower=0.0, upper=1.0):
    """
    Constrain a scalar value to the interval [lower, upper].
    """

    if lower > upper:
        raise ValueError(
            "The lower bound cannot exceed the upper bound."
        )

    return float(np.clip(value, lower, upper))


def safe_divide(numerator, denominator, default=0.0):
    """
    Divide two values while avoiding division by a denominator
    that is numerically equal to zero.
    """

    if abs(float(denominator)) <= NUMERICAL_EPSILON:
        return float(default)

    return float(numerator) / float(denominator)


# ============================================================
# Whole-blade input to pathway-output conversion
# ============================================================

def resource_output_yield(pathway):
    """Return usable output per unit of accepted whole-blade mass."""

    if pathway not in RECOVERABLE_FEEDSTOCK_FRACTION:
        raise KeyError(f"Unknown resource-recovery pathway: {pathway}")

    if pathway not in RECOVERY_EFFICIENCY:
        raise KeyError(f"Missing recovery efficiency for: {pathway}")

    return float(
        RECOVERABLE_FEEDSTOCK_FRACTION[pathway]
        * RECOVERY_EFFICIENCY[pathway]
    )


assert np.isclose(
    resource_output_yield("solvolysis"),
    0.587 * 0.85,
)


# ============================================================
# Exogenous WTB end-of-life inflow
# ============================================================

def decommissioned_wtb_inflow(year):
    """
    Calculate the annual post-2025 WTB end-of-life inflow.

    The inflow follows an exogenous compound-growth trajectory:

        E_t = E_0 * (1 + g) ** (t - START_YEAR)
    """

    year = int(year)

    if year < START_YEAR:
        raise ValueError(
            f"year must be greater than or equal to {START_YEAR}."
        )

    elapsed_years = year - START_YEAR

    return float(
        BASE_DECOMMISSIONED_WTB
        * (1.0 + ANNUAL_EOL_GROWTH) ** elapsed_years
    )


# ============================================================
# Experience-based processing-cost learning
# ============================================================

def learning_cost(
    initial_cost,
    cumulative_throughput,
    reference_throughput,
    learning_exponent,
):
    """
    Calculate the processing cost after experience-based learning.

    Cost declines once cumulative throughput exceeds the reference
    experience level:

        c_t = c_0 * (Q_t / Q_ref) ** (-beta)

    Before reaching Q_ref, the cost remains equal to c_0.
    """

    initial_cost = float(initial_cost)
    cumulative_throughput = float(cumulative_throughput)
    reference_throughput = float(reference_throughput)
    learning_exponent = float(learning_exponent)

    if initial_cost < 0.0:
        raise ValueError("initial_cost cannot be negative.")

    if cumulative_throughput < 0.0:
        raise ValueError(
            "cumulative_throughput cannot be negative."
        )

    if reference_throughput <= 0.0:
        raise ValueError(
            "reference_throughput must be strictly positive."
        )

    if learning_exponent < 0.0:
        raise ValueError(
            "learning_exponent cannot be negative."
        )

    effective_throughput = max(
        cumulative_throughput,
        reference_throughput,
    )

    experience_ratio = (
        effective_throughput
        / reference_throughput
    )

    return float(
        initial_cost
        * experience_ratio ** (-learning_exponent)
    )


# ============================================================
# Conversion of pathway scores into desired shares
# ============================================================

def normalize_scores_to_shares(
    scores,
    sensitivity=1.0,
):
    """
    Convert pathway-attractiveness scores into desired shares
    using a numerically stable softmax transformation.

    Higher sensitivity produces a stronger concentration of the
    desired flow in the most attractive pathway.
    """

    if not scores:
        return {}

    sensitivity = float(sensitivity)

    if sensitivity < 0.0:
        raise ValueError("sensitivity cannot be negative.")

    pathway_names = tuple(scores.keys())

    score_values = np.asarray(
        [
            float(scores[pathway])
            for pathway in pathway_names
        ],
        dtype=float,
    )

    if not np.all(np.isfinite(score_values)):
        raise ValueError(
            "All pathway scores must be finite."
        )

    if sensitivity <= NUMERICAL_EPSILON:
        equal_share = 1.0 / len(pathway_names)

        return {
            pathway: equal_share
            for pathway in pathway_names
        }

    scaled_scores = sensitivity * score_values

    # Subtracting the maximum score makes the softmax numerically
    # stable without changing the resulting shares.
    scaled_scores -= np.max(scaled_scores)

    exponential_scores = np.exp(
        np.clip(scaled_scores, -60.0, 60.0)
    )

    denominator = float(exponential_scores.sum())

    if denominator <= NUMERICAL_EPSILON:
        equal_share = 1.0 / len(pathway_names)

        return {
            pathway: equal_share
            for pathway in pathway_names
        }

    shares = exponential_scores / denominator

    return {
        pathway: float(share)
        for pathway, share in zip(
            pathway_names,
            shares,
        )
    }


# ============================================================
# Capacity-constrained pathway allocation
# ============================================================

def apply_capacity_constraints(
    desired_flows,
    capacities,
    rerouting_compatibility=None,
    max_rerouting_share=None,
    return_diagnostics=False,
):
    """
    Apply pathway capacities and limited compatibility-constrained rerouting.

    The first allocation respects the route selected by operators:

        initial_actual_k = min(desired_k, capacity_k)

    Only a bounded share of flow rejected from its preferred route may be
    redirected, and only to technologically compatible routes with spare
    capacity. Consequently, system-wide spare capacity and untreated flow
    may coexist, allowing a genuine capacity-allocation mismatch to emerge.

    Parameters
    ----------
    desired_flows : dict
        Desired WTB flow by pathway.

    capacities : dict
        Opening treatment capacity by pathway.

    rerouting_compatibility : dict, optional
        Nested origin-destination compatibility coefficients in [0, 1].

    max_rerouting_share : dict, optional
        Maximum share of rejected origin flow that may be rerouted.

    return_diagnostics : bool, default=False
        If True, returns a diagnostic dictionary. Otherwise preserves the
        original three-object API.
    """
    if set(desired_flows) != set(capacities):
        raise ValueError(
            "desired_flows and capacities must contain the same pathways."
        )

    compatibility = (
        REROUTING_COMPATIBILITY
        if rerouting_compatibility is None
        else rerouting_compatibility
    )

    rerouting_limits = (
        MAX_REROUTING_SHARE
        if max_rerouting_share is None
        else max_rerouting_share
    )

    pathway_names = tuple(desired_flows.keys())

    desired = {
        pathway: max(0.0, float(desired_flows[pathway]))
        for pathway in pathway_names
    }

    available_capacity = {
        pathway: max(0.0, float(capacities[pathway]))
        for pathway in pathway_names
    }

    initial_actual_flows = {
        pathway: min(desired[pathway], available_capacity[pathway])
        for pathway in pathway_names
    }

    actual_flows = initial_actual_flows.copy()

    initial_rejected_by_pathway = {
        pathway: max(0.0, desired[pathway] - initial_actual_flows[pathway])
        for pathway in pathway_names
    }

    spare_capacity = {
        pathway: max(
            0.0,
            available_capacity[pathway] - actual_flows[pathway],
        )
        for pathway in pathway_names
    }

    reallocated_from_pathway = {
        pathway: 0.0
        for pathway in pathway_names
    }

    reallocated_to_pathway = {
        pathway: 0.0
        for pathway in pathway_names
    }

    rerouting_matrix = {
        origin: {
            destination: 0.0
            for destination in pathway_names
        }
        for origin in pathway_names
    }

    # Process the largest rejected origin flows first. This deterministic
    # priority avoids a hidden dependence on dictionary order.
    origins = sorted(
        pathway_names,
        key=lambda pathway: initial_rejected_by_pathway[pathway],
        reverse=True,
    )

    for origin in origins:
        rejected = initial_rejected_by_pathway[origin]

        if rejected <= NUMERICAL_EPSILON:
            continue

        reroutable = rejected * bounded(
            rerouting_limits.get(origin, 0.0)
        )

        remaining = reroutable

        # Iterative proportional allocation prevents a destination that
        # reaches capacity from blocking the use of other compatible routes.
        while remaining > NUMERICAL_EPSILON:
            eligible_destinations = [
                destination
                for destination in pathway_names
                if destination != origin
                and spare_capacity[destination] > NUMERICAL_EPSILON
                and bounded(
                    compatibility.get(origin, {}).get(destination, 0.0)
                ) > NUMERICAL_EPSILON
            ]

            if not eligible_destinations:
                break

            weights = {
                destination: (
                    bounded(
                        compatibility.get(origin, {}).get(destination, 0.0)
                    )
                    * spare_capacity[destination]
                )
                for destination in eligible_destinations
            }

            total_weight = sum(weights.values())

            if total_weight <= NUMERICAL_EPSILON:
                break

            allocated_this_round = 0.0

            for destination in eligible_destinations:
                proposed = remaining * safe_divide(
                    weights[destination],
                    total_weight,
                    default=0.0,
                )

                allocation = min(
                    proposed,
                    spare_capacity[destination],
                )

                if allocation <= NUMERICAL_EPSILON:
                    continue

                actual_flows[destination] += allocation
                spare_capacity[destination] -= allocation
                reallocated_from_pathway[origin] += allocation
                reallocated_to_pathway[destination] += allocation
                rerouting_matrix[origin][destination] += allocation
                allocated_this_round += allocation

            if allocated_this_round <= NUMERICAL_EPSILON:
                break

            remaining -= allocated_this_round

    unmet_flows_by_pathway = {
        pathway: max(
            0.0,
            initial_rejected_by_pathway[pathway]
            - reallocated_from_pathway[pathway],
        )
        for pathway in pathway_names
    }

    total_desired_flow = sum(desired.values())
    total_available_capacity = sum(available_capacity.values())
    total_actual_flow = sum(actual_flows.values())
    total_unmet_flow = sum(unmet_flows_by_pathway.values())
    total_initial_rejected = sum(initial_rejected_by_pathway.values())
    total_reallocated = sum(reallocated_from_pathway.values())
    total_unused_capacity = sum(spare_capacity.values())

    aggregate_capacity_shortage = max(
        0.0,
        total_desired_flow - total_available_capacity,
    )

    capacity_allocation_mismatch = max(
        0.0,
        total_unmet_flow - aggregate_capacity_shortage,
    )

    diagnostics = {
        "initial_actual_flows": initial_actual_flows,
        "initial_rejected_by_pathway": initial_rejected_by_pathway,
        "reallocated_from_pathway": reallocated_from_pathway,
        "reallocated_to_pathway": reallocated_to_pathway,
        "rerouting_matrix": rerouting_matrix,
        "spare_capacity_after_rerouting": spare_capacity,
        "total_initial_rejected_flow": float(total_initial_rejected),
        "total_reallocated_flow": float(total_reallocated),
        "total_unused_capacity": float(total_unused_capacity),
        "aggregate_capacity_shortage": float(aggregate_capacity_shortage),
        "capacity_allocation_mismatch": float(capacity_allocation_mismatch),
        "allocation_balance_error": float(
            abs(total_desired_flow - total_actual_flow - total_unmet_flow)
        ),
        "rerouting_balance_error": float(
            abs(total_initial_rejected - total_reallocated - total_unmet_flow)
        ),
    }

    if return_diagnostics:
        return {
            "actual_flows": actual_flows,
            "total_unmet_flow": float(total_unmet_flow),
            "unmet_flows_by_pathway": unmet_flows_by_pathway,
            "diagnostics": diagnostics,
        }

    return (
        actual_flows,
        float(total_unmet_flow),
        unmet_flows_by_pathway,
    )


# ============================================================
# Proportional downstream material allocation
# ============================================================

def proportional_allocation(
    available_by_pathway,
    total_demand,
):
    """
    Allocate limited downstream demand proportionally across the
    available quantities supplied by the different pathways.

    Total utilization cannot exceed either total material availability
    or total downstream demand.
    """

    total_demand = max(
        0.0,
        float(total_demand),
    )

    availability = {
        pathway: max(
            0.0,
            float(quantity),
        )
        for pathway, quantity
        in available_by_pathway.items()
    }

    total_available = sum(availability.values())

    if (
        total_available <= NUMERICAL_EPSILON
        or total_demand <= NUMERICAL_EPSILON
    ):
        return {
            pathway: 0.0
            for pathway in availability
        }

    total_allocated = min(
        total_available,
        total_demand,
    )

    return {
        pathway: (
            total_allocated
            * safe_divide(
                quantity,
                total_available,
            )
        )
        for pathway, quantity
        in availability.items()
    }


# ============================================================
# Open-loop and closed-loop recovered-material utilization
# ============================================================

def compute_open_closed_utilization(
    recovered_material,
    open_loop_demand,
    closed_loop_demand,
    new_blade_glass_fibre_demand,
    manufacturer_adoption,
    recycled_content_mandate,
    pathway_certification_score,
):
    """Allocate recovered outputs under common demand and eligibility rules.

    Voluntary closed-loop demand is activated by realized manufacturer
    adoption. Mandatory demand is driven by an independent new-blade
    glass-fibre demand series. A pathway is eligible for closed-loop use only
    when both its material quality and certification score pass common
    thresholds. Remaining output can enter pathway-specific open-loop markets.
    """
    required_pathways = set(MATERIAL_RECOVERY_PATHWAYS)
    missing_pathways = required_pathways - set(recovered_material)
    missing_certification = required_pathways - set(pathway_certification_score)
    missing_open_loop_shares = required_pathways - set(
        OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY
    )
    if missing_pathways:
        raise ValueError(f"recovered_material is missing: {sorted(missing_pathways)}")
    if missing_certification:
        raise ValueError(
            "pathway_certification_score is missing: "
            f"{sorted(missing_certification)}"
        )
    if missing_open_loop_shares:
        raise ValueError(
            "OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY is missing: "
            f"{sorted(missing_open_loop_shares)}"
        )

    material_available = {
        pathway: max(0.0, float(recovered_material[pathway]))
        for pathway in MATERIAL_RECOVERY_PATHWAYS
    }
    open_loop_demand = max(0.0, float(open_loop_demand))
    potential_closed_loop_demand = max(0.0, float(closed_loop_demand))
    new_blade_glass_fibre_demand = max(
        0.0, float(new_blade_glass_fibre_demand)
    )
    manufacturer_adoption = bounded(manufacturer_adoption)
    recycled_content_mandate = bounded(recycled_content_mandate)
    mandate_demand_factor = bounded(CLOSED_LOOP_MANDATE_DEMAND_FACTOR)
    offtake_realization_rate = bounded(
        CLOSED_LOOP_OFFTAKE_REALIZATION_RATE
    )

    open_loop_demand_shares = {
        pathway: bounded(OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY[pathway])
        for pathway in MATERIAL_RECOVERY_PATHWAYS
    }
    if sum(open_loop_demand_shares.values()) > 1.0 + NUMERICAL_EPSILON:
        raise ValueError(
            "OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY must sum to no more than 1.0."
        )

    adoption_activated_closed_loop_demand = (
        potential_closed_loop_demand * manufacturer_adoption
    )
    mandate_activated_closed_loop_demand = (
        mandate_demand_factor
        * recycled_content_mandate
        * new_blade_glass_fibre_demand
    )
    mandatory_closed_loop_demand = min(
        potential_closed_loop_demand,
        mandate_activated_closed_loop_demand,
    )
    remaining_potential_demand = max(
        0.0, potential_closed_loop_demand - mandatory_closed_loop_demand
    )
    voluntary_closed_loop_demand = min(
        adoption_activated_closed_loop_demand,
        remaining_potential_demand,
    )
    effective_closed_loop_demand = (
        mandatory_closed_loop_demand + voluntary_closed_loop_demand
    )

    closed_loop_eligibility_by_pathway = {
        pathway: bool(
            MATERIAL_QUALITY[pathway] >= CLOSED_LOOP_MIN_MATERIAL_QUALITY
            and bounded(pathway_certification_score[pathway])
            >= CLOSED_LOOP_MIN_CERTIFICATION_SCORE
        )
        for pathway in MATERIAL_RECOVERY_PATHWAYS
    }
    eligible_closed_loop_supply_by_pathway = {
        pathway: (
            material_available[pathway]
            if closed_loop_eligibility_by_pathway[pathway]
            else 0.0
        )
        for pathway in MATERIAL_RECOVERY_PATHWAYS
    }
    high_quality_recovered_material = sum(
        eligible_closed_loop_supply_by_pathway.values()
    )

    technically_matched_mandatory_supply = min(
        high_quality_recovered_material,
        mandatory_closed_loop_demand,
    )
    remaining_eligible_material = max(
        0.0,
        high_quality_recovered_material
        - technically_matched_mandatory_supply,
    )
    technically_matched_voluntary_supply = min(
        remaining_eligible_material,
        voluntary_closed_loop_demand,
    )
    realized_mandatory_closed_loop_use = (
        technically_matched_mandatory_supply
        * offtake_realization_rate
    )
    realized_voluntary_closed_loop_use = (
        technically_matched_voluntary_supply
        * offtake_realization_rate
    )
    total_closed_loop_utilization = (
        realized_mandatory_closed_loop_use
        + realized_voluntary_closed_loop_use
    )
    closed_loop_use_by_pathway = {
        pathway: (
            total_closed_loop_utilization
            * safe_divide(
                eligible_closed_loop_supply_by_pathway[pathway],
                high_quality_recovered_material,
                default=0.0,
            )
        )
        for pathway in MATERIAL_RECOVERY_PATHWAYS
    }

    unmet_mandatory_closed_loop_demand = max(
        0.0,
        mandatory_closed_loop_demand - realized_mandatory_closed_loop_use,
    )
    unmet_voluntary_closed_loop_demand = max(
        0.0,
        voluntary_closed_loop_demand - realized_voluntary_closed_loop_use,
    )
    total_unmet_closed_loop_demand = (
        unmet_mandatory_closed_loop_demand
        + unmet_voluntary_closed_loop_demand
    )
    mandatory_compliance_ratio = safe_divide(
        realized_mandatory_closed_loop_use,
        mandatory_closed_loop_demand,
        default=0.0,
    )
    closed_loop_supply_coverage_ratio = safe_divide(
        total_closed_loop_utilization,
        effective_closed_loop_demand,
        default=0.0,
    )

    open_loop_demand_by_pathway = {
        pathway: open_loop_demand * open_loop_demand_shares[pathway]
        for pathway in MATERIAL_RECOVERY_PATHWAYS
    }
    effective_open_loop_demand = sum(open_loop_demand_by_pathway.values())
    unallocated_open_loop_demand = max(
        0.0, open_loop_demand - effective_open_loop_demand
    )
    available_for_open_loop = {
        pathway: max(
            0.0,
            material_available[pathway] - closed_loop_use_by_pathway[pathway],
        )
        for pathway in MATERIAL_RECOVERY_PATHWAYS
    }
    open_loop_use_by_pathway = {
        pathway: min(
            available_for_open_loop[pathway],
            open_loop_demand_by_pathway[pathway],
        )
        for pathway in MATERIAL_RECOVERY_PATHWAYS
    }
    unmet_open_loop_demand_by_pathway = {
        pathway: max(
            0.0,
            open_loop_demand_by_pathway[pathway]
            - open_loop_use_by_pathway[pathway],
        )
        for pathway in MATERIAL_RECOVERY_PATHWAYS
    }
    total_open_loop_use = sum(open_loop_use_by_pathway.values())
    total_unmet_open_loop_demand = sum(
        unmet_open_loop_demand_by_pathway.values()
    )
    unutilized_material_by_pathway = {
        pathway: max(
            0.0,
            material_available[pathway]
            - closed_loop_use_by_pathway[pathway]
            - open_loop_use_by_pathway[pathway],
        )
        for pathway in MATERIAL_RECOVERY_PATHWAYS
    }
    total_unutilized_material = sum(unutilized_material_by_pathway.values())

    if not np.isclose(
        effective_closed_loop_demand,
        voluntary_closed_loop_demand + mandatory_closed_loop_demand,
        atol=NUMERICAL_EPSILON,
        rtol=1e-9,
    ):
        raise AssertionError("Effective closed-loop demand is inconsistent.")
    if total_closed_loop_utilization > (
        high_quality_recovered_material + NUMERICAL_EPSILON
    ):
        raise AssertionError("Closed-loop use exceeds eligible supply.")
    if not np.isclose(
        total_unmet_closed_loop_demand,
        effective_closed_loop_demand - total_closed_loop_utilization,
        atol=NUMERICAL_EPSILON,
        rtol=1e-9,
    ):
        raise AssertionError("Unmet closed-loop demand is inconsistent.")
    total_available_material = sum(material_available.values())
    if not np.isclose(
        total_closed_loop_utilization
        + total_open_loop_use
        + total_unutilized_material,
        total_available_material,
        atol=max(NUMERICAL_EPSILON, 1e-9),
        rtol=1e-9,
    ):
        raise AssertionError("Recovered-material allocation does not balance.")

    return {
        "potential_closed_loop_demand": float(potential_closed_loop_demand),
        "closed_loop_offtake_realization_rate": float(
            offtake_realization_rate
        ),
        "new_blade_glass_fibre_demand": float(new_blade_glass_fibre_demand),
        "adoption_activated_closed_loop_demand": float(
            adoption_activated_closed_loop_demand
        ),
        "mandate_activated_closed_loop_demand": float(
            mandate_activated_closed_loop_demand
        ),
        "voluntary_closed_loop_demand": float(voluntary_closed_loop_demand),
        "mandatory_closed_loop_demand": float(mandatory_closed_loop_demand),
        "effective_closed_loop_demand": float(effective_closed_loop_demand),
        "closed_loop_eligibility_by_pathway": (
            closed_loop_eligibility_by_pathway
        ),
        "eligible_closed_loop_supply_by_pathway": (
            eligible_closed_loop_supply_by_pathway
        ),
        "high_quality_recovered_material": float(
            high_quality_recovered_material
        ),
        "realized_voluntary_closed_loop_use": float(
            realized_voluntary_closed_loop_use
        ),
        "realized_mandatory_closed_loop_use": float(
            realized_mandatory_closed_loop_use
        ),
        "realized_closed_loop_use": float(total_closed_loop_utilization),
        "total_closed_loop_utilization": float(
            total_closed_loop_utilization
        ),
        "unmet_voluntary_closed_loop_demand": float(
            unmet_voluntary_closed_loop_demand
        ),
        "unmet_mandatory_closed_loop_demand": float(
            unmet_mandatory_closed_loop_demand
        ),
        "total_unmet_closed_loop_demand": float(
            total_unmet_closed_loop_demand
        ),
        "mandatory_compliance_ratio": float(mandatory_compliance_ratio),
        "closed_loop_supply_coverage_ratio": float(
            closed_loop_supply_coverage_ratio
        ),
        "open_loop_demand_by_pathway": open_loop_demand_by_pathway,
        "effective_open_loop_demand": float(effective_open_loop_demand),
        "unallocated_open_loop_demand": float(unallocated_open_loop_demand),
        "unmet_open_loop_demand_by_pathway": (
            unmet_open_loop_demand_by_pathway
        ),
        "total_unmet_open_loop_demand": float(
            total_unmet_open_loop_demand
        ),
        "closed_loop_use_by_pathway": closed_loop_use_by_pathway,
        "open_loop_use_by_pathway": open_loop_use_by_pathway,
        "total_open_loop_use": float(total_open_loop_use),
        "unutilized_material_by_pathway": unutilized_material_by_pathway,
        "total_unutilized_material": float(total_unutilized_material),
    }


# ============================================================
# Basic function checks
# ============================================================

assert np.isclose(logistic(0.0), 0.5)
assert bounded(-0.5) == 0.0
assert bounded(1.5) == 1.0
assert safe_divide(10.0, 0.0) == 0.0

assert np.isclose(
    decommissioned_wtb_inflow(START_YEAR),
    BASE_DECOMMISSIONED_WTB,
)

_test_shares = normalize_scores_to_shares(
    {
        "a": 1.0,
        "b": 2.0,
        "c": 3.0,
    },
    sensitivity=1.0,
)

assert np.isclose(
    sum(_test_shares.values()),
    1.0,
)

_test_actual, _test_unmet, _test_unmet_by_pathway = (
    apply_capacity_constraints(
        desired_flows={
            "a": 80.0,
            "b": 20.0,
        },
        capacities={
            "a": 40.0,
            "b": 50.0,
        },
        rerouting_compatibility={
            "a": {"b": 1.0},
            "b": {"a": 1.0},
        },
        max_rerouting_share={
            "a": 1.0,
            "b": 1.0,
        },
    )
)

assert np.isclose(
    sum(_test_actual.values()),
    90.0,
)

assert np.isclose(
    _test_unmet,
    10.0,
)

assert np.isclose(
    _test_unmet_by_pathway["a"],
    10.0,
)

assert np.isclose(
    sum(_test_actual.values()) + _test_unmet,
    sum({"a": 80.0, "b": 20.0}.values()),
)

_test_utilization = compute_open_closed_utilization(
    recovered_material={
        "mechanical_recycling": 30.0,
        "pyrolysis": 10.0,
        "solvolysis": 20.0,
    },
    open_loop_demand=25.0,
    closed_loop_demand=20.0,
    new_blade_glass_fibre_demand=24.0,
    manufacturer_adoption=0.50,
    recycled_content_mandate=0.25,
    pathway_certification_score={
        "mechanical_recycling": 1.0,
        "pyrolysis": 1.0,
        "solvolysis": 1.0,
    },
)

assert np.isclose(
    _test_utilization["effective_closed_loop_demand"],
    _test_utilization["voluntary_closed_loop_demand"]
    + _test_utilization["mandatory_closed_loop_demand"],
)

assert np.isclose(
    _test_utilization["total_closed_loop_utilization"],
    _test_utilization["realized_voluntary_closed_loop_use"]
    + _test_utilization["realized_mandatory_closed_loop_use"],
)

assert np.isclose(
    _test_utilization["total_unmet_closed_loop_demand"],
    _test_utilization["effective_closed_loop_demand"]
    - _test_utilization["total_closed_loop_utilization"],
)

assert (
    _test_utilization["total_closed_loop_utilization"]
    <= _test_utilization["high_quality_recovered_material"]
    + NUMERICAL_EPSILON
)

assert not _test_utilization[
    "closed_loop_eligibility_by_pathway"
]["mechanical_recycling"]
assert not _test_utilization[
    "closed_loop_eligibility_by_pathway"
]["pyrolysis"]
assert _test_utilization[
    "closed_loop_eligibility_by_pathway"
]["solvolysis"]

In [ ]:
# ============================================================
# 5. Active ABM population settings
# ============================================================

# The operative agent populations are initialized as pandas DataFrames
# inside ABMLayer (Cell 7). Keeping the population settings here avoids
# a second, disconnected implementation of the same agent types.

N_OPERATORS = 100
N_RECYCLERS = 40
N_MANUFACTURERS = 60

# Common seed used for deterministic runs and population checks.
AGENT_POPULATION_SEED = 42


In [ ]:
BASE_CLOSED_LOOP_DEMAND_GROWTH = 0.01

# ============================================================
# 6. System Dynamics layer
# ============================================================


class SDLayer:
    """
    Aggregate System Dynamics layer of the hybrid SD-ABM model.

    The SD layer represents:

    - annual and cumulative WTB end-of-life inflows;
    - untreated WTB stock;
    - pathway-specific treatment capacity;
    - recovered-material inventories;
    - cumulative technology throughput;
    - learning-by-doing cost reductions;
    - solvolysis technology maturity;
    - open-loop and closed-loop material demand;
    - economic, environmental, and circularity indicators.

    Recycler investment, switching, contraction, and exit decisions
    are generated by the ABM layer and transferred to the SD layer
    as pathway-specific capacity changes.
    """

    def __init__(self, scenario_config, initial_capacity=None):
        """
        Initialize the aggregate state for one scenario.
        """

        self.config = scenario_config

        # --------------------------------------------------------
        # Time and WTB stocks
        # --------------------------------------------------------

        self.current_year = START_YEAR

        self.untreated_stock = 0.0

        self.cumulative_decommissioned = 0.0
        self.cumulative_treated = 0.0

        # --------------------------------------------------------
        # Treatment capacity
        # --------------------------------------------------------

        self.capacity = INITIAL_CAPACITY.copy()

        if initial_capacity is not None:
            for pathway in PATHWAYS:
                if pathway in initial_capacity:
                    self.capacity[pathway] = max(
                        0.0,
                        float(initial_capacity[pathway]),
                    )

        # Commercial solvolysis capacity cannot exist before the
        # technology reaches its maturity threshold.
        self.capacity["solvolysis"] = 0.0

        # --------------------------------------------------------
        # Recovered-material inventories
        # --------------------------------------------------------

        self.recovered_stock = {
            pathway: 0.0
            for pathway in MATERIAL_RECOVERY_PATHWAYS
        }

        # --------------------------------------------------------
        # Technology learning
        # --------------------------------------------------------

        self.cumulative_throughput = (
            INITIAL_CUMULATIVE_THROUGHPUT.copy()
        )

        self.processing_cost = (
            INITIAL_PROCESSING_COST.copy()
        )

        # --------------------------------------------------------
        # Solvolysis maturity
        # --------------------------------------------------------

        self.trl_solvolysis = float(
            INITIAL_TRL_SOLVOLYSIS
        )

        self.solvolysis_available = (
            self.trl_solvolysis
            >= TRL_THRESHOLD_SOLVOLYSIS - TRL_NUMERICAL_TOLERANCE
        )

        # --------------------------------------------------------
        # Recovered-material demand
        # --------------------------------------------------------

        self.open_loop_demand = float(
            BASE_OPEN_LOOP_DEMAND
        )

        self.closed_loop_demand = float(
            BASE_CLOSED_LOOP_DEMAND
        )

        self.new_blade_glass_fibre_demand = float(
            BASE_NEW_BLADE_GLASS_FIBRE_DEMAND
        )

        # Historical market-formation signals used by recycler
        # expectations in the next annual decision cycle.
        self.last_manufacturer_realized_adoption_share = 0.0
        self.last_manufacturer_adoption_readiness = 0.0
        self.last_effective_closed_loop_demand = 0.0
        self.last_realized_closed_loop_flow = 0.0
        self.last_closed_loop_supply_reliability = (
            INITIAL_OBSERVED_SUPPLY_RELIABILITY
        )
        self.last_closed_loop_demand_gap = 0.0
        self.last_open_loop_demand_gap = 0.0
        self.last_material_sales_ratio = (
            INITIAL_MATERIAL_SALES_RATIO.copy()
        )
        # Scenario-dependent solvolysis initial sales ratio:
        # EPR and mandate signals create advance purchase commitments.
        self.last_material_sales_ratio["solvolysis"] = 0.0

    # ============================================================
    # Solvolysis maturity
    # ============================================================

    def update_solvolysis_maturity(self):
        """
        Advance solvolysis maturity by one simulation period.

        The baseline maturity trajectory is modified by the explicit
        scenario-specific TRL acceleration parameter.

        Reaching the maturity threshold makes solvolysis commercially
        available, but does not automatically create capacity.
        """

        trl_growth = (
            BASE_TRL_GROWTH_SOLVOLYSIS
            + self.config.solvolysis_trl_acceleration_per_year
        )

        self.trl_solvolysis = min(
            9.0,
            self.trl_solvolysis
            + max(0.0, float(trl_growth)),
        )

        self.solvolysis_available = (
            self.trl_solvolysis
            >= TRL_THRESHOLD_SOLVOLYSIS - TRL_NUMERICAL_TOLERANCE
        )

        if not self.solvolysis_available:
            self.capacity["solvolysis"] = 0.0

    # ============================================================
    # Demand evolution
    # ============================================================

    def update_demands(self, year):
        """
        Update potential open-loop and closed-loop demand.

        Open-loop demand follows its common baseline growth rate.

        Closed-loop demand follows the greater of the baseline growth
        rate and the scenario-specific growth rate.

        Manufacturer adoption and recycled-content mandates activate
        this potential demand later in the material-allocation stage.
        """

        year = int(year)

        if year < START_YEAR:
            raise ValueError(
                f"year must be greater than or equal to {START_YEAR}."
            )

        years_since_start = year - START_YEAR

        self.open_loop_demand = (
            BASE_OPEN_LOOP_DEMAND
            * (
                1.0
                + OPEN_LOOP_DEMAND_GROWTH
            ) ** years_since_start
        )

        closed_loop_growth_rate = max(
            BASE_CLOSED_LOOP_DEMAND_GROWTH,
            self.config.closed_loop_demand_growth,
        )

        self.closed_loop_demand = (
            BASE_CLOSED_LOOP_DEMAND
            * (
                1.0
                + closed_loop_growth_rate
            ) ** years_since_start
        )

        self.new_blade_glass_fibre_demand = (
            BASE_NEW_BLADE_GLASS_FIBRE_DEMAND
            * (1.0 + NEW_BLADE_GLASS_FIBRE_DEMAND_GROWTH)
            ** years_since_start
        )

    # ============================================================
    # Processing costs
    # ============================================================

    def update_costs(self):
        """
        Update pathway-specific processing costs.

        Resource-recovery pathways experience learning-by-doing.
        Technology support reduces the effective initial cost of
        solvolysis.

        Reuse and repurposing retain fixed stylized costs.
        """

        for pathway in RESOURCE_RECOVERY_PATHWAYS:
            initial_cost = float(
                INITIAL_PROCESSING_COST[pathway]
            )

            effective_initial_cost = initial_cost

            self.processing_cost[pathway] = learning_cost(
                initial_cost=effective_initial_cost,
                cumulative_throughput=(
                    self.cumulative_throughput[pathway]
                ),
                reference_throughput=(
                    REFERENCE_CUMULATIVE_THROUGHPUT[pathway]
                ),
                learning_exponent=(
                    LEARNING_EXPONENT[pathway]
                ),
            )

        self.processing_cost["reuse"] = float(
            INITIAL_PROCESSING_COST["reuse"]
        )

        self.processing_cost["repurposing"] = float(
            INITIAL_PROCESSING_COST["repurposing"]
        )

    # ============================================================
    # Capacity evolution
    # ============================================================

    def update_capacity(self, abm_outputs):
        """
        Synchronize recycler-controlled capacity with active ABM agents.

        Mechanical recycling, pyrolysis, recovery, and solvolysis use the
        aggregated active recycler population as the single source of truth.
        Direct reuse and repurposing retain their aggregate SD capacities.
        """
        closing_recycler_capacity = abm_outputs.get(
            "closing_recycler_capacity_by_pathway"
        )

        if closing_recycler_capacity is not None:
            for pathway in RESOURCE_RECOVERY_PATHWAYS:
                capacity_value = float(
                    closing_recycler_capacity.get(pathway, 0.0)
                )

                if not np.isfinite(capacity_value):
                    raise ValueError(
                        f"Non-finite closing recycler capacity for {pathway}."
                    )

                self.capacity[pathway] = max(0.0, capacity_value)
        else:
            # Legacy fallback: accept incremental capacity_changes dict.
            # This path is retained for backward compatibility only;
            # production code should always supply closing_recycler_capacity_by_pathway.
            capacity_changes = abm_outputs.get("capacity_changes", {})

            for pathway in PATHWAYS:
                capacity_change = float(
                    capacity_changes.get(pathway, 0.0)
                )

                if not np.isfinite(capacity_change):
                    raise ValueError(
                        f"Non-finite capacity change for {pathway}."
                    )

                self.capacity[pathway] = max(
                    0.0,
                    self.capacity[pathway] + capacity_change,
                )

        if not self.solvolysis_available:
            self.capacity["solvolysis"] = 0.0

    def get_system_state(self, year):
        """
        Return the opening system state observed by ABM agents.

        Demand and processing costs are updated before agents make
        their current-period decisions.

        Solvolysis maturity is advanced only at the end of the period,
        so agents observe the maturity available at the beginning of
        the current year.
        """

        year = int(year)

        if year < START_YEAR or year > END_YEAR:
            raise ValueError(
                f"year must lie between {START_YEAR} and {END_YEAR}."
            )

        self.current_year = year

        self.update_demands(year)
        self.update_costs()

        available_pathways = [
            pathway
            for pathway in PATHWAYS
            if (
                pathway != "solvolysis"
                or self.solvolysis_available
            )
        ]

        total_available_capacity = sum(
            self.capacity[pathway]
            for pathway in available_pathways
        )

        return {
            "year": year,

            # Regulatory state
            "landfill_available": (
                self.config.landfill_allowed
            ),

            # Available pathways
            "available_pathways": available_pathways,

            # Annual and cumulative WTB state
            "annual_decommissioned": (
                decommissioned_wtb_inflow(year)
            ),
            "untreated_stock": self.untreated_stock,
            "cumulative_decommissioned": (
                self.cumulative_decommissioned
            ),
            "cumulative_treated": (
                self.cumulative_treated
            ),

            # Capacity
            "capacity": self.capacity.copy(),
            "total_available_capacity": (
                total_available_capacity
            ),

            # Costs and maturity
            "processing_cost": (
                self.processing_cost.copy()
            ),
            "trl_solvolysis": self.trl_solvolysis,
            "solvolysis_available": (
                self.solvolysis_available
            ),

            # Market state
            "open_loop_demand": self.open_loop_demand,
            "closed_loop_demand": self.closed_loop_demand,
            "new_blade_glass_fibre_demand": (
                self.new_blade_glass_fibre_demand
            ),
            "last_manufacturer_realized_adoption_share": (
                self.last_manufacturer_realized_adoption_share
            ),
            "last_manufacturer_adoption_readiness": (
                self.last_manufacturer_adoption_readiness
            ),
            "last_effective_closed_loop_demand": (
                self.last_effective_closed_loop_demand
            ),
            "last_realized_closed_loop_flow": (
                self.last_realized_closed_loop_flow
            ),
            "last_closed_loop_supply_reliability": (
                self.last_closed_loop_supply_reliability
            ),
            "last_closed_loop_demand_gap": (
                self.last_closed_loop_demand_gap
            ),
            "last_open_loop_demand_gap": (
                self.last_open_loop_demand_gap
            ),
            "last_material_sales_ratio": (
                self.last_material_sales_ratio.copy()
            ),
            "recovered_stock": (
                self.recovered_stock.copy()
            ),

            # Learning state
            "cumulative_throughput": (
                self.cumulative_throughput.copy()
            ),
        }

    # ============================================================
    # Annual stock-flow update
    # ============================================================

    def update(self, abm_outputs, year):
        """
        Update SD stocks and flows using current-period ABM outputs.

        Sequence:

        1. validate treatment flows;
        2. enforce opening capacity and material availability;
        3. update untreated WTB stock;
        4. generate recovered material;
        5. allocate material to closed- and open-loop uses;
        6. update recovered-material inventories;
        7. update treatment capacity;
        8. advance solvolysis maturity.
        """

        year = int(year)

        if year < START_YEAR or year > END_YEAR:
            raise ValueError(
                f"year must lie between {START_YEAR} and {END_YEAR}."
            )

        self.current_year = year

        decommissioned = decommissioned_wtb_inflow(
            year
        )

        opening_untreated_stock = float(
            self.untreated_stock
        )

        opening_trl_solvolysis = float(
            self.trl_solvolysis
        )

        opening_solvolysis_available = bool(
            self.solvolysis_available
        )

        opening_capacity = self.capacity.copy()

        opening_recovered_stock = (
            self.recovered_stock.copy()
        )

        self.cumulative_decommissioned += (
            decommissioned
        )

        # --------------------------------------------------------
        # Treatment flows supplied by ABM
        # --------------------------------------------------------

        raw_actual_flows = abm_outputs.get(
            "actual_flows",
            {},
        )

        actual_flows = {}

        for pathway in PATHWAYS:
            pathway_flow = float(
                raw_actual_flows.get(
                    pathway,
                    0.0,
                )
            )

            if not np.isfinite(pathway_flow):
                raise ValueError(
                    f"Non-finite treatment flow for {pathway}."
                )

            actual_flows[pathway] = max(
                0.0,
                pathway_flow,
            )

        # Solvolysis cannot treat material before commercial maturity.
        if not opening_solvolysis_available:
            actual_flows["solvolysis"] = 0.0

        # Current treatment cannot exceed opening capacity because
        # current-period investments become available next period.
        for pathway in PATHWAYS:
            actual_flows[pathway] = min(
                actual_flows[pathway],
                max(
                    0.0,
                    opening_capacity[pathway],
                ),
            )

        total_treated = sum(
            actual_flows.values()
        )

        # Available WTB material includes the annual inflow and the
        # stock accumulated in previous periods.
        total_available_wtb = (
            opening_untreated_stock
            + decommissioned
        )

        # This is a safeguard against inconsistent ABM output.
        if (
            total_treated
            > total_available_wtb
            + NUMERICAL_EPSILON
        ):
            scaling_factor = safe_divide(
                total_available_wtb,
                total_treated,
                default=0.0,
            )

            actual_flows = {
                pathway: flow * scaling_factor
                for pathway, flow
                in actual_flows.items()
            }

            total_treated = sum(
                actual_flows.values()
            )

        self.cumulative_treated += (
            total_treated
        )

        # --------------------------------------------------------
        # Untreated WTB stock
        # --------------------------------------------------------

        self.untreated_stock = max(
            0.0,
            total_available_wtb
            - total_treated,
        )

        untreated_stock_change = (
            self.untreated_stock
            - opening_untreated_stock
        )

        net_untreated_stock_increase = max(
            0.0,
            untreated_stock_change,
        )

        stock_drawdown = max(
            0.0,
            -untreated_stock_change,
        )

        # --------------------------------------------------------
        # Recovered-material generation
        # --------------------------------------------------------

        recovered_generation = {
            pathway: 0.0
            for pathway in MATERIAL_RECOVERY_PATHWAYS
        }

        for pathway in MATERIAL_RECOVERY_PATHWAYS:
            processed_flow = actual_flows[pathway]

            recovered_generation[pathway] = (
                processed_flow
                * resource_output_yield(pathway)
            )

            self.cumulative_throughput[pathway] += (
                processed_flow
            )

        # Recovery produces a resource credit but does not generate
        # a conventional secondary-material inventory.
        recovery_processed = actual_flows.get(
            "recovery",
            0.0,
        )

        self.cumulative_throughput["recovery"] += (
            recovery_processed
        )

        recovery_resource_credit = (
            recovery_processed
            * resource_output_yield("recovery")
        )

        expected_solvolysis_generation = (
            actual_flows["solvolysis"]
            * RECOVERABLE_FEEDSTOCK_FRACTION["solvolysis"]
            * RECOVERY_EFFICIENCY["solvolysis"]
        )

        if not np.isclose(
            recovered_generation["solvolysis"],
            expected_solvolysis_generation,
            atol=NUMERICAL_EPSILON,
            rtol=1e-9,
        ):
            raise AssertionError(
                "Solvolysis recovered output is inconsistent with "
                "whole-blade flow, glass-fibre mass fraction, and recovery efficiency."
            )

        # --------------------------------------------------------
        # Total recovered material available during the period
        # --------------------------------------------------------

        inventory_losses = {
            pathway: (
                opening_recovered_stock[pathway]
                * RECOVERED_STOCK_DECAY_RATE[pathway]
            )
            for pathway in MATERIAL_RECOVERY_PATHWAYS
        }

        marketable_opening_recovered_stock = {
            pathway: max(
                0.0,
                opening_recovered_stock[pathway]
                - inventory_losses[pathway],
            )
            for pathway in MATERIAL_RECOVERY_PATHWAYS
        }

        available_recovered_material = {
            pathway: (
                marketable_opening_recovered_stock[pathway]
                + recovered_generation[pathway]
            )
            for pathway in MATERIAL_RECOVERY_PATHWAYS
        }

        # --------------------------------------------------------
        # Closed-loop and open-loop utilization
        # --------------------------------------------------------

        manufacturer_adoption = bounded(
            abm_outputs.get(
                "manufacturer_realized_adoption_share",
                abm_outputs.get(
                    "manufacturer_adoption_readiness",
                    0.0,
                ),
            )
        )

        utilization_results = (
            compute_open_closed_utilization(
                recovered_material=(
                    available_recovered_material
                ),
                open_loop_demand=(
                    self.open_loop_demand
                ),
                closed_loop_demand=(
                    self.closed_loop_demand
                ),
                new_blade_glass_fibre_demand=(
                    self.new_blade_glass_fibre_demand
                ),
                manufacturer_adoption=(
                    manufacturer_adoption
                ),
                recycled_content_mandate=(
                    self.config.recycled_content_mandate
                ),
                pathway_certification_score={
                    "mechanical_recycling": 1.0,
                    "pyrolysis": 1.0,
                    "solvolysis": logistic(
                        CERTIFICATION_SENSITIVITY
                        * (
                            opening_trl_solvolysis
                            - TRL_THRESHOLD_SOLVOLYSIS
                        )
                    ),
                },
            )
        )

        open_loop_utilization = (
            utilization_results[
                "open_loop_use_by_pathway"
            ]
        )

        closed_loop_utilization = (
            utilization_results[
                "closed_loop_use_by_pathway"
            ]
        )

        # Persist realized market-formation signals for next-year
        # recycler expectations.
        self.last_manufacturer_realized_adoption_share = (
            manufacturer_adoption
        )
        self.last_manufacturer_adoption_readiness = bounded(
            abm_outputs.get("manufacturer_adoption_readiness", 0.0)
        )
        self.last_effective_closed_loop_demand = float(
            utilization_results[
                "effective_closed_loop_demand"
            ]
        )
        self.last_realized_closed_loop_flow = float(
            sum(closed_loop_utilization.values())
        )

        self.last_closed_loop_demand_gap = max(
            0.0,
            self.last_effective_closed_loop_demand
            - self.last_realized_closed_loop_flow,
        )

        if self.last_effective_closed_loop_demand > NUMERICAL_EPSILON:
            observed_delivery_ratio = bounded(
                self.last_realized_closed_loop_flow
                / self.last_effective_closed_loop_demand
            )
            self.last_closed_loop_supply_reliability = bounded(
                (1.0 - SUPPLY_RELIABILITY_SMOOTHING)
                * self.last_closed_loop_supply_reliability
                + SUPPLY_RELIABILITY_SMOOTHING
                * observed_delivery_ratio
            )

        self.last_open_loop_demand_gap = max(
            0.0,
            self.open_loop_demand
            - float(utilization_results["total_open_loop_use"]),
        )

        realized_material_sales = {
            pathway: (
                float(open_loop_utilization.get(pathway, 0.0))
                + float(closed_loop_utilization.get(pathway, 0.0))
            )
            for pathway in MATERIAL_RECOVERY_PATHWAYS
        }

        current_material_sales_ratio = {}

        for pathway in MATERIAL_RECOVERY_PATHWAYS:
            available_quantity = available_recovered_material[pathway]

            if available_quantity > NUMERICAL_EPSILON:
                observed_ratio = bounded(
                    safe_divide(
                        realized_material_sales[pathway],
                        available_quantity,
                        default=0.0,
                    )
                )

                smoothed_ratio = (
                    (1.0 - MATERIAL_SALES_RATIO_SMOOTHING)
                    * self.last_material_sales_ratio[pathway]
                    + MATERIAL_SALES_RATIO_SMOOTHING
                    * observed_ratio
                )

                self.last_material_sales_ratio[pathway] = bounded(
                    smoothed_ratio
                )

            current_material_sales_ratio[pathway] = (
                self.last_material_sales_ratio[pathway]
            )

        # Resource recovery does not create a stored secondary material.
        self.last_material_sales_ratio["recovery"] = 1.0
        current_material_sales_ratio["recovery"] = 1.0

        unutilized_material = (
            utilization_results[
                "unutilized_material_by_pathway"
            ]
        )

        # Unutilized recovered material becomes the closing inventory
        # and can be supplied to downstream demand in later periods.
        self.recovered_stock = {
            pathway: max(
                0.0,
                float(
                    unutilized_material.get(
                        pathway,
                        0.0,
                    )
                ),
            )
            for pathway in MATERIAL_RECOVERY_PATHWAYS
        }

        # --------------------------------------------------------
        # Capacity evolution
        # --------------------------------------------------------

        self.update_capacity(
            abm_outputs=abm_outputs
        )

        # --------------------------------------------------------
        # End-of-period maturity update
        # --------------------------------------------------------

        self.update_solvolysis_maturity()

        return {
            "year": year,

            # Opening state
            "opening_untreated_stock": (
                opening_untreated_stock
            ),
            "opening_trl_solvolysis": (
                opening_trl_solvolysis
            ),
            "opening_solvolysis_available": (
                opening_solvolysis_available
            ),
            "opening_capacity": opening_capacity,
            "opening_recovered_stock": (
                opening_recovered_stock
            ),

            # Annual WTB flows
            "decommissioned": decommissioned,
            "actual_flows": actual_flows,
            "total_treated": total_treated,

            # Closing WTB stock
            "closing_untreated_stock": (
                self.untreated_stock
            ),
            "untreated_stock_change": (
                untreated_stock_change
            ),
            "net_untreated_stock_increase": (
                net_untreated_stock_increase
            ),
            "stock_drawdown": stock_drawdown,

            # Recovered materials
            "recovered_generation": (
                recovered_generation
            ),
            "available_recovered_material": (
                available_recovered_material
            ),
            "closing_recovered_stock": (
                self.recovered_stock.copy()
            ),
            "inventory_losses": inventory_losses,
            "material_sales_ratio_by_pathway": (
                current_material_sales_ratio
            ),
            "recovery_resource_credit": (
                recovery_resource_credit
            ),

            # Material utilization
            "open_loop_utilization_by_pathway": (
                open_loop_utilization
            ),
            "closed_loop_utilization_by_pathway": (
                closed_loop_utilization
            ),
            "unutilized_material_by_pathway": (
                unutilized_material
            ),
            "total_open_loop_utilization": (
                utilization_results[
                    "total_open_loop_use"
                ]
            ),
            "total_closed_loop_utilization": (
                utilization_results[
                    "realized_closed_loop_use"
                ]
            ),
            "total_unutilized_material": (
                utilization_results[
                    "total_unutilized_material"
                ]
            ),
            "high_quality_recovered_material": (
                utilization_results[
                    "high_quality_recovered_material"
                ]
            ),
            "closed_loop_eligibility_by_pathway": (
                utilization_results[
                    "closed_loop_eligibility_by_pathway"
                ]
            ),
            "eligible_closed_loop_supply_by_pathway": (
                utilization_results[
                    "eligible_closed_loop_supply_by_pathway"
                ]
            ),

            # Demand activation and realization
            "potential_closed_loop_demand": (
                utilization_results[
                    "potential_closed_loop_demand"
                ]
            ),
            "new_blade_glass_fibre_demand": (
                utilization_results[
                    "new_blade_glass_fibre_demand"
                ]
            ),
            "adoption_activated_closed_loop_demand": (
                utilization_results[
                    "adoption_activated_closed_loop_demand"
                ]
            ),
            "mandate_activated_closed_loop_demand": (
                utilization_results[
                    "mandate_activated_closed_loop_demand"
                ]
            ),
            "voluntary_closed_loop_demand": (
                utilization_results[
                    "voluntary_closed_loop_demand"
                ]
            ),
            "mandatory_closed_loop_demand": (
                utilization_results[
                    "mandatory_closed_loop_demand"
                ]
            ),
            "effective_closed_loop_demand": (
                utilization_results[
                    "effective_closed_loop_demand"
                ]
            ),
            "realized_voluntary_closed_loop_use": (
                utilization_results[
                    "realized_voluntary_closed_loop_use"
                ]
            ),
            "realized_mandatory_closed_loop_use": (
                utilization_results[
                    "realized_mandatory_closed_loop_use"
                ]
            ),
            "unmet_voluntary_closed_loop_demand": (
                utilization_results[
                    "unmet_voluntary_closed_loop_demand"
                ]
            ),
            "unmet_mandatory_closed_loop_demand": (
                utilization_results[
                    "unmet_mandatory_closed_loop_demand"
                ]
            ),
            "total_unmet_closed_loop_demand": (
                utilization_results[
                    "total_unmet_closed_loop_demand"
                ]
            ),
            "mandatory_compliance_ratio": (
                utilization_results[
                    "mandatory_compliance_ratio"
                ]
            ),
            "closed_loop_supply_coverage_ratio": (
                utilization_results[
                    "closed_loop_supply_coverage_ratio"
                ]
            ),

            # Closing technology state
            "closing_trl_solvolysis": (
                self.trl_solvolysis
            ),
            "closing_solvolysis_available": (
                self.solvolysis_available
            ),
            "closing_capacity": (
                self.capacity.copy()
            ),
        }

    # ============================================================
    # Performance indicators
    # ============================================================

    def compute_indicators(
        self,
        abm_outputs,
        sd_update_outputs,
    ):
        """
        Compute annual technology, circularity, capacity,
        economic, and environmental indicators.

        Configuration classification is intentionally excluded and
        should be performed during result post-processing.
        """

        actual_flows = (
            sd_update_outputs["actual_flows"]
        )

        decommissioned = float(
            sd_update_outputs["decommissioned"]
        )

        total_treated = float(
            sd_update_outputs["total_treated"]
        )

        opening_capacity = (
            sd_update_outputs["opening_capacity"]
        )

        open_loop_utilization = (
            sd_update_outputs[
                "open_loop_utilization_by_pathway"
            ]
        )

        closed_loop_utilization = (
            sd_update_outputs[
                "closed_loop_utilization_by_pathway"
            ]
        )

        total_open_loop_utilization = sum(
            open_loop_utilization.values()
        )

        total_closed_loop_utilization = sum(
            closed_loop_utilization.values()
        )

        total_material_utilization = (
            total_open_loop_utilization
            + total_closed_loop_utilization
        )

        total_available_material = sum(
            sd_update_outputs[
                "available_recovered_material"
            ].values()
        )

        total_unutilized_material = sum(
            sd_update_outputs[
                "unutilized_material_by_pathway"
            ].values()
        )

        # --------------------------------------------------------
        # Untreated stock indicators
        # --------------------------------------------------------

        untreated_stock_share = safe_divide(
            self.untreated_stock,
            self.cumulative_decommissioned,
            default=0.0,
        )

        net_untreated_stock_increase = (
            sd_update_outputs[
                "net_untreated_stock_increase"
            ]
        )

        net_untreated_stock_increase_share = (
            safe_divide(
                net_untreated_stock_increase,
                decommissioned,
                default=0.0,
            )
        )

        # --------------------------------------------------------
        # Recovered-material utilization indicators
        # --------------------------------------------------------

        recovered_material_utilization_share = (
            safe_divide(
                total_material_utilization,
                total_available_material,
                default=0.0,
            )
        )

        unutilized_material_share = safe_divide(
            total_unutilized_material,
            total_available_material,
            default=0.0,
        )

        open_loop_utilization_share = safe_divide(
            total_open_loop_utilization,
            total_material_utilization,
            default=0.0,
        )

        closed_loop_utilization_share = safe_divide(
            total_closed_loop_utilization,
            total_material_utilization,
            default=0.0,
        )

        # --------------------------------------------------------
        # Treatment pathway indicators
        # --------------------------------------------------------

        direct_reuse_flow = sum(
            actual_flows.get(
                pathway,
                0.0,
            )
            for pathway in DIRECT_REUSE_PATHWAYS
        )

        # Incumbent open-loop material recycling includes only
        # mechanical recycling and pyrolysis. Energy recovery is
        # reported separately because it does not recirculate material.
        incumbent_open_loop_material_flow = sum(
            actual_flows.get(
                pathway,
                0.0,
            )
            for pathway in INCUMBENT_OPEN_LOOP_MATERIAL_PATHWAYS
        )

        energy_recovery_flow = sum(
            actual_flows.get(
                pathway,
                0.0,
            )
            for pathway in ENERGY_RECOVERY_PATHWAYS
        )

        solvolysis_treated_flow = actual_flows.get(
            "solvolysis",
            0.0,
        )

        direct_reuse_share = safe_divide(
            direct_reuse_flow,
            total_treated,
            default=0.0,
        )

        incumbent_open_loop_material_treatment_share = (
            safe_divide(
                incumbent_open_loop_material_flow,
                total_treated,
                default=0.0,
            )
        )

        energy_recovery_treatment_share = safe_divide(
            energy_recovery_flow,
            total_treated,
            default=0.0,
        )

        solvolysis_treatment_share = safe_divide(
            solvolysis_treated_flow,
            total_treated,
            default=0.0,
        )

        # --------------------------------------------------------
        # Closed-loop equivalent WTB flow
        # --------------------------------------------------------

        closed_loop_material_flow = (
            closed_loop_utilization.get(
                "solvolysis",
                0.0,
            )
        )

        closed_loop_equivalent_wtb_flow = (
            safe_divide(
                closed_loop_material_flow,
                resource_output_yield("solvolysis"),
                default=0.0,
            )
        )

        closed_loop_wtb_flow_share = safe_divide(
            closed_loop_equivalent_wtb_flow,
            total_treated,
            default=0.0,
        )

        # --------------------------------------------------------
        # Annual inflow ratios
        # --------------------------------------------------------

        annual_direct_reuse_to_inflow_ratio = (
            safe_divide(
                direct_reuse_flow,
                decommissioned,
                default=0.0,
            )
        )

        annual_open_loop_material_to_inflow_ratio = (
            safe_divide(
                incumbent_open_loop_material_flow,
                decommissioned,
                default=0.0,
            )
        )

        annual_energy_recovery_to_inflow_ratio = (
            safe_divide(
                energy_recovery_flow,
                decommissioned,
                default=0.0,
            )
        )

        annual_solvolysis_to_inflow_ratio = (
            safe_divide(
                solvolysis_treated_flow,
                decommissioned,
                default=0.0,
            )
        )

        annual_treatment_to_inflow_ratio = (
            safe_divide(
                total_treated,
                decommissioned,
                default=0.0,
            )
        )

        # --------------------------------------------------------
        # Virgin-material displacement
        # --------------------------------------------------------

        virgin_material_displacement = 0.0

        for pathway in MATERIAL_RECOVERY_PATHWAYS:
            utilized_material = (
                open_loop_utilization.get(
                    pathway,
                    0.0,
                )
                + closed_loop_utilization.get(
                    pathway,
                    0.0,
                )
            )

            virgin_material_displacement += (
                utilized_material
                * SUBSTITUTION_FACTOR[pathway]
            )

        virgin_material_displacement += (
            sd_update_outputs[
                "recovery_resource_credit"
            ]
            * SUBSTITUTION_FACTOR["recovery"]
        )

        # --------------------------------------------------------
        # Avoided environmental-impact index
        # --------------------------------------------------------

        exploratory_environmental_proxy = 0.0

        for pathway in MATERIAL_RECOVERY_PATHWAYS:
            utilized_material = (
                open_loop_utilization.get(
                    pathway,
                    0.0,
                )
                + closed_loop_utilization.get(
                    pathway,
                    0.0,
                )
            )

            exploratory_environmental_proxy += (
                utilized_material
                * (
                    ENVIRONMENTAL_INTENSITY_VIRGIN[pathway]
                    - ENVIRONMENTAL_INTENSITY_RECOVERED[pathway]
                )
            )

        exploratory_environmental_proxy += (
            sd_update_outputs[
                "recovery_resource_credit"
            ]
            * (
                ENVIRONMENTAL_INTENSITY_VIRGIN["recovery"]
                - ENVIRONMENTAL_INTENSITY_RECOVERED["recovery"]
            )
        )

        # --------------------------------------------------------
        # Average processing cost
        # --------------------------------------------------------

        weighted_processing_cost = sum(
            actual_flows[pathway]
            * self.processing_cost[pathway]
            for pathway in PATHWAYS
        )

        average_processing_cost = safe_divide(
            weighted_processing_cost,
            total_treated,
            default=0.0,
        )

        # --------------------------------------------------------
        # Capacity indicators
        # --------------------------------------------------------

        total_opening_capacity = sum(
            opening_capacity.values()
        )

        total_closing_capacity = sum(
            self.capacity.values()
        )

        treatment_capacity_utilization = (
            safe_divide(
                total_treated,
                total_opening_capacity,
                default=0.0,
            )
        )

        total_available_wtb = (
            decommissioned
            + sd_update_outputs[
                "opening_untreated_stock"
            ]
        )

        opening_capacity_gap = max(
            0.0,
            total_available_wtb
            - total_opening_capacity,
        )

        # --------------------------------------------------------
        # Collect indicators
        # --------------------------------------------------------

        return {
            # Time
            "year": sd_update_outputs["year"],

            # WTB inflow, stocks, and treatment
            "decommissioned": decommissioned,
            "total_treated": total_treated,
            "cumulative_decommissioned": (
                self.cumulative_decommissioned
            ),
            "cumulative_treated": (
                self.cumulative_treated
            ),
            "untreated_stock": (
                self.untreated_stock
            ),
            "untreated_stock_share": (
                untreated_stock_share
            ),
            "untreated_stock_change": (
                sd_update_outputs[
                    "untreated_stock_change"
                ]
            ),
            "net_untreated_stock_increase": (
                net_untreated_stock_increase
            ),
            "net_untreated_stock_increase_share": (
                net_untreated_stock_increase_share
            ),
            "stock_drawdown": (
                sd_update_outputs["stock_drawdown"]
            ),

            # Capacity
            "total_opening_capacity": (
                total_opening_capacity
            ),
            "total_closing_capacity": (
                total_closing_capacity
            ),
            "opening_capacity_gap": (
                opening_capacity_gap
            ),
            "treatment_capacity_utilization": (
                treatment_capacity_utilization
            ),

            # Material utilization
            "total_available_recovered_material": (
                total_available_material
            ),
            "total_open_loop_utilization": (
                total_open_loop_utilization
            ),
            "total_closed_loop_utilization": (
                total_closed_loop_utilization
            ),
            "total_unutilized_material": (
                total_unutilized_material
            ),
            "recovered_material_utilization_share": (
                recovered_material_utilization_share
            ),
            "unutilized_material_share": (
                unutilized_material_share
            ),
            "open_loop_utilization_share": (
                open_loop_utilization_share
            ),
            "closed_loop_utilization_share": (
                closed_loop_utilization_share
            ),

            # Treatment pathway diffusion
            "direct_reuse_flow": direct_reuse_flow,
            "direct_reuse_share": direct_reuse_share,
            "incumbent_open_loop_material_flow": (
                incumbent_open_loop_material_flow
            ),
            "incumbent_open_loop_material_treatment_share": (
                incumbent_open_loop_material_treatment_share
            ),
            "energy_recovery_flow": energy_recovery_flow,
            "energy_recovery_treatment_share": (
                energy_recovery_treatment_share
            ),
            "solvolysis_treated_flow": (
                solvolysis_treated_flow
            ),
            "solvolysis_treatment_share": (
                solvolysis_treatment_share
            ),
            "solvolysis_recoverable_glass_fibre_input": (
                solvolysis_treated_flow
                * RECOVERABLE_FEEDSTOCK_FRACTION["solvolysis"]
            ),
            "solvolysis_recovery_efficiency": (
                RECOVERY_EFFICIENCY["solvolysis"]
            ),
            "solvolysis_whole_wtb_output_yield": (
                resource_output_yield("solvolysis")
            ),

            # Closed-loop outcome
            "closed_loop_material_flow": (
                closed_loop_material_flow
            ),
            "closed_loop_equivalent_wtb_flow": (
                closed_loop_equivalent_wtb_flow
            ),
            "closed_loop_wtb_flow_share": (
                closed_loop_wtb_flow_share
            ),

            # Annual inflow ratios
            "annual_direct_reuse_to_inflow_ratio": (
                annual_direct_reuse_to_inflow_ratio
            ),
            "annual_open_loop_material_to_inflow_ratio": (
                annual_open_loop_material_to_inflow_ratio
            ),
            "annual_energy_recovery_to_inflow_ratio": (
                annual_energy_recovery_to_inflow_ratio
            ),
            "annual_solvolysis_to_inflow_ratio": (
                annual_solvolysis_to_inflow_ratio
            ),
            "annual_treatment_to_inflow_ratio": (
                annual_treatment_to_inflow_ratio
            ),

            # Demand
            "open_loop_demand": (
                self.open_loop_demand
            ),
            "potential_closed_loop_demand": (
                self.closed_loop_demand
            ),
            "effective_closed_loop_demand": (
                sd_update_outputs[
                    "effective_closed_loop_demand"
                ]
            ),
            "high_quality_recovered_material": (
                sd_update_outputs[
                    "high_quality_recovered_material"
                ]
            ),
            "adoption_activated_closed_loop_demand": (
                sd_update_outputs[
                    "adoption_activated_closed_loop_demand"
                ]
            ),
            "mandate_activated_closed_loop_demand": (
                sd_update_outputs[
                    "mandate_activated_closed_loop_demand"
                ]
            ),

            # Economic outcome and internal environmental diagnostic
            "virgin_material_displacement": (
                virgin_material_displacement
            ),
            "exploratory_environmental_proxy": (
                exploratory_environmental_proxy
            ),
            "average_processing_cost": (
                average_processing_cost
            ),

            # Opening technology maturity
            "opening_trl_solvolysis": (
                sd_update_outputs[
                    "opening_trl_solvolysis"
                ]
            ),
            "opening_solvolysis_available": (
                sd_update_outputs[
                    "opening_solvolysis_available"
                ]
            ),

            # Closing technology maturity
            "trl_solvolysis": (
                self.trl_solvolysis
            ),
            "solvolysis_available": (
                self.solvolysis_available
            ),

            # ABM summary output
            "manufacturer_realized_adoption_share": (
                float(
                    abm_outputs.get(
                        "manufacturer_realized_adoption_share",
                        abm_outputs.get(
                            "manufacturer_adoption_readiness",
                            0.0,
                        ),
                    )
                )
            ),
        }


# ============================================================
# Basic SD-layer initialization checks
# ============================================================

_test_scenario_name = next(
    iter(SCENARIOS)
)

_test_sd_layer = SDLayer(
    scenario_config=SCENARIOS[_test_scenario_name]
)

_test_state = _test_sd_layer.get_system_state(
    START_YEAR
)

assert _test_state["year"] == START_YEAR

assert np.isclose(
    _test_state["annual_decommissioned"],
    BASE_DECOMMISSIONED_WTB,
)

assert set(_test_state["capacity"]) == set(PATHWAYS)

assert set(
    _test_state["recovered_stock"]
) == set(MATERIAL_RECOVERY_PATHWAYS)

assert (
    _test_state["capacity"]["solvolysis"]
    == 0.0
)

assert (
    "solvolysis"
    in _test_state["available_pathways"]
) == _test_state["solvolysis_available"]


In [ ]:
# ============================================================
# 7. Agent-Based Model layer
# ============================================================

class ABMLayer:
    """
    Agent-Based Model layer of the hybrid framework.

    The ABM represents heterogeneous decisions by:

    - wind farm operators, which allocate EoL blade flows;
    - recyclers, which evaluate technology profitability and capacity changes;
    - manufacturers/end-users, which decide whether to adopt recovered materials.

    Agent-level decisions are aggregated before being transferred to the
    System Dynamics layer.
    """

    def __init__(
        self,
        scenario_config,
        rng=None,
        n_operators=N_OPERATORS,
        n_recyclers=N_RECYCLERS,
        n_manufacturers=N_MANUFACTURERS,
    ):
        self.config = scenario_config
        self.rng = rng if rng is not None else np.random.default_rng()

        self.n_operators = int(n_operators)
        self.n_recyclers = int(n_recyclers)
        self.n_manufacturers = int(n_manufacturers)

        if self.n_operators <= 0:
            raise ValueError("n_operators must be positive.")

        if self.n_recyclers <= 0:
            raise ValueError("n_recyclers must be positive.")

        if self.n_manufacturers <= 0:
            raise ValueError("n_manufacturers must be positive.")

        self._initialize_operator_population()
        self._initialize_recycler_population()
        self._initialize_manufacturer_population()

        self.solvolysis_entry_signal_years = 0


    # ============================================================
    # Population initialization
    # ============================================================

    def _initialize_operator_population(self):
        """
        Creates heterogeneous operator preferences.

        Cost orientation is drawn around the population mean.
        Environmental orientation is defined as its complement.
        """
        concentration = 20.0

        alpha = (
            MEAN_OPERATOR_COST_WEIGHT
            * concentration
        )

        beta = (
            1.0 - MEAN_OPERATOR_COST_WEIGHT
        ) * concentration

        cost_weights = self.rng.beta(
            alpha,
            beta,
            size=self.n_operators,
        )

        choice_sensitivities = np.clip(
            self.rng.normal(
                loc=MEAN_OPERATOR_CHOICE_SENSITIVITY,
                scale=0.35,
                size=self.n_operators,
            ),
            0.50,
            5.00,
        )

        self.operators = pd.DataFrame({
            "operator_id": np.arange(self.n_operators),
            "cost_weight": cost_weights,
            "environment_weight": 1.0 - cost_weights,
            "choice_sensitivity": choice_sensitivities,
        })


    def _initialize_recycler_population(self):
        """
        Create heterogeneous recyclers while guaranteeing representation of
        every incumbent technology with positive initial SD capacity.
        """
        incumbent_technologies = np.array(
            INCUMBENT_TREATMENT_PATHWAYS,
            dtype=object,
        )

        weights = np.array([
            INITIAL_PATHWAY_SHARES[pathway]
            for pathway in incumbent_technologies
        ], dtype=float)
        weights = weights / weights.sum()

        if self.n_recyclers >= len(incumbent_technologies):
            remaining_agents = (
                self.n_recyclers
                - len(incumbent_technologies)
            )
            raw_counts = weights * remaining_agents
            counts = np.floor(raw_counts).astype(int) + 1

            residual = self.n_recyclers - int(counts.sum())
            fractional_order = np.argsort(
                -(raw_counts - np.floor(raw_counts))
            )

            for position in range(residual):
                counts[fractional_order[position]] += 1

            technologies = np.concatenate([
                np.repeat(pathway, count)
                for pathway, count
                in zip(incumbent_technologies, counts)
            ])
            self.rng.shuffle(technologies)
        else:
            technologies = self.rng.choice(
                incumbent_technologies,
                size=self.n_recyclers,
                replace=False,
                p=weights,
            )

        switching_thresholds = np.clip(
            self.rng.normal(
                loc=ABM_RECYCLER_SWITCHING_THRESHOLD_MEAN,
                scale=ABM_RECYCLER_SWITCHING_THRESHOLD_STD,
                size=self.n_recyclers,
            ),
            ABM_RECYCLER_SWITCHING_THRESHOLD_MIN,
            None,
        )

        discount_rates = np.clip(
            self.rng.normal(
                loc=MEAN_RECYCLER_DISCOUNT_RATE,
                scale=STD_RECYCLER_DISCOUNT_RATE,
                size=self.n_recyclers,
            ),
            0.02,
            0.20,
        )

        exit_thresholds = np.clip(
            self.rng.normal(
                loc=MEAN_RECYCLER_EXIT_THRESHOLD,
                scale=STD_RECYCLER_EXIT_THRESHOLD,
                size=self.n_recyclers,
            ),
            0.01,
            0.50,
        )

        compatibility_scores = np.array([
            SOLVOLYSIS_RETROFIT_COMPATIBILITY[technology]
            for technology in technologies
        ], dtype=float)

        self.recyclers = pd.DataFrame({
            "recycler_id": np.arange(self.n_recyclers),
            "technology": technologies,
            "capacity": np.zeros(self.n_recyclers),
            "switching_threshold": switching_thresholds,
            "discount_rate": discount_rates,
            "exit_threshold": exit_thresholds,
            "retrofit_compatibility": compatibility_scores,
            "active": True,
            "underperformance_years": 0,
            "positive_expansion_signal_years": 0,
            "positive_switch_signal_years": 0,
            "switch_advantage_ema": 0.0,
            "utilization_ema": 0.0,
            "profit_ema": 0.0,
            "sales_ratio_ema": 0.0,
            "last_switch_year": START_YEAR - SWITCH_COOLDOWN_YEARS,
            "last_expansion_year": START_YEAR - EXPANSION_COOLDOWN_YEARS,
        })

        # Allocate each pathway's exact aggregate initial capacity across
        # the agents assigned to that technology.
        for pathway in incumbent_technologies:
            mask = self.recyclers["technology"] == pathway
            number_in_pathway = int(mask.sum())

            if number_in_pathway <= 0:
                raise RuntimeError(
                    f"No recycler was initialized for {pathway}."
                )

            individual_capacity = (
                INITIAL_CAPACITY[pathway]
                / number_in_pathway
            )

            self.recyclers.loc[mask, "capacity"] = (
                individual_capacity
            )

    def aggregate_active_capacity(self):
        """Return active recycler capacity by recovery pathway."""
        aggregate = {
            pathway: 0.0
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        }

        active = self.recyclers.loc[
            self.recyclers["active"]
        ]

        grouped = active.groupby("technology")["capacity"].sum()

        for pathway, capacity in grouped.items():
            if pathway in aggregate:
                aggregate[pathway] = max(0.0, float(capacity))

        return aggregate

    def _initialize_manufacturer_population(self):
        """
        Creates heterogeneous manufacturer adoption thresholds
        and technical-risk perceptions.
        """
        adoption_thresholds = np.clip(
            self.rng.normal(
                loc=MEAN_MANUFACTURER_ADOPTION_THRESHOLD,
                scale=0.10,
                size=self.n_manufacturers,
            ),
            0.05,
            0.95,
        )

        risk_perceptions = np.clip(
            self.rng.normal(
                loc=MEAN_PERCEIVED_TECHNICAL_RISK,
                scale=0.08,
                size=self.n_manufacturers,
            ),
            0.05,
            0.90,
        )

        adoption_sensitivities = np.clip(
            self.rng.normal(
                loc=MEAN_MANUFACTURER_ADOPTION_SENSITIVITY,
                scale=0.60,
                size=self.n_manufacturers,
            ),
            1.0,
            10.0,
        )

        self.manufacturers = pd.DataFrame({
            "manufacturer_id": np.arange(self.n_manufacturers),
            "adoption_threshold": adoption_thresholds,
            "perceived_risk": risk_perceptions,
            "adoption_sensitivity": adoption_sensitivities,
            "adopted": False,
            "adoption_year": np.nan,
        })


    # ============================================================
    # Operator pathway choice
    # ============================================================

    @staticmethod
    def pathway_environmental_score(pathway):
        """
        Returns the stylised environmental-performance score of an EoL route.
        """
        return PATHWAY_ENVIRONMENTAL_SCORE.get(pathway, 0.0)


    def effective_gate_fee(self, pathway):
        """Gate fee paid by the operator and received by the recycler."""
        return float(
            BASE_GATE_FEE[pathway]
            * (1.0 + EPR_GATE_FEE_MULTIPLIER * self.config.epr_strength)
        )

    def operator_private_pathway_cost(self, pathway, omega_t):
        """Private pathway cost observed by an EoL blade owner."""
        if pathway in RESOURCE_RECOVERY_PATHWAYS:
            return self.effective_gate_fee(pathway)
        return float(omega_t["processing_cost"][pathway])

    def operator_pathway_scores(
        self,
        omega_t,
        operator,
    ):
        """
        Computes normalized pathway attractiveness scores for one operator.
        """
        available_pathways = omega_t[
            "available_pathways"
        ]

        costs = {
            pathway: self.operator_private_pathway_cost(
                pathway, omega_t
            )
            for pathway in available_pathways
        }

        available_costs = np.array([
            costs[pathway]
            for pathway in available_pathways
        ])

        minimum_cost = float(
            available_costs.min()
        )

        maximum_cost = float(
            available_costs.max()
        )

        cost_range = max(
            maximum_cost - minimum_cost,
            NUMERICAL_EPSILON,
        )

        scores = {}

        for pathway in available_pathways:
            normalized_cost = (
                costs[pathway] - minimum_cost
            ) / cost_range

            cost_attractiveness = (
                1.0 - normalized_cost
            )

            environmental_score = (
                self.pathway_environmental_score(
                    pathway
                )
            )

            score = (
                operator["cost_weight"]
                * cost_attractiveness
                + operator["environment_weight"]
                * environmental_score
            )

            # A recycled-content mandate does not directly alter
            # operator preferences. Its effect is transmitted through
            # mandatory downstream demand and recycler economics.

            # Recovery remains an accessible incumbent route.
            if pathway == "recovery":
                score += 0.05

            scores[pathway] = score

        return scores


    def allocate_operator_flows(
        self,
        omega_t,
        year,
    ):
        """
        Aggregates heterogeneous operator decisions into desired pathway flows.

        Operators allocate both the new annual EoL flow and a limited share
        of the accumulated untreated stock.
        """
        new_decommissioned = (
            decommissioned_wtb_inflow(year)
        )

        untreated_stock = max(
            0.0,
            omega_t.get(
                "untreated_stock",
                0.0,
            ),
        )

        # Only part of the historical backlog is reconsidered in one period.
        backlog_reallocation_rate = bounded(
            BASE_BACKLOG_REALLOCATION_RATE,
            lower=0.05,
            upper=0.35,
        )

        backlog_available = (
            untreated_stock
            * backlog_reallocation_rate
        )

        total_allocatable_flow = (
            new_decommissioned
            + backlog_available
        )

        flow_per_operator = (
            total_allocatable_flow
            / self.n_operators
        )

        desired_flows = {
            pathway: 0.0
            for pathway in PATHWAYS
        }

        probability_sums = {
            pathway: 0.0
            for pathway in PATHWAYS
        }

        for _, operator in self.operators.iterrows():
            scores = self.operator_pathway_scores(
                omega_t=omega_t,
                operator=operator,
            )

            pathway_probabilities = (
                normalize_scores_to_shares(
                    scores=scores,
                    sensitivity=operator[
                        "choice_sensitivity"
                    ],
                )
            )

            years_since_start = max(0, int(year) - START_YEAR)
            inertia_weight = bounded(
                INITIAL_ALLOCATION_INERTIA
                * np.exp(
                    -INITIAL_ALLOCATION_INERTIA_DECAY
                    * years_since_start
                )
            )

            initial_available_total = sum(
                INITIAL_PATHWAY_SHARES[pathway]
                for pathway in pathway_probabilities
            )

            initial_available_shares = {
                pathway: safe_divide(
                    INITIAL_PATHWAY_SHARES[pathway],
                    initial_available_total,
                    default=0.0,
                )
                for pathway in pathway_probabilities
            }

            pathway_probabilities = {
                pathway: (
                    (1.0 - inertia_weight)
                    * pathway_probabilities[pathway]
                    + inertia_weight
                    * initial_available_shares[pathway]
                )
                for pathway in pathway_probabilities
            }

            probability_sum = sum(pathway_probabilities.values())
            pathway_probabilities = {
                pathway: safe_divide(
                    probability,
                    probability_sum,
                    default=0.0,
                )
                for pathway, probability
                in pathway_probabilities.items()
            }

            selected_pathway = self.rng.choice(
                list(pathway_probabilities.keys()),
                p=list(pathway_probabilities.values()),
            )

            desired_flows[selected_pathway] += (
                flow_per_operator
            )

            for pathway, probability in (
                pathway_probabilities.items()
            ):
                probability_sums[pathway] += (
                    probability
                )

        mean_pathway_probabilities = {
            pathway: (
                probability_sums[pathway]
                / self.n_operators
            )
            for pathway in PATHWAYS
        }

        return {
            "desired_flows": desired_flows,
            "mean_pathway_probabilities": (
                mean_pathway_probabilities
            ),
            "new_decommissioned": (
                new_decommissioned
            ),
            "backlog_considered": (
                backlog_available
            ),
            "total_allocatable_flow": (
                total_allocatable_flow
            ),
        }


    # ============================================================
    # Recycler economics
    # ============================================================

    def recycler_unit_revenue(
        self,
        pathway,
        expected_sales_ratio=1.0,
    ):
        """
        Calculate net recycler revenue per treated WTB unit.

        Gate fees accrue when WTB material is accepted. Material revenue is
        earned only for the expected sold share of recovered output; unsold
        output incurs an inventory-carrying penalty.
        """
        expected_sales_ratio = bounded(expected_sales_ratio)

        recovered_output_price = RECOVERED_MATERIAL_PRICE[pathway]
        gate_fee = self.effective_gate_fee(pathway)

        if pathway in MATERIAL_RECOVERY_PATHWAYS:
            recovered_output = resource_output_yield(pathway)
            sold_material_revenue = (
                recovered_output
                * recovered_output_price
                * expected_sales_ratio
            )
            inventory_carrying_cost = (
                recovered_output
                * recovered_output_price
                * (1.0 - expected_sales_ratio)
                * INVENTORY_HOLDING_COST_RATE
            )
        else:
            # Recovery creates an immediate resource credit rather than a
            # stored secondary-material inventory.
            sold_material_revenue = (
                RECOVERY_EFFICIENCY[pathway]
                * recovered_output_price
            )
            inventory_carrying_cost = 0.0

        return (
            gate_fee
            + sold_material_revenue
            - inventory_carrying_cost
        )

    def effective_solvolysis_investment_multiplier(self):
        """CAPEX multiplier for solvolysis investment and entry."""
        return max(
            0.45,
            1.0
            - TECH_SUPPORT_INVESTMENT_REDUCTION
            * self.config.technology_support,
        )

    def effective_solvolysis_switching_multiplier(self):
        """CAPEX multiplier for solvolysis switching and retrofit."""
        return max(
            0.45,
            1.0
            - 0.30 * self.config.technology_support,
        )

    def prospective_closed_loop_market(self, omega_t):
        """Demand signal visible before realized adoption or first entry."""
        readiness = bounded(
            omega_t.get("last_manufacturer_adoption_readiness", 0.0)
        )
        prospective_demand = max(
            0.0,
            float(omega_t.get("closed_loop_demand", 0.0)) * readiness,
        )
        effective_demand = max(
            float(omega_t.get("last_effective_closed_loop_demand", 0.0)),
            prospective_demand,
        )
        demand_gap = max(
            float(omega_t.get("last_closed_loop_demand_gap", 0.0)),
            prospective_demand
            - float(omega_t.get("last_realized_closed_loop_flow", 0.0)),
            0.0,
        )
        return effective_demand, demand_gap, readiness

    def realized_annual_profit(
        self,
        pathway,
        omega_t,
        realized_throughput,
        realized_sales_ratio,
        installed_capacity,
    ):
        """Observed one-period operating profit, excluding project CAPEX."""
        realized_throughput = max(0.0, float(realized_throughput))
        installed_capacity = max(0.0, float(installed_capacity))
        unit_revenue = self.recycler_unit_revenue(
            pathway=pathway,
            expected_sales_ratio=realized_sales_ratio,
        )
        variable_cost = (
            float(omega_t["processing_cost"][pathway])
            + float(BASE_LOGISTICS_COST[pathway])
        )
        return float(
            (unit_revenue - variable_cost) * realized_throughput
            - ANNUAL_FIXED_CAPACITY_COST[pathway] * installed_capacity
        )

    def expected_discounted_profit(
        self,
        pathway,
        omega_t,
        expected_throughput,
        discount_rate,
        expected_sales_ratio=1.0,
        capacity_limit=None,
        include_investment=False,
        investment_capacity=0.0,
        include_switching=False,
        switching_capacity=0.0,
        retrofit_compatibility=1.0,
    ):
        """Compute expected discounted operating profit or project NPV."""
        if (
            pathway == "solvolysis"
            and not omega_t["solvolysis_available"]
        ):
            return -float(RECYCLER_INVESTMENT_COST[pathway])

        expected_throughput = max(0.0, float(expected_throughput))
        expected_sales_ratio = bounded(expected_sales_ratio)

        if capacity_limit is not None:
            capacity_limit = max(0.0, float(capacity_limit))

        revenue_per_unit = self.recycler_unit_revenue(
            pathway=pathway,
            expected_sales_ratio=expected_sales_ratio,
        )

        processing_cost = float(
            omega_t["processing_cost"][pathway]
        )
        logistics_cost = float(BASE_LOGISTICS_COST[pathway])
        unit_margin = revenue_per_unit - processing_cost - logistics_cost

        if pathway == "solvolysis":
            expected_growth = (
                self.config.closed_loop_demand_growth
            )
        else:
            expected_growth = (
                OPEN_LOOP_DEMAND_GROWTH
            )

        discounted_profit = 0.0

        for horizon_year in range(RECYCLER_PROFIT_HORIZON):
            expected_volume = (
                expected_throughput
                * (1.0 + expected_growth) ** horizon_year
            )

            if capacity_limit is not None:
                expected_volume = min(
                    expected_volume,
                    capacity_limit,
                )

            annual_fixed_cost = (
                ANNUAL_FIXED_CAPACITY_COST[pathway]
                * (
                    capacity_limit
                    if capacity_limit is not None
                    else expected_throughput
                )
            )

            annual_profit = (
                unit_margin * expected_volume
                - annual_fixed_cost
            )

            discounted_profit += (
                annual_profit
                / (1.0 + discount_rate) ** horizon_year
            )

        if include_investment:
            investment_capacity = max(
                0.0,
                float(investment_capacity),
            )
            investment_cost = (
                RECYCLER_INVESTMENT_COST[pathway]
                * EXPANSION_CAPEX_MULTIPLIER
                * investment_capacity
            )

            if pathway == "solvolysis":
                # Investment cost is reduced only by explicit
                # technology support and EPR-related transition funding.
                # Demand-pull and recycled-content mandates affect market
                # revenues and demand rather than physical CAPEX.
                investment_cost *= self.effective_solvolysis_investment_multiplier()

            discounted_profit -= investment_cost

        if include_switching:
            switching_capacity = max(
                0.0,
                float(switching_capacity),
            )
            compatibility = max(
                0.25,
                bounded(retrofit_compatibility),
            )

            switching_cost = (
                SOLVOLYSIS_RETROFIT_FIXED_COST
                + RECYCLER_SWITCHING_COST["solvolysis"]
                + SOLVOLYSIS_RETROFIT_CAPEX_PER_CAPACITY
                * switching_capacity
                / compatibility
            )

            # Switching costs are reduced by explicit technology
            # support and coordination. Demand-side instruments affect
            # expected revenues rather than the one-time retrofit cost.
            switching_cost *= self.effective_solvolysis_switching_multiplier()

            discounted_profit -= switching_cost

        return float(discounted_profit)

    def update_recycler_agents(
        self,
        omega_t,
        actual_flows,
        desired_flows,
        unmet_flows_by_pathway,
        year,
    ):
        """
        Update recycler capacity using observed utilization, downstream sales,
        project NPV, persistent signals, and retrofit compatibility.
        """
        capacity_changes = {
            pathway: 0.0
            for pathway in PATHWAYS
        }
        pathway_profit_sums = {
            pathway: 0.0
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        }
        pathway_profit_counts = {
            pathway: 0
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        }

        switches_to_solvolysis = 0
        recycler_exits = 0
        recycler_expansions = 0
        recycler_contractions = 0
        total_expansion_capex = 0.0
        total_switching_capex = 0.0
        total_entry_capex = 0.0
        new_solvolysis_entries = 0

        aggregate_capacity_by_pathway = self.aggregate_active_capacity()
        active_indices = self.recyclers.index[
            self.recyclers["active"]
        ].to_numpy()
        active_indices = self.rng.permutation(active_indices)

        eligible_mask = (
            self.recyclers["active"]
            & (self.recyclers["technology"] != "solvolysis")
            & (
                self.recyclers["retrofit_compatibility"]
                >= MIN_SOLVOLYSIS_RETROFIT_COMPATIBILITY
            )
        )

        potential_converted_capacity = 0.0
        for _, candidate in self.recyclers.loc[eligible_mask].iterrows():
            compatibility = bounded(candidate["retrofit_compatibility"])
            conversion_efficiency = bounded(
                SOLVOLYSIS_CONVERSION_BASE
                + SOLVOLYSIS_CONVERSION_COMPATIBILITY_EFFECT
                * compatibility,
                lower=0.30,
                upper=0.70,
            )
            potential_converted_capacity += (
                float(candidate["capacity"])
                * conversion_efficiency
            )

        annual_switch_share = bounded(
            0.08,
            lower=0.05,
            upper=MAX_ANNUAL_SWITCH_SHARE,
        )
        eligible_incumbent_capacity = float(
            self.recyclers.loc[eligible_mask, "capacity"].sum()
        )
        maximum_switched_incumbent_capacity = (
            annual_switch_share * eligible_incumbent_capacity
        )
        switched_incumbent_capacity = 0.0

        last_sales_ratios = omega_t.get(
            "last_material_sales_ratio",
            INITIAL_MATERIAL_SALES_RATIO,
        )

        for index in active_indices:
            recycler = self.recyclers.loc[index]

            if not bool(recycler["active"]):
                continue

            current_technology = str(recycler["technology"])
            current_capacity = max(
                float(recycler["capacity"]),
                NUMERICAL_EPSILON,
            )

            pathway_total_capacity = max(
                aggregate_capacity_by_pathway.get(
                    current_technology,
                    0.0,
                ),
                NUMERICAL_EPSILON,
            )

            current_observed_flow = (
                float(actual_flows.get(current_technology, 0.0))
                * current_capacity
                / pathway_total_capacity
            )

            utilization_rate = bounded(
                safe_divide(
                    current_observed_flow,
                    current_capacity,
                    default=0.0,
                )
            )

            expected_sales_ratio = (
                1.0
                if current_technology == "recovery"
                else bounded(
                    last_sales_ratios.get(current_technology, 0.0)
                )
            )

            current_profit = self.realized_annual_profit(
                pathway=current_technology,
                omega_t=omega_t,
                realized_throughput=current_observed_flow,
                realized_sales_ratio=expected_sales_ratio,
                installed_capacity=current_capacity,
            )
            incumbent_continuation_npv = self.expected_discounted_profit(
                pathway=current_technology,
                omega_t=omega_t,
                expected_throughput=current_observed_flow,
                discount_rate=float(recycler["discount_rate"]),
                expected_sales_ratio=expected_sales_ratio,
                capacity_limit=current_capacity,
            )

            pathway_profit_sums[current_technology] += current_profit
            pathway_profit_counts[current_technology] += 1

            utilization_ema = (
                (1.0 - RECYCLER_EMA_SMOOTHING) * float(recycler["utilization_ema"])
                + RECYCLER_EMA_SMOOTHING * utilization_rate
            )
            profit_ema = (
                (1.0 - RECYCLER_EMA_SMOOTHING) * float(recycler["profit_ema"])
                + RECYCLER_EMA_SMOOTHING * current_profit
            )
            sales_ratio_ema = (
                (1.0 - RECYCLER_EMA_SMOOTHING) * float(recycler["sales_ratio_ema"])
                + RECYCLER_EMA_SMOOTHING * expected_sales_ratio
            )

            self.recyclers.at[index, "utilization_ema"] = utilization_ema
            self.recyclers.at[index, "profit_ema"] = profit_ema
            self.recyclers.at[index, "sales_ratio_ema"] = sales_ratio_ema

            # ----------------------------------------------------
            # Contraction and exit under persistent underperformance
            # ----------------------------------------------------
            underperforming = (
                current_profit < float(recycler["exit_threshold"])
                or (
                    utilization_ema < 0.30
                    and sales_ratio_ema < 0.55
                )
            )

            underperformance_years = (
                int(recycler["underperformance_years"]) + 1
                if underperforming
                else 0
            )
            self.recyclers.at[
                index, "underperformance_years"
            ] = underperformance_years

            if (
                underperformance_years
                >= RECYCLER_NEGATIVE_PROFIT_YEARS_FOR_CONTRACTION
            ):
                contraction = (
                    current_capacity
                    * RECYCLER_CAPACITY_CONTRACTION_RATE
                )
                remaining_capacity = max(
                    0.0,
                    current_capacity - contraction,
                )

                self.recyclers.at[index, "capacity"] = remaining_capacity
                capacity_changes[current_technology] -= contraction
                recycler_contractions += 1
                current_capacity = remaining_capacity

                if (
                    underperformance_years >= 6
                    or current_capacity <= 0.25
                ):
                    # Remove all remaining capacity when the agent exits.
                    capacity_changes[current_technology] -= current_capacity
                    self.recyclers.at[index, "capacity"] = 0.0
                    self.recyclers.at[index, "active"] = False
                    recycler_exits += 1
                    continue

            if current_capacity <= NUMERICAL_EPSILON:
                continue

            switched = False

            # ----------------------------------------------------
            # Solvolysis switching decision
            # ----------------------------------------------------
            compatibility = bounded(
                self.recyclers.at[index, "retrofit_compatibility"]
            )
            cooldown_satisfied = (
                year
                - int(self.recyclers.at[index, "last_switch_year"])
                >= SWITCH_COOLDOWN_YEARS
            )

            can_consider_solvolysis = (
                bool(omega_t["solvolysis_available"])
                and current_technology != "solvolysis"
                and compatibility
                >= MIN_SOLVOLYSIS_RETROFIT_COMPATIBILITY
                and cooldown_satisfied
                and (
                    switched_incumbent_capacity + current_capacity
                    <= maximum_switched_incumbent_capacity
                    + NUMERICAL_EPSILON
                )
            )

            if can_consider_solvolysis:
                effective_demand, demand_gap, _ = (
                    self.prospective_closed_loop_market(omega_t)
                )
                open_loop_gap = max(
                    0.0,
                    float(
                        omega_t.get(
                            "last_open_loop_demand_gap",
                            0.0,
                        )
                    ),
                )

                potential_material_market = (
                    effective_demand
                    + 0.20 * open_loop_gap
                )

                demand_pressure = bounded(
                    safe_divide(
                        demand_gap,
                        max(effective_demand, NUMERICAL_EPSILON),
                        default=0.0,
                    )
                )

                conversion_efficiency = bounded(
                    SOLVOLYSIS_CONVERSION_BASE
                    + SOLVOLYSIS_CONVERSION_COMPATIBILITY_EFFECT
                    * compatibility,
                    lower=0.30,
                    upper=0.70,
                )

                converted_capacity = (
                    current_capacity
                    * conversion_efficiency
                )

                existing_solvolysis_capacity = max(
                    0.0,
                    aggregate_capacity_by_pathway.get(
                        "solvolysis", 0.0
                    ),
                )

                prospective_supply_capacity = max(
                    converted_capacity,
                    existing_solvolysis_capacity
                    + 0.35 * potential_converted_capacity,
                )

                prospective_market_share = bounded(
                    safe_divide(
                        converted_capacity,
                        prospective_supply_capacity,
                        default=0.0,
                    ),
                    lower=0.0,
                    upper=0.35,
                )

                expected_material_sales = (
                    potential_material_market
                    * prospective_market_share
                )

                # Throughput limited by both capacity and expected feedstock.
                # Each recycler captures a market share of available feedstock.
                total_solvolysis_cap = max(
                    float(
                        omega_t["capacity"].get("solvolysis", 0.0)
                    ),
                    NUMERICAL_EPSILON,
                )
                # Only the backlog fraction released this year + fresh inflow.
                # Use same backlog rate as operators to keep expectations consistent.
                scenario_backlog_rate = min(
                    0.35,
                    max(
                        0.05,
                        BASE_BACKLOG_REALLOCATION_RATE,
                    ),
                )
                accessible_backlog = (
                    float(omega_t.get("untreated_stock", 0.0))
                    * scenario_backlog_rate
                )
                feedstock_supply = (
                    accessible_backlog
                    + float(omega_t.get("annual_decommissioned", 0.0))
                )
                # Include competing candidates in denominator to avoid
                # multiple recyclers anticipating the same feedstock.
                # potential_converted_capacity already includes this candidate;
                # do not add converted_capacity again.
                feedstock_market_share = safe_divide(
                    converted_capacity,
                    total_solvolysis_cap
                    + potential_converted_capacity,
                    default=1.0,
                )
                projected_feedstock = feedstock_supply * feedstock_market_share
                solvolysis_expected_throughput = min(
                    converted_capacity * 0.85,
                    projected_feedstock,
                )

                expected_solvolysis_output = (
                    solvolysis_expected_throughput
                    * resource_output_yield("solvolysis")
                )
                expected_solvolysis_sales = min(
                    expected_material_sales,
                    expected_solvolysis_output,
                )
                expected_solvolysis_sales_ratio = bounded(
                    safe_divide(
                        expected_solvolysis_sales,
                        expected_solvolysis_output,
                        default=0.0,
                    )
                )

                solvolysis_profit = self.expected_discounted_profit(
                    pathway="solvolysis",
                    omega_t=omega_t,
                    expected_throughput=solvolysis_expected_throughput,
                    discount_rate=float(recycler["discount_rate"]),
                    expected_sales_ratio=(
                        expected_solvolysis_sales_ratio
                    ),
                    capacity_limit=converted_capacity,
                    include_switching=True,
                    switching_capacity=converted_capacity,
                    retrofit_compatibility=compatibility,
                )

                # Multiplicative shift preserves heterogeneity.
                # Additive + PROFIT_SCALE collapsed all thresholds to 0.
                effective_switching_threshold = max(
                    5.0,
                    float(recycler["switching_threshold"]),
                )

                net_switching_advantage = (
                    solvolysis_profit
                    - incumbent_continuation_npv
                    - effective_switching_threshold
                )

                switch_advantage_ema = (
                    0.50
                    * float(recycler["switch_advantage_ema"])
                    + 0.50 * net_switching_advantage
                )
                self.recyclers.at[
                    index, "switch_advantage_ema"
                ] = switch_advantage_ema

                # demand_gate requires observed market evidence.
                # Policy instruments affect revenue/costs; they do not
                # substitute for actual demand signal (no policy bypass).
                demand_gate = (
                    demand_pressure
                    >= MIN_SOLVOLYSIS_DEMAND_PRESSURE
                )

                switch_signal = (
                    net_switching_advantage > 0.0
                    and demand_gate
                    and safe_divide(
                        solvolysis_expected_throughput,
                        converted_capacity,
                        default=0.0,
                    ) >= 0.25
                    and expected_solvolysis_sales_ratio >= 0.40
                )

                switch_signal_years = (
                    int(recycler["positive_switch_signal_years"]) + 1
                    if switch_signal
                    else 0
                )
                self.recyclers.at[
                    index, "positive_switch_signal_years"
                ] = switch_signal_years

                if switch_signal_years >= SWITCH_SIGNAL_YEARS:
                    economic_probability = max(
                        0.0,
                        2.0
                        * (
                            logistic(
                                switch_advantage_ema
                                / RECYCLER_PROFIT_SCALE
                            )
                            - 0.5
                        ),
                    )
                    switch_probability = bounded(
                        economic_probability
                        * compatibility,
                        lower=0.0,
                        upper=0.70,
                    )
                else:
                    switch_probability = 0.0

                if self.rng.random() < switch_probability:
                    old_capacity = current_capacity

                    capacity_changes[current_technology] -= old_capacity
                    capacity_changes["solvolysis"] += converted_capacity

                    self.recyclers.at[index, "technology"] = "solvolysis"
                    self.recyclers.at[index, "capacity"] = converted_capacity
                    self.recyclers.at[index, "last_switch_year"] = year
                    self.recyclers.at[
                        index, "positive_switch_signal_years"
                    ] = 0
                    self.recyclers.at[
                        index, "switch_advantage_ema"
                    ] = 0.0
                    self.recyclers.at[
                        index, "positive_expansion_signal_years"
                    ] = 0
                    self.recyclers.at[index, "underperformance_years"] = 0
                    self.recyclers.at[index, "retrofit_compatibility"] = 1.0

                    estimated_switch_cost = (
                        SOLVOLYSIS_RETROFIT_FIXED_COST
                        + RECYCLER_SWITCHING_COST["solvolysis"]
                        + SOLVOLYSIS_RETROFIT_CAPEX_PER_CAPACITY
                        * converted_capacity
                        / max(0.25, compatibility)
                    )
                    estimated_switch_cost *= self.effective_solvolysis_switching_multiplier()
                    total_switching_capex += estimated_switch_cost

                    switches_to_solvolysis += 1
                    switched_incumbent_capacity += old_capacity
                    switched = True

            if switched:
                continue

            # ----------------------------------------------------
            # Capacity expansion decision
            # ----------------------------------------------------
            current_technology = str(
                self.recyclers.at[index, "technology"]
            )
            current_capacity = max(
                0.0,
                float(self.recyclers.at[index, "capacity"]),
            )

            desired_pathway_flow = max(
                0.0,
                float(desired_flows.get(current_technology, 0.0)),
            )
            unmet_pathway_flow = max(
                0.0,
                float(
                    unmet_flows_by_pathway.get(
                        current_technology, 0.0
                    )
                ),
            )
            pathway_demand_pressure = bounded(
                safe_divide(
                    unmet_pathway_flow,
                    max(desired_pathway_flow, NUMERICAL_EPSILON),
                    default=0.0,
                )
            )

            expansion_signal = (
                utilization_ema >= EXPANSION_UTILIZATION_THRESHOLD
                and sales_ratio_ema >= EXPANSION_MIN_SALES_RATIO
                and profit_ema > 0.0
                and (
                    pathway_demand_pressure > 0.05
                    or utilization_ema >= 0.90
                )
                and (
                    year
                    - int(self.recyclers.at[index, "last_expansion_year"])
                    >= EXPANSION_COOLDOWN_YEARS
                )
            )

            expansion_signal_years = (
                int(recycler["positive_expansion_signal_years"]) + 1
                if expansion_signal
                else 0
            )
            self.recyclers.at[
                index, "positive_expansion_signal_years"
            ] = expansion_signal_years

            if (
                expansion_signal_years >= EXPANSION_SIGNAL_YEARS
                and current_capacity > NUMERICAL_EPSILON
            ):
                utilization_strength = bounded(
                    safe_divide(
                        utilization_ema
                        - EXPANSION_UTILIZATION_THRESHOLD,
                        1.0 - EXPANSION_UTILIZATION_THRESHOLD,
                        default=0.0,
                    )
                )
                sales_strength = bounded(
                    safe_divide(
                        sales_ratio_ema - EXPANSION_MIN_SALES_RATIO,
                        1.0 - EXPANSION_MIN_SALES_RATIO,
                        default=0.0,
                    )
                )
                evidence_strength = bounded(
                    0.50 * utilization_strength
                    + 0.30 * sales_strength
                    + 0.20 * pathway_demand_pressure
                )

                expansion_rate = (
                    MAX_ANNUAL_CAPACITY_EXPANSION[current_technology]
                    * evidence_strength
                )
                expansion_amount = current_capacity * expansion_rate

                expected_incremental_throughput = (
                    expansion_amount
                    * min(0.90, max(0.75, utilization_ema))
                )

                project_npv = self.expected_discounted_profit(
                    pathway=current_technology,
                    omega_t=omega_t,
                    expected_throughput=expected_incremental_throughput,
                    discount_rate=float(recycler["discount_rate"]),
                    expected_sales_ratio=sales_ratio_ema,
                    capacity_limit=expansion_amount,
                    include_investment=True,
                    investment_capacity=expansion_amount,
                )

                investment_probability = max(
                    0.0,
                    2.0
                    * (
                        logistic(project_npv / RECYCLER_PROFIT_SCALE)
                        - 0.5
                    ),
                )
                investment_probability = bounded(
                    investment_probability,
                    lower=0.0,
                    upper=EXPANSION_MAX_DECISION_PROBABILITY,
                )

                if (
                    project_npv > 0.0
                    and expansion_amount > NUMERICAL_EPSILON
                    and self.rng.random() < investment_probability
                ):
                    self.recyclers.at[index, "capacity"] = (
                        current_capacity + expansion_amount
                    )
                    self.recyclers.at[index, "last_expansion_year"] = year
                    self.recyclers.at[
                        index, "positive_expansion_signal_years"
                    ] = 0
                    capacity_changes[current_technology] += expansion_amount
                    total_expansion_capex += (
                        RECYCLER_INVESTMENT_COST[current_technology]
                        * EXPANSION_CAPEX_MULTIPLIER
                        * expansion_amount
                    )
                    recycler_expansions += 1

        # ----------------------------------------------------
        # Greenfield solvolysis entry
        # ----------------------------------------------------
        if bool(omega_t["solvolysis_available"]):
            effective_demand, demand_gap, entry_readiness = (
                self.prospective_closed_loop_market(omega_t)
            )
            open_loop_gap = max(
                0.0,
                float(
                    omega_t.get(
                        "last_open_loop_demand_gap",
                        0.0,
                    )
                ),
            )

            entry_demand_pressure = bounded(
                safe_divide(
                    demand_gap,
                    max(effective_demand, NUMERICAL_EPSILON),
                    default=0.0,
                )
            )

            entry_capacity = NEW_SOLVOLYSIS_ENTRY_CAPACITY
            existing_solvolysis_capacity = max(
                0.0,
                self.aggregate_active_capacity().get(
                    "solvolysis",
                    0.0,
                ),
            )
            potential_material_market = (
                effective_demand
                + 0.20 * open_loop_gap
            )
            entry_market_share = bounded(
                safe_divide(
                    entry_capacity,
                    existing_solvolysis_capacity
                    + entry_capacity,
                    default=0.0,
                ),
                lower=0.0,
                upper=0.30,
            )
            expected_entry_sales = (
                potential_material_market
                * entry_market_share
            )
            # Throughput limited by capacity AND accessible feedstock.
            scenario_backlog_rate_entry = min(
                0.35,
                max(
                    0.05,
                    BASE_BACKLOG_REALLOCATION_RATE,
                ),
            )
            accessible_backlog_entry = (
                float(omega_t.get("untreated_stock", 0.0))
                * scenario_backlog_rate_entry
            )
            feedstock_supply_entry = (
                accessible_backlog_entry
                + float(omega_t.get("annual_decommissioned", 0.0))
            )
            entry_feedstock_share = safe_divide(
                entry_capacity,
                float(omega_t["capacity"].get("solvolysis", 0.0))
                + potential_converted_capacity
                + entry_capacity,
                default=1.0,
            )
            projected_entry_feedstock = (
                feedstock_supply_entry * entry_feedstock_share
            )
            expected_entry_throughput = min(
                entry_capacity * 0.85,
                projected_entry_feedstock,
            )
            expected_entry_output = (
                expected_entry_throughput
                * resource_output_yield("solvolysis")
            )
            expected_entry_sales_capped = min(
                expected_entry_sales,
                expected_entry_output,
            )
            expected_entry_sales_ratio = bounded(
                safe_divide(
                    expected_entry_sales_capped,
                    expected_entry_output,
                    default=0.0,
                )
            )

            representative_discount_rate = float(
                self.recyclers.loc[
                    self.recyclers["active"],
                    "discount_rate",
                ].mean()
            )

            entry_npv = self.expected_discounted_profit(
                pathway="solvolysis",
                omega_t=omega_t,
                expected_throughput=expected_entry_throughput,
                discount_rate=representative_discount_rate,
                expected_sales_ratio=expected_entry_sales_ratio,
                capacity_limit=entry_capacity,
                include_investment=True,
                investment_capacity=entry_capacity,
            )
            entry_npv -= NEW_SOLVOLYSIS_ENTRY_FIXED_COST

            entry_signal = (
                entry_npv > 0.0
                and entry_demand_pressure
                >= MIN_SOLVOLYSIS_DEMAND_PRESSURE
                and expected_entry_sales_ratio >= 0.55
                and safe_divide(
                    expected_entry_throughput,
                    entry_capacity,
                    default=0.0,
                ) >= 0.35
            )

            self.solvolysis_entry_signal_years = (
                self.solvolysis_entry_signal_years + 1
                if entry_signal
                else 0
            )

            if (
                self.solvolysis_entry_signal_years
                >= SOLVOLYSIS_ENTRY_SIGNAL_YEARS
            ):
                entry_probability = max(
                    0.0,
                    2.0
                    * (
                        logistic(entry_npv / RECYCLER_PROFIT_SCALE)
                        - 0.5
                    ),
                )
                entry_probability = bounded(
                    entry_probability,
                    lower=0.0,
                    upper=SOLVOLYSIS_ENTRY_MAX_PROBABILITY,
                )

                if self.rng.random() < entry_probability:
                    new_recycler_id = (
                        int(self.recyclers["recycler_id"].max()) + 1
                        if not self.recyclers.empty
                        else 0
                    )
                    new_recycler = {
                        "recycler_id": new_recycler_id,
                        "technology": "solvolysis",
                        "capacity": entry_capacity,
                        "switching_threshold": float(
                            self.recyclers.loc[
                                self.recyclers["active"],
                                "switching_threshold",
                            ].mean()
                            if self.recyclers["active"].any()
                            else ABM_RECYCLER_SWITCHING_THRESHOLD_MIN
                        ),
                        "discount_rate": representative_discount_rate,
                        "exit_threshold": float(
                            np.clip(
                                self.rng.normal(
                                    loc=MEAN_RECYCLER_EXIT_THRESHOLD,
                                    scale=STD_RECYCLER_EXIT_THRESHOLD,
                                ),
                                0.01,
                                0.50,
                            )
                        ),
                        "retrofit_compatibility": 1.0,
                        "active": True,
                        "underperformance_years": 0,
                        "positive_expansion_signal_years": 0,
                        "positive_switch_signal_years": 0,
                        "switch_advantage_ema": 0.0,
                        "utilization_ema": 0.0,
                        "profit_ema": 0.0,
                        "sales_ratio_ema": 0.0,
                        "last_switch_year": year,
                        "last_expansion_year": year,
                    }
                    self.recyclers = pd.concat(
                        [
                            self.recyclers,
                            pd.DataFrame([new_recycler]),
                        ],
                        ignore_index=True,
                    )

                    capacity_changes["solvolysis"] += entry_capacity
                    new_solvolysis_entries = 1
                    total_entry_capex = (
                        RECYCLER_INVESTMENT_COST["solvolysis"]
                        * EXPANSION_CAPEX_MULTIPLIER
                        * entry_capacity
                        * self.effective_solvolysis_investment_multiplier()
                        + NEW_SOLVOLYSIS_ENTRY_FIXED_COST
                    )
                    self.solvolysis_entry_signal_years = 0

        mean_profits = {
            pathway: safe_divide(
                pathway_profit_sums[pathway],
                pathway_profit_counts[pathway],
                default=0.0,
            )
            for pathway in RESOURCE_RECOVERY_PATHWAYS
        }

        closing_capacity = self.aggregate_active_capacity()

        return {
            "capacity_changes": capacity_changes,
            "closing_recycler_capacity_by_pathway": closing_capacity,
            "mean_recycler_profit": mean_profits,
            "switches_to_solvolysis": switches_to_solvolysis,
            "switched_incumbent_capacity": float(
                switched_incumbent_capacity
            ),
            "maximum_switched_incumbent_capacity": float(
                maximum_switched_incumbent_capacity
            ),
            "recycler_exits": recycler_exits,
            "recycler_expansions": recycler_expansions,
            "recycler_contractions": recycler_contractions,
            "total_expansion_capex": float(total_expansion_capex),
            "total_switching_capex": float(total_switching_capex),
            "total_entry_capex": float(total_entry_capex),
            "total_annual_capex": float(
                total_expansion_capex
                + total_switching_capex
                + total_entry_capex
            ),
            "new_solvolysis_entries": int(new_solvolysis_entries),
            "active_recyclers": int(self.recyclers["active"].sum()),
            "solvolysis_recyclers": int(
                (
                    self.recyclers["active"]
                    & (
                        self.recyclers["technology"]
                        == "solvolysis"
                    )
                ).sum()
            ),
        }

    def manufacturer_adoption_signal(
        self,
        omega_t,
        year,
    ):
        """
        Update persistent voluntary manufacturer adoption after
        commercial availability.

        The recycled-content mandate is deliberately excluded from the
        voluntary adoption probability. It creates a separate mandatory
        demand component in the SD material-allocation mechanism.

        The mean probability represents current readiness. The realized
        share is a stock of firms that have adopted, not a fresh annual
        Bernoulli sample of the whole population.
        """
        effective_solvolysis_price = (
            RECOVERED_MATERIAL_PRICE["solvolysis"]
        )

        price_advantage = bounded(
            (
                VIRGIN_MATERIAL_PRICE
                - effective_solvolysis_price
            )
            / VIRGIN_MATERIAL_PRICE,
            lower=-1.0,
            upper=1.0,
        )

        quality_score = MATERIAL_QUALITY["solvolysis"]
        certification_score = logistic(
            CERTIFICATION_SENSITIVITY
            * (
                omega_t["trl_solvolysis"]
                - TRL_THRESHOLD_SOLVOLYSIS
            )
        )

        solvolysis_capacity = float(
            omega_t["capacity"].get("solvolysis", 0.0)
        )
        potential_closed_loop_demand = max(
            float(omega_t["closed_loop_demand"]),
            NUMERICAL_EPSILON,
        )
        solvolysis_material_output_capacity = (
            solvolysis_capacity
            * resource_output_yield("solvolysis")
        )
        capacity_reliability = bounded(
            solvolysis_material_output_capacity
            / potential_closed_loop_demand
        )
        observed_delivery_reliability = bounded(
            omega_t.get(
                "last_closed_loop_supply_reliability",
                INITIAL_OBSERVED_SUPPLY_RELIABILITY,
            )
        )
        supply_reliability = bounded(
            0.75 * observed_delivery_reliability
            + 0.25 * capacity_reliability
        )

        voluntary_policy_effect = (
            0.25 * self.config.demand_pull
        )

        individual_probabilities = []
        individual_risks = []

        for _, manufacturer in self.manufacturers.iterrows():
            perceived_risk = bounded(
                manufacturer["perceived_risk"]
                * (
                    1.0
                    - 0.45 * self.config.coordination_strength
                )
                * (1.0 - 0.25 * certification_score)
            )
            adoption_threshold = bounded(
                manufacturer["adoption_threshold"],
                lower=0.0,
                upper=1.0,
            )
            adoption_score = (
                0.20 * price_advantage
                + 0.30 * quality_score
                + 0.25 * supply_reliability
                + 0.20 * certification_score
                + voluntary_policy_effect
                - perceived_risk
                - adoption_threshold
            )
            adoption_probability = logistic(
                manufacturer["adoption_sensitivity"]
                * adoption_score
            )
            individual_probabilities.append(adoption_probability)
            individual_risks.append(perceived_risk)

        probabilities = np.asarray(
            individual_probabilities,
            dtype=float,
        )
        mean_adoption_probability = float(probabilities.mean())

        new_adoptions = 0
        abandonments = 0

        if bool(omega_t["solvolysis_available"]):
            adopted_mask = self.manufacturers["adopted"].to_numpy(
                dtype=bool
            )

            review_draws = self.rng.random(self.n_manufacturers)
            adoption_draws = self.rng.random(self.n_manufacturers)
            abandonment_draws = self.rng.random(self.n_manufacturers)

            # Realized adoption requires actual solvolysis supply capacity.
            # Readiness (probabilities) can build even without supply,
            # but commitment needs a credible provider (capacity > 0).
            solvolysis_capacity_available = (
                float(
                    omega_t["capacity"].get("solvolysis", 0.0)
                ) > NUMERICAL_EPSILON
            )
            new_mask = (
                ~adopted_mask
                & solvolysis_capacity_available
                & (review_draws < MANUFACTURER_ANNUAL_REVIEW_RATE)
                & (adoption_draws < probabilities)
            )

            abandonment_probability = bounded(
                MANUFACTURER_BASE_ABANDONMENT_RATE
                * (1.0 - supply_reliability)
                * (1.0 + float(np.mean(individual_risks)))
            )
            abandon_mask = (
                adopted_mask
                & (abandonment_draws < abandonment_probability)
            )

            if new_mask.any():
                self.manufacturers.loc[new_mask, "adopted"] = True
                self.manufacturers.loc[new_mask, "adoption_year"] = year
                new_adoptions = int(new_mask.sum())

            if abandon_mask.any():
                self.manufacturers.loc[abandon_mask, "adopted"] = False
                self.manufacturers.loc[abandon_mask, "adoption_year"] = np.nan
                abandonments = int(abandon_mask.sum())

        realized_adoption_share = float(
            self.manufacturers["adopted"].mean()
        )

        return {
            "manufacturer_adoption_readiness": (
                mean_adoption_probability
            ),
            "manufacturer_realized_adoption_share": (
                realized_adoption_share
            ),
            "new_manufacturer_adoptions": new_adoptions,
            "manufacturer_abandonments": abandonments,
            "perceived_risk": float(np.mean(individual_risks)),
            "supply_reliability": supply_reliability,
            "certification_score": certification_score,
            "price_advantage": price_advantage,
        }

    def step(
        self,
        omega_t,
        year,
    ):
        """Execute one annual ABM decision cycle."""
        operator_outputs = self.allocate_operator_flows(
            omega_t=omega_t,
            year=year,
        )

        allocation_result = apply_capacity_constraints(
            desired_flows=operator_outputs["desired_flows"],
            capacities=omega_t["capacity"],
            return_diagnostics=True,
        )

        actual_flows = allocation_result["actual_flows"]
        unmet_flow = allocation_result["total_unmet_flow"]
        unmet_flows_by_pathway = allocation_result[
            "unmet_flows_by_pathway"
        ]
        allocation_diagnostics = allocation_result["diagnostics"]

        recycler_outputs = self.update_recycler_agents(
            omega_t=omega_t,
            actual_flows=actual_flows,
            desired_flows=operator_outputs["desired_flows"],
            unmet_flows_by_pathway=unmet_flows_by_pathway,
            year=year,
        )

        manufacturer_outputs = self.manufacturer_adoption_signal(
            omega_t=omega_t,
            year=year,
        )

        return {
            "desired_flows": operator_outputs["desired_flows"],
            "actual_flows": actual_flows,
            "unmet_flow": unmet_flow,
            "unmet_flows_by_pathway": unmet_flows_by_pathway,
            "pathway_shares_desired": operator_outputs[
                "mean_pathway_probabilities"
            ],
            "new_decommissioned": operator_outputs["new_decommissioned"],
            "backlog_considered": operator_outputs["backlog_considered"],
            "total_allocatable_flow": operator_outputs[
                "total_allocatable_flow"
            ],
            "allocation_diagnostics": allocation_diagnostics,
            "initial_rejected_flow": allocation_diagnostics[
                "total_initial_rejected_flow"
            ],
            "reallocated_flow": allocation_diagnostics[
                "total_reallocated_flow"
            ],
            "unused_capacity_after_rerouting": allocation_diagnostics[
                "total_unused_capacity"
            ],
            "total_capacity_shortage": allocation_diagnostics[
                "aggregate_capacity_shortage"
            ],
            "capacity_allocation_mismatch": allocation_diagnostics[
                "capacity_allocation_mismatch"
            ],
            **recycler_outputs,
            **manufacturer_outputs,
        }

# ============================================================
# Active ABM population initialization checks
# ============================================================

_test_abm_scenario_name = next(iter(SCENARIOS))


def _build_test_abm_layer():
    # Build the active populations under the common test seed.
    return ABMLayer(
        scenario_config=SCENARIOS[_test_abm_scenario_name],
        rng=np.random.default_rng(AGENT_POPULATION_SEED),
        n_operators=N_OPERATORS,
        n_recyclers=N_RECYCLERS,
        n_manufacturers=N_MANUFACTURERS,
    )


_test_abm_layer = _build_test_abm_layer()

# The operational populations have one canonical representation.
assert isinstance(_test_abm_layer.operators, pd.DataFrame)
assert isinstance(_test_abm_layer.recyclers, pd.DataFrame)
assert isinstance(_test_abm_layer.manufacturers, pd.DataFrame)

assert len(_test_abm_layer.operators) == N_OPERATORS
assert len(_test_abm_layer.recyclers) == N_RECYCLERS
assert len(_test_abm_layer.manufacturers) == N_MANUFACTURERS

assert _test_abm_layer.operators["operator_id"].is_unique
assert _test_abm_layer.recyclers["recycler_id"].is_unique
assert _test_abm_layer.manufacturers["manufacturer_id"].is_unique

assert np.allclose(
    (
        _test_abm_layer.operators["cost_weight"]
        + _test_abm_layer.operators["environment_weight"]
    ).to_numpy(),
    1.0,
)

_test_abm_capacity = (
    _test_abm_layer.aggregate_active_capacity()
)

for _pathway in RESOURCE_RECOVERY_PATHWAYS:
    assert np.isclose(
        _test_abm_capacity[_pathway],
        INITIAL_CAPACITY[_pathway],
    )

for _pathway in INCUMBENT_TREATMENT_PATHWAYS:
    assert (
        _test_abm_layer.recyclers["technology"]
        == _pathway
    ).any()

assert not (
    _test_abm_layer.recyclers["technology"]
    == "solvolysis"
).any()

assert not _test_abm_layer.manufacturers["adopted"].any()

for _pathway in RESOURCE_RECOVERY_PATHWAYS:
    assert np.isclose(
        _test_abm_layer.operator_private_pathway_cost(
            _pathway,
            {"processing_cost": INITIAL_PROCESSING_COST},
        ),
        _test_abm_layer.effective_gate_fee(_pathway),
    )

assert _test_abm_layer.recyclers[
    "switching_threshold"
].ge(ABM_RECYCLER_SWITCHING_THRESHOLD_MIN).all()

assert _test_abm_layer.recyclers[
    "discount_rate"
].between(0.0, 1.0, inclusive="neither").all()

assert _test_abm_layer.recyclers[
    "exit_threshold"
].between(0.0, 1.0).all()

assert _test_abm_layer.manufacturers[
    "adoption_threshold"
].between(0.0, 1.0).all()

assert _test_abm_layer.manufacturers[
    "perceived_risk"
].between(0.0, 1.0).all()

# Heterogeneity must be present in every behavioral population.
assert _test_abm_layer.operators["cost_weight"].nunique() > 1
assert _test_abm_layer.operators["choice_sensitivity"].nunique() > 1
assert _test_abm_layer.recyclers["switching_threshold"].nunique() > 1
assert _test_abm_layer.recyclers["discount_rate"].nunique() > 1
assert _test_abm_layer.recyclers["exit_threshold"].nunique() > 1
assert _test_abm_layer.manufacturers["adoption_threshold"].nunique() > 1
assert _test_abm_layer.manufacturers["perceived_risk"].nunique() > 1
assert _test_abm_layer.manufacturers["adoption_sensitivity"].nunique() > 1

# Rebuilding with the same seed must reproduce all active populations.
_test_abm_layer_repeat = _build_test_abm_layer()

pd.testing.assert_frame_equal(
    _test_abm_layer.operators,
    _test_abm_layer_repeat.operators,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    _test_abm_layer.recyclers,
    _test_abm_layer_repeat.recyclers,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    _test_abm_layer.manufacturers,
    _test_abm_layer_repeat.manufacturers,
    check_exact=True,
)



In [ ]:
# ============================================================
# 8. Hybrid SD-ABM model runner
# ============================================================

class WTBHybridModel:
    """
    Hybrid SD-ABM model runner.

    The model couples:

    - SDLayer:
      aggregate stocks, flows, treatment capacity, recovered-material
      inventories, technology maturity, learning-by-doing, market demand,
      and system-level indicators;

    - ABMLayer:
      heterogeneous decisions by wind farm operators, recyclers,
      and manufacturers or downstream users.

    Annual coupling sequence:

        1. agents observe the opening SD state;
        2. agents make decentralized decisions;
        3. decisions are aggregated by the ABM layer;
        4. the SD layer updates stocks, flows, capacity, and maturity;
        5. annual indicators are calculated and recorded.
    """

    def __init__(
        self,
        scenario_config,
        start_year=START_YEAR,
        end_year=END_YEAR,
        seed=None,
        n_operators=N_OPERATORS,
        n_recyclers=N_RECYCLERS,
        n_manufacturers=N_MANUFACTURERS,
    ):
        if end_year < start_year:
            raise ValueError(
                "end_year must be equal to or greater than start_year."
            )

        self.config = scenario_config
        self.start_year = int(start_year)
        self.end_year = int(end_year)

        self.years = np.arange(
            self.start_year,
            self.end_year + 1,
            dtype=int,
        )

        self.seed = seed
        self.rng = np.random.default_rng(seed)

        self.abm = ABMLayer(
            scenario_config=scenario_config,
            rng=self.rng,
            n_operators=n_operators,
            n_recyclers=n_recyclers,
            n_manufacturers=n_manufacturers,
        )

        initial_capacity = INITIAL_CAPACITY.copy()
        initial_capacity.update(
            self.abm.aggregate_active_capacity()
        )

        self.sd = SDLayer(
            scenario_config=scenario_config,
            initial_capacity=initial_capacity,
        )

        self.history = []


    # ============================================================
    # Annual simulation step
    # ============================================================

    def step(self, year):
        """
        Executes one annual bidirectional SD-ABM coupling cycle.
        """
        year = int(year)

        # --------------------------------------------------------
        # 1. Opening system state observed by agents
        # --------------------------------------------------------

        system_state = self.sd.get_system_state(
            year=year
        )

        opening_capacity = (
            system_state["capacity"].copy()
        )

        opening_untreated_stock = float(
            system_state["untreated_stock"]
        )

        opening_trl_solvolysis = float(
            system_state["trl_solvolysis"]
        )

        opening_solvolysis_available = bool(
            system_state["solvolysis_available"]
        )

        # --------------------------------------------------------
        # 2. Agent decisions
        # --------------------------------------------------------

        abm_outputs = self.abm.step(
            omega_t=system_state,
            year=year,
        )

        required_abm_outputs = {
            "desired_flows",
            "actual_flows",
            "unmet_flow",
            "unmet_flows_by_pathway",
            "pathway_shares_desired",
            "capacity_changes",
            "closing_recycler_capacity_by_pathway",
            "manufacturer_adoption_readiness",
            "manufacturer_realized_adoption_share",
            "total_allocatable_flow",
        }

        missing_outputs = (
            required_abm_outputs
            - set(abm_outputs)
        )

        if missing_outputs:
            raise KeyError(
                "ABMLayer.step() is missing required outputs: "
                f"{sorted(missing_outputs)}"
            )

        # --------------------------------------------------------
        # 3. Aggregate SD update
        # --------------------------------------------------------

        sd_update_outputs = self.sd.update(
            abm_outputs=abm_outputs,
            year=year,
        )

        # --------------------------------------------------------
        # 3a. Validate mandatory SD market outputs
        # --------------------------------------------------------

        required_sd_market_outputs = {
            "potential_closed_loop_demand",
            "voluntary_closed_loop_demand",
            "mandatory_closed_loop_demand",
            "effective_closed_loop_demand",
            "realized_voluntary_closed_loop_use",
            "realized_mandatory_closed_loop_use",
            "unmet_voluntary_closed_loop_demand",
            "unmet_mandatory_closed_loop_demand",
            "total_unmet_closed_loop_demand",
            "mandatory_compliance_ratio",
            "closed_loop_supply_coverage_ratio",
            "total_closed_loop_utilization",
            "high_quality_recovered_material",
        }

        missing_sd_market_outputs = (
            required_sd_market_outputs
            - set(sd_update_outputs)
        )

        if missing_sd_market_outputs:
            raise KeyError(
                "SDLayer.update() is missing required market outputs: "
                f"{sorted(missing_sd_market_outputs)}"
            )

        # Canonical closed-loop market variables
        potential_closed_loop_demand = float(
            sd_update_outputs["potential_closed_loop_demand"]
        )
        voluntary_closed_loop_demand = float(
            sd_update_outputs["voluntary_closed_loop_demand"]
        )
        mandatory_closed_loop_demand = float(
            sd_update_outputs["mandatory_closed_loop_demand"]
        )
        effective_closed_loop_demand = float(
            sd_update_outputs["effective_closed_loop_demand"]
        )
        realized_voluntary_closed_loop_use = float(
            sd_update_outputs["realized_voluntary_closed_loop_use"]
        )
        realized_mandatory_closed_loop_use = float(
            sd_update_outputs["realized_mandatory_closed_loop_use"]
        )
        unmet_voluntary_closed_loop_demand = float(
            sd_update_outputs["unmet_voluntary_closed_loop_demand"]
        )
        unmet_mandatory_closed_loop_demand = float(
            sd_update_outputs["unmet_mandatory_closed_loop_demand"]
        )
        total_unmet_closed_loop_demand = float(
            sd_update_outputs["total_unmet_closed_loop_demand"]
        )
        total_closed_loop_utilization = float(
            sd_update_outputs["total_closed_loop_utilization"]
        )
        high_quality_recovered_material = float(
            sd_update_outputs["high_quality_recovered_material"]
        )
        mandatory_compliance_ratio = float(
            sd_update_outputs["mandatory_compliance_ratio"]
        )
        closed_loop_supply_coverage_ratio = float(
            sd_update_outputs["closed_loop_supply_coverage_ratio"]
        )

        # --------------------------------------------------------
        # 3b. Closed-loop market consistency checks
        # --------------------------------------------------------

        consistency_atol = max(
            float(NUMERICAL_EPSILON),
            1e-9,
        )

        if not np.isclose(
            effective_closed_loop_demand,
            voluntary_closed_loop_demand
            + mandatory_closed_loop_demand,
            atol=consistency_atol,
            rtol=1e-9,
        ):
            raise AssertionError(
                "Effective closed-loop demand is inconsistent with "
                "voluntary plus mandatory demand. "
                f"Year={year}, effective={effective_closed_loop_demand}, "
                f"voluntary={voluntary_closed_loop_demand}, "
                f"mandatory={mandatory_closed_loop_demand}."
            )

        if not np.isclose(
            total_closed_loop_utilization,
            realized_voluntary_closed_loop_use
            + realized_mandatory_closed_loop_use,
            atol=consistency_atol,
            rtol=1e-9,
        ):
            raise AssertionError(
                "Total closed-loop utilization is inconsistent with "
                "realized voluntary plus mandatory use. "
                f"Year={year}, total={total_closed_loop_utilization}, "
                f"voluntary={realized_voluntary_closed_loop_use}, "
                f"mandatory={realized_mandatory_closed_loop_use}."
            )

        if (
            total_closed_loop_utilization
            > effective_closed_loop_demand
            + consistency_atol
        ):
            raise AssertionError(
                "Closed-loop utilization exceeds effective demand. "
                f"Year={year}, use={total_closed_loop_utilization}, "
                f"demand={effective_closed_loop_demand}."
            )

        if (
            total_closed_loop_utilization
            > high_quality_recovered_material
            + consistency_atol
        ):
            raise AssertionError(
                "Closed-loop utilization exceeds available "
                "high-quality recovered material. "
                f"Year={year}, use={total_closed_loop_utilization}, "
                f"supply={high_quality_recovered_material}."
            )

        if not np.isclose(
            total_unmet_closed_loop_demand,
            unmet_voluntary_closed_loop_demand
            + unmet_mandatory_closed_loop_demand,
            atol=consistency_atol,
            rtol=1e-9,
        ):
            raise AssertionError(
                "Total unmet closed-loop demand is inconsistent with "
                "its voluntary and mandatory components. "
                f"Year={year}, total={total_unmet_closed_loop_demand}, "
                f"voluntary={unmet_voluntary_closed_loop_demand}, "
                f"mandatory={unmet_mandatory_closed_loop_demand}."
            )

        if not np.isclose(
            total_unmet_closed_loop_demand,
            max(
                0.0,
                effective_closed_loop_demand
                - total_closed_loop_utilization,
            ),
            atol=consistency_atol,
            rtol=1e-9,
        ):
            raise AssertionError(
                "Unmet closed-loop demand does not equal effective "
                "demand minus realized utilization. "
                f"Year={year}, unmet={total_unmet_closed_loop_demand}, "
                f"demand={effective_closed_loop_demand}, "
                f"use={total_closed_loop_utilization}."
            )

        if (
            realized_voluntary_closed_loop_use
            > voluntary_closed_loop_demand
            + consistency_atol
        ):
            raise AssertionError(
                "Realized voluntary closed-loop use exceeds "
                "voluntary demand."
            )

        if (
            realized_mandatory_closed_loop_use
            > mandatory_closed_loop_demand
            + consistency_atol
        ):
            raise AssertionError(
                "Realized mandatory closed-loop use exceeds "
                "mandatory demand."
            )

        closing_recycler_capacity = abm_outputs.get(
            "closing_recycler_capacity_by_pathway",
            {},
        )
        capacity_sync_error = max(
            (
                abs(
                    float(self.sd.capacity.get(pathway, 0.0))
                    - float(
                        closing_recycler_capacity.get(pathway, 0.0)
                    )
                )
                for pathway in RESOURCE_RECOVERY_PATHWAYS
            ),
            default=0.0,
        )

        if capacity_sync_error > CAPACITY_SYNC_TOLERANCE:
            raise AssertionError(
                "SD and ABM recycler capacities are inconsistent. "
                f"Maximum error: {capacity_sync_error}"
            )

        # --------------------------------------------------------
        # 4. System-level indicators
        # --------------------------------------------------------

        indicators = self.sd.compute_indicators(
            abm_outputs=abm_outputs,
            sd_update_outputs=sd_update_outputs,
        )

        required_indicator_outputs = {
            "closed_loop_material_flow",
            "total_closed_loop_utilization",
            "high_quality_recovered_material",
        }

        missing_indicator_outputs = (
            required_indicator_outputs
            - set(indicators)
        )

        if missing_indicator_outputs:
            raise KeyError(
                "SDLayer.compute_indicators() is missing required "
                "closed-loop indicators: "
                f"{sorted(missing_indicator_outputs)}"
            )

        if not np.isclose(
            float(indicators["closed_loop_material_flow"]),
            total_closed_loop_utilization,
            atol=consistency_atol,
            rtol=1e-9,
        ):
            raise AssertionError(
                "closed_loop_material_flow from compute_indicators() "
                "does not equal total_closed_loop_utilization from "
                "SDLayer.update(). "
                f"Year={year}, indicator="
                f"{float(indicators['closed_loop_material_flow'])}, "
                f"update={total_closed_loop_utilization}."
            )

        if not np.isclose(
            float(indicators["total_closed_loop_utilization"]),
            total_closed_loop_utilization,
            atol=consistency_atol,
            rtol=1e-9,
        ):
            raise AssertionError(
                "The closed-loop utilization reported by "
                "compute_indicators() differs from SDLayer.update()."
            )

        if not np.isclose(
            float(indicators["high_quality_recovered_material"]),
            high_quality_recovered_material,
            atol=consistency_atol,
            rtol=1e-9,
        ):
            raise AssertionError(
                "The high-quality recovered-material indicator differs "
                "from the SD annual-update output."
            )

        # --------------------------------------------------------
        # 5. Annual record
        # --------------------------------------------------------

        row = {
            # Identification
"year": year,
"scenario": self.config.name,
"scenario_label": SCENARIO_LABELS.get(
    self.config.name,
    self.config.name,
),
"seed": self.seed,

            # Annual WTB inflow and treatment
            "decommissioned": indicators[
                "decommissioned"
            ],
            "total_treated": indicators[
                "total_treated"
            ],
            "annual_treatment_to_inflow_ratio": indicators[
                "annual_treatment_to_inflow_ratio"
            ],
            "desired_total_flow": sum(
                float(value)
                for value
                in abm_outputs["desired_flows"].values()
            ),
            "total_allocatable_flow": float(
                abm_outputs.get(
                    "total_allocatable_flow",
                    sum(
                        float(value)
                        for value
                        in abm_outputs["desired_flows"].values()
                    ),
                )
            ),
            "unmet_flow": float(
                abm_outputs.get(
                    "unmet_flow",
                    0.0,
                )
            ),
            "backlog_considered": float(
                abm_outputs.get(
                    "backlog_considered",
                    0.0,
                )
            ),

            # Untreated WTB stock
            "opening_untreated_stock": (
                opening_untreated_stock
            ),
            "untreated_stock": indicators[
                "untreated_stock"
            ],
            "untreated_stock_share": indicators[
                "untreated_stock_share"
            ],
            "net_untreated_stock_increase": indicators[
                "net_untreated_stock_increase"
            ],
            "net_untreated_stock_increase_share": indicators[
                "net_untreated_stock_increase_share"
            ],
            "stock_drawdown": indicators[
                "stock_drawdown"
            ],
            "cumulative_decommissioned": indicators[
                "cumulative_decommissioned"
            ],
            "cumulative_treated": indicators[
                "cumulative_treated"
            ],

            # Treatment pathway diffusion
            "direct_reuse_flow": indicators[
                "direct_reuse_flow"
            ],
            "direct_reuse_share": indicators[
                "direct_reuse_share"
            ],
            "incumbent_open_loop_material_flow": indicators[
                "incumbent_open_loop_material_flow"
            ],
            "incumbent_open_loop_material_treatment_share": indicators[
                "incumbent_open_loop_material_treatment_share"
            ],
            "energy_recovery_flow": indicators[
                "energy_recovery_flow"
            ],
            "energy_recovery_treatment_share": indicators[
                "energy_recovery_treatment_share"
            ],
            "solvolysis_treated_flow": indicators[
                "solvolysis_treated_flow"
            ],
            "solvolysis_treatment_share": indicators[
                "solvolysis_treatment_share"
            ],
            "solvolysis_recoverable_glass_fibre_input": indicators[
                "solvolysis_recoverable_glass_fibre_input"
            ],
            "solvolysis_recovery_efficiency": indicators[
                "solvolysis_recovery_efficiency"
            ],
            "solvolysis_whole_wtb_output_yield": indicators[
                "solvolysis_whole_wtb_output_yield"
            ],

            # Closed-loop outcome
            "closed_loop_material_flow": indicators[
                "closed_loop_material_flow"
            ],
            "closed_loop_equivalent_wtb_flow": indicators[
                "closed_loop_equivalent_wtb_flow"
            ],
            "closed_loop_wtb_flow_share": indicators[
                "closed_loop_wtb_flow_share"
            ],

            # Annual inflow ratios
            "annual_direct_reuse_to_inflow_ratio": indicators[
                "annual_direct_reuse_to_inflow_ratio"
            ],
            "annual_open_loop_material_to_inflow_ratio": indicators[
                "annual_open_loop_material_to_inflow_ratio"
            ],
            "annual_energy_recovery_to_inflow_ratio": indicators[
                "annual_energy_recovery_to_inflow_ratio"
            ],
            "annual_solvolysis_to_inflow_ratio": indicators[
                "annual_solvolysis_to_inflow_ratio"
            ],

            # Recovered-material utilization
"open_loop_demand": indicators[
    "open_loop_demand"
],
"total_available_recovered_material": indicators[
    "total_available_recovered_material"
],
"total_open_loop_utilization": indicators[
    "total_open_loop_utilization"
],
            "total_closed_loop_utilization": indicators[
                "total_closed_loop_utilization"
            ],
            "recovered_material_utilization_share": indicators[
                "recovered_material_utilization_share"
            ],
            "open_loop_utilization_share": indicators[
                "open_loop_utilization_share"
            ],
            "closed_loop_utilization_share": indicators[
                "closed_loop_utilization_share"
            ],
            "total_unutilized_material": indicators[
                "total_unutilized_material"
            ],
            "unutilized_material_share": indicators[
                "unutilized_material_share"
            ],

            # Capacity
            "total_opening_capacity": indicators[
                "total_opening_capacity"
            ],
            "total_closing_capacity": indicators[
                "total_closing_capacity"
            ],
            "opening_capacity_gap": indicators[
                "opening_capacity_gap"
            ],
            # aggregate_capacity_shortage = system-wide demand > total capacity
            "aggregate_capacity_shortage": float(
                abm_outputs.get("total_capacity_shortage", 0.0)
            ),
            "capacity_allocation_mismatch": float(
                abm_outputs.get("capacity_allocation_mismatch", 0.0)
            ),
            "initial_rejected_flow": float(
                abm_outputs.get("initial_rejected_flow", 0.0)
            ),
            "reallocated_flow": float(
                abm_outputs.get("reallocated_flow", 0.0)
            ),
            "unused_capacity_after_rerouting": float(
                abm_outputs.get(
                    "unused_capacity_after_rerouting", 0.0
                )
            ),
            "backlog_not_reconsidered": max(
                0.0,
                opening_untreated_stock
                - float(abm_outputs.get("backlog_considered", 0.0)),
            ),
            "closed_loop_demand_gap": (
                total_unmet_closed_loop_demand
            ),
            "high_quality_material_not_used_closed_loop": max(
                0.0,
                high_quality_recovered_material
                - total_closed_loop_utilization,
            ),
            "capacity_sync_error": float(capacity_sync_error),
            "treatment_capacity_utilization": indicators[
                "treatment_capacity_utilization"
            ],

            # Economic outcome and internal environmental diagnostic
            "virgin_material_displacement": indicators[
                "virgin_material_displacement"
            ],
            "exploratory_environmental_proxy": indicators[
                "exploratory_environmental_proxy"
            ],
            "average_processing_cost": indicators[
                "average_processing_cost"
            ],

            # Market formation
            # These variables are taken directly from the SD annual-update
            # outputs, which are their canonical source.
            "new_blade_glass_fibre_demand": float(
                sd_update_outputs[
                    "new_blade_glass_fibre_demand"
                ]
            ),
            "potential_closed_loop_demand": (
                potential_closed_loop_demand
            ),
            "voluntary_closed_loop_demand": (
                voluntary_closed_loop_demand
            ),
            "mandatory_closed_loop_demand": (
                mandatory_closed_loop_demand
            ),
            "effective_closed_loop_demand": (
                effective_closed_loop_demand
            ),
            "realized_voluntary_closed_loop_use": (
                realized_voluntary_closed_loop_use
            ),
            "realized_mandatory_closed_loop_use": (
                realized_mandatory_closed_loop_use
            ),
            "unmet_voluntary_closed_loop_demand": (
                unmet_voluntary_closed_loop_demand
            ),
            "unmet_mandatory_closed_loop_demand": (
                unmet_mandatory_closed_loop_demand
            ),
            "total_unmet_closed_loop_demand": (
                total_unmet_closed_loop_demand
            ),
            "mandatory_compliance_ratio": (
                mandatory_compliance_ratio
            ),
            "mandatory_compliance_applicable": bool(
                mandatory_closed_loop_demand
                > NUMERICAL_EPSILON
            ),
            "closed_loop_supply_coverage_ratio": (
                closed_loop_supply_coverage_ratio
            ),
            "closed_loop_demand_active": bool(
                effective_closed_loop_demand
                > NUMERICAL_EPSILON
            ),
            "high_quality_recovered_material": (
                high_quality_recovered_material
            ),

            # Solvolysis technology state
            "opening_trl_solvolysis": (
                opening_trl_solvolysis
            ),
            "opening_solvolysis_available": (
                opening_solvolysis_available
            ),
            "trl_solvolysis": indicators[
                "trl_solvolysis"
            ],
            "solvolysis_available": indicators[
                "solvolysis_available"
            ],

            # Manufacturer or downstream-user outcomes
            "manufacturer_adoption_readiness": float(
                abm_outputs.get(
                    "manufacturer_adoption_readiness",
                    0.0,
                )
            ),
            "manufacturer_realized_adoption_share": float(
                abm_outputs.get(
                    "manufacturer_realized_adoption_share",
                    0.0,
                )
            ),
            "perceived_risk": float(
                abm_outputs.get(
                    "perceived_risk",
                    0.0,
                )
            ),
            "supply_reliability": float(
                abm_outputs.get(
                    "supply_reliability",
                    0.0,
                )
            ),
            "certification_score": float(
                abm_outputs.get(
                    "certification_score",
                    0.0,
                )
            ),
            "price_advantage": float(
                abm_outputs.get(
                    "price_advantage",
                    0.0,
                )
            ),

            # Recycler outcomes
            "switches_to_solvolysis": int(
                abm_outputs.get(
                    "switches_to_solvolysis",
                    0,
                )
            ),
            "switched_incumbent_capacity": float(
                abm_outputs.get(
                    "switched_incumbent_capacity",
                    0.0,
                )
            ),
            "maximum_switched_incumbent_capacity": float(
                abm_outputs.get(
                    "maximum_switched_incumbent_capacity",
                    0.0,
                )
            ),
            "recycler_exits": int(
                abm_outputs.get(
                    "recycler_exits",
                    0,
                )
            ),
            "recycler_expansions": int(
                abm_outputs.get(
                    "recycler_expansions",
                    0,
                )
            ),
            "recycler_contractions": int(
                abm_outputs.get(
                    "recycler_contractions",
                    0,
                )
            ),
            "total_expansion_capex": float(
                abm_outputs.get("total_expansion_capex", 0.0)
            ),
            "total_switching_capex": float(
                abm_outputs.get("total_switching_capex", 0.0)
            ),
            "total_entry_capex": float(
                abm_outputs.get("total_entry_capex", 0.0)
            ),
            "total_annual_capex": float(
                abm_outputs.get("total_annual_capex", 0.0)
            ),
            "new_solvolysis_entries": int(
                abm_outputs.get("new_solvolysis_entries", 0)
            ),
            "new_manufacturer_adoptions": int(
                abm_outputs.get("new_manufacturer_adoptions", 0)
            ),
            "manufacturer_abandonments": int(
                abm_outputs.get("manufacturer_abandonments", 0)
            ),
            "active_recyclers": int(
                abm_outputs.get(
                    "active_recyclers",
                    0,
                )
            ),
            "solvolysis_recyclers": int(
                abm_outputs.get(
                    "solvolysis_recyclers",
                    0,
                )
            ),
        }

        # --------------------------------------------------------
        # Pathway-specific flows, capacities, and costs
        # --------------------------------------------------------

        for pathway in PATHWAYS:
            row[
                f"desired_flow_{pathway}"
            ] = float(
                abm_outputs[
                    "desired_flows"
                ].get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"flow_{pathway}"
            ] = float(
                sd_update_outputs[
                    "actual_flows"
                ].get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"unmet_flow_{pathway}"
            ] = float(
                abm_outputs[
                    "unmet_flows_by_pathway"
                ].get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"desired_share_{pathway}"
            ] = float(
                abm_outputs[
                    "pathway_shares_desired"
                ].get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"opening_capacity_{pathway}"
            ] = float(
                opening_capacity.get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"capacity_{pathway}"
            ] = float(
                self.sd.capacity.get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"capacity_change_{pathway}"
            ] = float(
                abm_outputs[
                    "capacity_changes"
                ].get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"cost_{pathway}"
            ] = float(
                self.sd.processing_cost.get(
                    pathway,
                    0.0,
                )
            )

            row[f"operator_private_cost_{pathway}"] = float(
                self.abm.operator_private_pathway_cost(
                    pathway, system_state
                )
            )

            if pathway in RESOURCE_RECOVERY_PATHWAYS:
                row[f"gate_fee_{pathway}"] = float(
                    self.abm.effective_gate_fee(pathway)
                )

        # --------------------------------------------------------
        # Recovered-material indicators
        # --------------------------------------------------------

        for pathway in MATERIAL_RECOVERY_PATHWAYS:
            row[f"closed_loop_eligible_{pathway}"] = bool(
                sd_update_outputs[
                    "closed_loop_eligibility_by_pathway"
                ].get(pathway, False)
            )
            row[f"eligible_closed_loop_supply_{pathway}"] = float(
                sd_update_outputs[
                    "eligible_closed_loop_supply_by_pathway"
                ].get(pathway, 0.0)
            )

            row[
                f"recovered_generation_{pathway}"
            ] = float(
                sd_update_outputs[
                    "recovered_generation"
                ].get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"available_recovered_material_{pathway}"
            ] = float(
                sd_update_outputs[
                    "available_recovered_material"
                ].get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"open_loop_utilization_{pathway}"
            ] = float(
                sd_update_outputs[
                    "open_loop_utilization_by_pathway"
                ].get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"closed_loop_utilization_{pathway}"
            ] = float(
                sd_update_outputs[
                    "closed_loop_utilization_by_pathway"
                ].get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"recovered_stock_{pathway}"
            ] = float(
                self.sd.recovered_stock.get(
                    pathway,
                    0.0,
                )
            )

            row[
                f"inventory_loss_{pathway}"
            ] = float(
                sd_update_outputs[
                    "inventory_losses"
                ].get(pathway, 0.0)
            )

            row[
                f"material_sales_ratio_{pathway}"
            ] = float(
                sd_update_outputs[
                    "material_sales_ratio_by_pathway"
                ].get(pathway, 0.0)
            )

        row["recovery_resource_credit"] = float(
            sd_update_outputs[
                "recovery_resource_credit"
            ]
        )

        # --------------------------------------------------------
        # Mean recycler profitability
        # --------------------------------------------------------

        mean_recycler_profit = (
            abm_outputs.get(
                "mean_recycler_profit",
                {},
            )
        )

        for pathway in RESOURCE_RECOVERY_PATHWAYS:
            row[
                f"mean_recycler_profit_{pathway}"
            ] = float(
                mean_recycler_profit.get(
                    pathway,
                    0.0,
                )
            )

        # --------------------------------------------------------
        # Annual record consistency checks
        # --------------------------------------------------------

        numeric_values = [
            value
            for value in row.values()
            if isinstance(
                value,
                (int, float, np.integer, np.floating),
            )
        ]

        if not all(
            np.isfinite(float(value))
            for value in numeric_values
        ):
            raise ValueError(
                f"Non-finite annual result detected in year {year}."
            )

        if (
            row["total_treated"]
            > row["opening_untreated_stock"]
            + row["decommissioned"]
            + NUMERICAL_EPSILON
        ):
            raise AssertionError(
                "Annual treatment exceeds available WTB material."
            )

        if row["mandatory_compliance_ratio"] < 0.0:
            raise AssertionError(
                "mandatory_compliance_ratio cannot be negative."
            )

        if (
            row["mandatory_compliance_ratio"]
            > 1.0 + consistency_atol
        ):
            raise AssertionError(
                "mandatory_compliance_ratio cannot exceed 1."
            )

        if (
            row["closed_loop_supply_coverage_ratio"]
            < -consistency_atol
        ):
            raise AssertionError(
                "closed_loop_supply_coverage_ratio cannot be negative."
            )

        if (
            row["closed_loop_supply_coverage_ratio"]
            > 1.0 + consistency_atol
        ):
            raise AssertionError(
                "closed_loop_supply_coverage_ratio cannot exceed 1."
            )

        self.history.append(row)

        return row


    # ============================================================
    # Full model run
    # ============================================================

    def run(self):
        """
        Runs the coupled model over the complete simulation horizon.
        """
        self.history = []

        for year in self.years:
            self.step(year)

        return pd.DataFrame(
            self.history
        )


# ============================================================
# Scenario execution
# ============================================================

def run_all_scenarios(
    seed=42,
    n_operators=N_OPERATORS,
    n_recyclers=N_RECYCLERS,
    n_manufacturers=N_MANUFACTURERS,
    verbose=True,
):
    """
    Runs one coupled simulation for each scenario.

    The same random seed is used for every scenario so that all scenarios
    start from equivalent heterogeneous populations and comparable random
    streams. This common-random-number design reduces stochastic noise in
    scenario comparisons.
    """
    scenario_results = []

    for scenario_name in SCENARIO_ORDER:
        scenario_config = SCENARIOS[
            scenario_name
        ]

        if verbose:
            print(
                f"Running scenario: "
                f"{SCENARIO_LABELS[scenario_name]} "
                f"(seed={seed})"
            )

        model = WTBHybridModel(
            scenario_config=scenario_config,
            start_year=START_YEAR,
            end_year=END_YEAR,
            seed=seed,
            n_operators=n_operators,
            n_recyclers=n_recyclers,
            n_manufacturers=n_manufacturers,
        )

        scenario_result = model.run()

        scenario_results.append(
            scenario_result
        )

    if not scenario_results:
        return pd.DataFrame()

    return pd.concat(
        scenario_results,
        ignore_index=True,
    )

# ============================================================
# Basic hybrid-runner checks
# ============================================================

_test_hybrid_scenario_name = next(iter(SCENARIOS))

_test_hybrid_model = WTBHybridModel(
    scenario_config=SCENARIOS[_test_hybrid_scenario_name],
    start_year=START_YEAR,
    end_year=START_YEAR,
    seed=AGENT_POPULATION_SEED,
    n_operators=N_OPERATORS,
    n_recyclers=N_RECYCLERS,
    n_manufacturers=N_MANUFACTURERS,
)

assert np.isclose(
    _test_hybrid_model.sd.capacity["solvolysis"],
    0.0,
)

_test_hybrid_result = _test_hybrid_model.run()

assert len(_test_hybrid_result) == 1
assert int(_test_hybrid_result.iloc[0]["year"]) == START_YEAR

_required_hybrid_columns = {
    "manufacturer_realized_adoption_share",
    "voluntary_closed_loop_demand",
    "mandatory_closed_loop_demand",
    "realized_voluntary_closed_loop_use",
    "realized_mandatory_closed_loop_use",
    "unmet_voluntary_closed_loop_demand",
    "unmet_mandatory_closed_loop_demand",
    "mandatory_compliance_ratio",
    "aggregate_capacity_shortage",
    "capacity_allocation_mismatch",
    "unused_capacity_after_rerouting",
}

assert _required_hybrid_columns.issubset(
    _test_hybrid_result.columns
)


In [ ]:
# ============================================================
# 9. Run simulations
# ============================================================

results = run_all_scenarios(
    seed=AGENT_POPULATION_SEED,
    verbose=True,
)

print("Results shape:", results.shape)
print("Scenarios:", results["scenario"].unique())
print(
    "Years:",
    results["year"].min(),
    "-",
    results["year"].max(),
)

results.head()

In [ ]:
# ============================================================
# Diagnostic: recovered-material supply versus market demand
# ============================================================

diagnostic_columns = [
    "scenario_label",
    "year",
    "open_loop_demand",
    "potential_closed_loop_demand",
    "effective_closed_loop_demand",
    "total_available_recovered_material",
    "total_open_loop_utilization",
    "total_closed_loop_utilization",
    "total_unutilized_material",
    "recovered_material_utilization_share",
    "unutilized_material_share",
]

missing_columns = [
    column
    for column in diagnostic_columns
    if column not in results.columns
]

if missing_columns:
    raise KeyError(
        "The results table is missing diagnostic columns: "
        f"{missing_columns}"
    )

material_market_diagnostic = (
    results[diagnostic_columns]
    .copy()
    .sort_values(
        ["scenario_label", "year"]
    )
    .reset_index(drop=True)
)

material_market_diagnostic[
    "total_effective_material_demand"
] = (
    material_market_diagnostic["open_loop_demand"]
    + material_market_diagnostic[
        "effective_closed_loop_demand"
    ]
)

material_market_diagnostic[
    "material_demand_supply_ratio"
] = np.where(
    material_market_diagnostic[
        "total_available_recovered_material"
    ] > NUMERICAL_EPSILON,
    material_market_diagnostic[
        "total_effective_material_demand"
    ]
    / material_market_diagnostic[
        "total_available_recovered_material"
    ],
    np.nan,
)

material_market_diagnostic[
    "open_loop_demand_surplus"
] = (
    material_market_diagnostic["open_loop_demand"]
    - material_market_diagnostic[
        "total_open_loop_utilization"
    ]
)

display(
    material_market_diagnostic[
        material_market_diagnostic["year"].isin(
            [2026, 2030, 2035, 2040, 2045, 2050]
        )
    ]
)

In [ ]:
# ============================================================
# Basic simulation-output checks
# ============================================================

expected_rows = (
    len(SCENARIO_ORDER)
    * len(YEARS)
)

assert len(results) == expected_rows

assert results["scenario"].nunique() == len(
    SCENARIO_ORDER
)

assert results["year"].min() == START_YEAR
assert results["year"].max() == END_YEAR

assert not results.duplicated(
    subset=["scenario", "year"]
).any()

numeric_results = results.select_dtypes(
    include=[np.number]
)

assert np.isfinite(
    numeric_results.to_numpy()
).all()

assert (
    results["untreated_stock"] >= 0.0
).all()

assert (
    results["total_treated"] >= 0.0
).all()

assert results[
    "manufacturer_realized_adoption_share"
].between(0.0, 1.0).all()

assert results[
    "mandatory_compliance_ratio"
].between(0.0, 1.0).all()

print("All basic simulation checks passed.")

In [ ]:
# ============================================================
# 10. Final-year results
# ============================================================

# ------------------------------------------------------------
# 1. Backward-compatible column harmonization
# ------------------------------------------------------------

# These mappings are used only when loading results generated
# with earlier notebook versions.
#
# NOTE:
# "open_loop_wtb_flow_share" is intentionally not mapped to
# "incumbent_open_loop_material_treatment_share".
#
# The former may include high-quality recovered material used
# outside the blade-manufacturing closed loop, whereas the latter
# measures treatment through incumbent open-loop technologies.
# Treating them as equivalent would be methodologically misleading.

column_renaming = {
    "recovered_utilization":
        "recovered_material_utilization_share",


    "closed_loop_wtb_flow":
        "closed_loop_equivalent_wtb_flow",
}

for old_name, new_name in column_renaming.items():
    if (
        new_name not in results.columns
        and old_name in results.columns
    ):
        results[new_name] = results[old_name]


# ------------------------------------------------------------
# 2. Basic input validation
# ------------------------------------------------------------

required_identification_columns = {
    "scenario",
    "year",
}

missing_identification_columns = (
    required_identification_columns
    - set(results.columns)
)

if missing_identification_columns:
    raise KeyError(
        "The results DataFrame is missing required columns: "
        f"{sorted(missing_identification_columns)}"
    )

if results.empty:
    raise ValueError(
        "The results DataFrame is empty. Run the simulations first."
    )


# ------------------------------------------------------------
# 3. Extract the final simulated year for each scenario
# ------------------------------------------------------------

final_results = (
    results
    .sort_values(
        ["scenario", "year"]
    )
    .groupby(
        "scenario",
        as_index=False,
    )
    .tail(1)
    .reset_index(drop=True)
)

final_results["scenario_label"] = (
    final_results["scenario"]
    .map(SCENARIO_LABELS)
    .fillna(final_results["scenario"])
)


# ------------------------------------------------------------
# 4. Final-year indicators to display
# ------------------------------------------------------------

final_columns = [
    "scenario_label",
    "year",

    # Technology diffusion and treatment structure
    "solvolysis_treatment_share",
    "incumbent_open_loop_material_treatment_share",
    "energy_recovery_treatment_share",

    # Circularity and material-use outcomes
    "closed_loop_wtb_flow_share",
    "recovered_material_utilization_share",
    "unutilized_material_share",

    # Untreated-stock outcome
    "untreated_stock_share",

    # Manufacturer adoption and demand formation
    "manufacturer_realized_adoption_share",
    "voluntary_closed_loop_demand",
    "mandatory_closed_loop_demand",
    "total_unmet_closed_loop_demand",
    "mandatory_compliance_ratio",
    "closed_loop_supply_coverage_ratio",

    # Capacity diagnostics
    "aggregate_capacity_shortage",
    "capacity_allocation_mismatch",
    "unused_capacity_after_rerouting",

    # Economic outcome and internal environmental diagnostic
    "average_processing_cost",
    "virgin_material_displacement",
]


# ------------------------------------------------------------
# 5. Validate final-year output columns
# ------------------------------------------------------------

missing_columns = [
    column
    for column in final_columns
    if column not in final_results.columns
]

if missing_columns:
    print(
        "Missing columns:",
        missing_columns,
    )

    print(
        "\nAvailable related columns:"
    )

    related_terms = [
        "open_loop",
        "closed_loop",
        "recovered",
        "solvolysis",
        "untreated",
        "manufacturer",
        "mandatory",
        "voluntary",
        "capacity",
        "utilization",
        "ghg",
        "cost",
    ]

    related_columns = [
        column
        for column in final_results.columns
        if any(
            term in column
            for term in related_terms
        )
    ]

    print(
        sorted(related_columns)
    )

else:
    final_results_display = (
        final_results[
            final_columns
        ]
        .copy()
    )

    share_columns = [
        "solvolysis_treatment_share",
        "incumbent_open_loop_material_treatment_share",
        "energy_recovery_treatment_share",
        "closed_loop_wtb_flow_share",
        "recovered_material_utilization_share",
        "unutilized_material_share",
        "untreated_stock_share",
        "manufacturer_realized_adoption_share",
        "mandatory_compliance_ratio",
        "closed_loop_supply_coverage_ratio",
    ]

    for column in share_columns:
        final_results_display[column] = (
            final_results_display[column]
            .astype(float)
            .clip(lower=0.0)
        )

    display(
        final_results_display
    )


# ------------------------------------------------------------
# 6. Final-year consistency checks
# ------------------------------------------------------------

expected_scenarios = set(
    SCENARIO_ORDER
)

observed_scenarios = set(
    final_results["scenario"]
)

missing_scenarios = (
    expected_scenarios
    - observed_scenarios
)

if missing_scenarios:
    raise AssertionError(
        "Final-year results are missing scenarios: "
        f"{sorted(missing_scenarios)}"
    )

if not (
    final_results["year"]
    == END_YEAR
).all():
    raise AssertionError(
        "At least one scenario does not contain results for END_YEAR."
    )

if (
    "manufacturer_realized_adoption_share"
    in final_results.columns
):
    assert final_results[
        "manufacturer_realized_adoption_share"
    ].between(
        0.0,
        1.0,
    ).all()

if (
    "mandatory_compliance_ratio"
    in final_results.columns
):
    assert final_results[
        "mandatory_compliance_ratio"
    ].between(
        0.0,
        1.0,
    ).all()

print(
    "Final-year results prepared successfully."
)


In [ ]:
diagnostic_columns = [
    "scenario_label",
    "year",
    "open_loop_demand",
    "available_recovered_material_mechanical_recycling",
    "available_recovered_material_pyrolysis",
    "available_recovered_material_solvolysis",
    "open_loop_utilization_mechanical_recycling",
    "open_loop_utilization_pyrolysis",
    "open_loop_utilization_solvolysis",
    "total_unutilized_material",
    "unutilized_material_share",
]

pathway_market_diagnostic = (
    results.loc[
        results["year"].isin([2040, 2045, 2050]),
        diagnostic_columns,
    ]
    .sort_values(["scenario_label", "year"])
    .reset_index(drop=True)
)

display(pathway_market_diagnostic)

pathway_market_diagnostic[
    "mechanical_open_loop_ceiling"
] = (
    pathway_market_diagnostic["open_loop_demand"]
    * OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY[
        "mechanical_recycling"
    ]
)

pathway_market_diagnostic[
    "pyrolysis_open_loop_ceiling"
] = (
    pathway_market_diagnostic["open_loop_demand"]
    * OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY[
        "pyrolysis"
    ]
)

pathway_market_diagnostic[
    "solvolysis_open_loop_ceiling"
] = (
    pathway_market_diagnostic["open_loop_demand"]
    * OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY[
        "solvolysis"
    ]
)

display(pathway_market_diagnostic)

In [ ]:
# ============================================================
# 11. Journal-style plotting configuration and utilities
# ============================================================

from pathlib import Path
import os
import re
import tempfile

import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Figure output directory
# ------------------------------------------------------------

OUTPUT_DIR = Path("tfsc_v21_1_outputs")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

for output_directory in (FIG_DIR, TABLE_DIR):
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# Journal-style global settings
# ------------------------------------------------------------

plt.rcParams.update({
    # Typography
    "font.family": "serif",
    "font.serif": [
        "Times New Roman",
        "Liberation Serif",
        "DejaVu Serif",
    ],
    "mathtext.fontset": "dejavuserif",

    # Font sizes
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8.5,
    "figure.titlesize": 12,

    # Lines and axes
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.8,
    "lines.markersize": 4,

    # Legend
    "legend.frameon": False,

    # Figure export
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,

    # Preserve editable text in vector files
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


# ------------------------------------------------------------
# Pathway labels and ordering
# ------------------------------------------------------------

PATHWAY_LABELS = {
    "reuse": "Reuse",
    "repurposing": "Repurposing",
    "mechanical_recycling": (
        "Mechanical recycling"
    ),
    "pyrolysis": "Pyrolysis",
    "recovery": "Recovery",
    "solvolysis": "Solvolysis",
}

# Use the pathway ordering defined in the model configuration.
PATHWAY_ORDER = list(PATHWAYS)

PATHWAY_FLOW_COLUMNS = {
    pathway: f"flow_{pathway}"
    for pathway in PATHWAY_ORDER
}


# ------------------------------------------------------------
# Pathway visual styles
# ------------------------------------------------------------
# Colors are combined with line styles and markers so that
# figures remain interpretable when printed in grayscale.

PATHWAY_COLORS = {
    "reuse": "#4477AA",
    "repurposing": "#66CCEE",
    "mechanical_recycling": "#228833",
    "pyrolysis": "#CCBB44",
    "recovery": "#AA3377",
    "solvolysis": "#EE6677",
}

PATHWAY_LINESTYLES = {
    "reuse": "-",
    "repurposing": "--",
    "mechanical_recycling": "-.",
    "pyrolysis": ":",
    "recovery": (0, (5, 2)),
    "solvolysis": (0, (3, 1, 1, 1)),
}

PATHWAY_MARKERS = {
    "reuse": "o",
    "repurposing": "s",
    "mechanical_recycling": "^",
    "pyrolysis": "D",
    "recovery": "v",
    "solvolysis": "P",
}


# ------------------------------------------------------------
# Scenario visual styles
# ------------------------------------------------------------

SCENARIO_COLORS = {
    "post_2025_baseline": "#4477AA",
    "demand_pull_circularity": "#CCBB44",
    "coordinated_transition": "#EE6677",
}

SCENARIO_LINESTYLES = {
    "post_2025_baseline": "--",
    "demand_pull_circularity": "-.",
    "coordinated_transition": "-",
}

SCENARIO_MARKERS = {
    "post_2025_baseline": "o",
    "demand_pull_circularity": "s",
    "coordinated_transition": "^",
}


# ------------------------------------------------------------
# Figure dimensions
# ------------------------------------------------------------
# Approximate journal-width dimensions in inches.

FIGSIZE_SINGLE_COLUMN = (
    3.50,
    2.75,
)

FIGSIZE_ONE_AND_HALF_COLUMN = (
    5.50,
    3.60,
)

FIGSIZE_DOUBLE_COLUMN = (
    7.20,
    4.80,
)


# ------------------------------------------------------------
# Plotting-configuration validation
# ------------------------------------------------------------

def validate_plotting_configuration():
    """
    Verifies that all scenarios and pathways have labels and styles.
    """
    missing_scenario_labels = [
        scenario
        for scenario in SCENARIO_ORDER
        if scenario not in SCENARIO_LABELS
    ]

    missing_scenario_colors = [
        scenario
        for scenario in SCENARIO_ORDER
        if scenario not in SCENARIO_COLORS
    ]

    missing_scenario_linestyles = [
        scenario
        for scenario in SCENARIO_ORDER
        if scenario not in SCENARIO_LINESTYLES
    ]

    missing_scenario_markers = [
        scenario
        for scenario in SCENARIO_ORDER
        if scenario not in SCENARIO_MARKERS
    ]

    missing_pathway_labels = [
        pathway
        for pathway in PATHWAY_ORDER
        if pathway not in PATHWAY_LABELS
    ]

    missing_pathway_colors = [
        pathway
        for pathway in PATHWAY_ORDER
        if pathway not in PATHWAY_COLORS
    ]

    missing_pathway_linestyles = [
        pathway
        for pathway in PATHWAY_ORDER
        if pathway not in PATHWAY_LINESTYLES
    ]

    missing_pathway_markers = [
        pathway
        for pathway in PATHWAY_ORDER
        if pathway not in PATHWAY_MARKERS
    ]

    validation_errors = {
        "missing_scenario_labels":
            missing_scenario_labels,

        "missing_scenario_colors":
            missing_scenario_colors,

        "missing_scenario_linestyles":
            missing_scenario_linestyles,

        "missing_scenario_markers":
            missing_scenario_markers,

        "missing_pathway_labels":
            missing_pathway_labels,

        "missing_pathway_colors":
            missing_pathway_colors,

        "missing_pathway_linestyles":
            missing_pathway_linestyles,

        "missing_pathway_markers":
            missing_pathway_markers,
    }

    validation_errors = {
        key: value
        for key, value in validation_errors.items()
        if value
    }

    if validation_errors:
        raise ValueError(
            "Incomplete plotting configuration: "
            f"{validation_errors}"
        )

    return True


# ------------------------------------------------------------
# Axis-formatting utilities
# ------------------------------------------------------------

def clean_axis(
    ax,
    horizontal_grid=True,
    vertical_grid=False,
):
    """
    Applies a clean journal-style format to a Matplotlib axis.
    """
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        direction="out",
        length=3.5,
        width=0.8,
    )

    if horizontal_grid:
        ax.grid(
            visible=True,
            axis="y",
            linestyle="--",
            linewidth=0.5,
            alpha=0.35,
        )
    else:
        ax.grid(visible=False, axis="y")

    if vertical_grid:
        ax.grid(
            visible=True,
            axis="x",
            linestyle="--",
            linewidth=0.5,
            alpha=0.25,
        )
    else:
        ax.grid(visible=False, axis="x")

    return ax


def add_panel_label(
    ax,
    label,
    x=-0.12,
    y=1.04,
):
    """
    Adds a panel label such as '(a)' or '(b)'.
    """
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        fontsize=10,
        fontweight="bold",
        verticalalignment="bottom",
        horizontalalignment="left",
    )

    return ax


def add_reference_line(
    ax,
    value,
    axis="y",
    label=None,
    linestyle=":",
    linewidth=1.0,
    alpha=0.75,
):
    """
    Adds a horizontal or vertical analytical reference line.
    """
    if axis == "y":
        return ax.axhline(
            y=value,
            linestyle=linestyle,
            linewidth=linewidth,
            alpha=alpha,
            label=label,
        )

    if axis == "x":
        return ax.axvline(
            x=value,
            linestyle=linestyle,
            linewidth=linewidth,
            alpha=alpha,
            label=label,
        )

    raise ValueError(
        "axis must be either 'x' or 'y'."
    )


# ------------------------------------------------------------
# Filename and figure-export utilities
# ------------------------------------------------------------

def sanitize_filename(filename):
    """
    Converts a proposed filename into a filesystem-safe stem.
    """
    filename = str(filename).strip()

    filename = re.sub(
        r"\s+",
        "_",
        filename,
    )

    filename = re.sub(
        r"[^A-Za-z0-9_.-]",
        "",
        filename,
    )

    filename = filename.strip("._")

    if not filename:
        raise ValueError(
            "filename must contain at least one valid character."
        )

    return filename


def save_figure(
    fig,
    filename,
    output_dir=FIG_DIR,
    formats=("pdf", "svg", "png"),
    dpi=600,
    close=False,
):
    """
    Saves a figure in vector and raster formats.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Figure to save.

    filename : str
        File stem without extension.

    output_dir : str or pathlib.Path
        Destination directory.

    formats : tuple of str
        Output formats.

    dpi : int
        Resolution for raster formats.

    close : bool
        Whether to close the figure after saving.

    Returns
    -------
    dict
        Mapping from file format to saved path.
    """
    if fig is None:
        raise ValueError(
            "fig cannot be None."
        )

    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    safe_filename = sanitize_filename(
        filename
    )

    allowed_formats = {
        "pdf",
        "svg",
        "png",
        "eps",
        "tiff",
        "tif",
    }

    saved_files = {}

    for file_format in formats:
        file_format = (
            str(file_format)
            .lower()
            .lstrip(".")
        )

        if file_format not in allowed_formats:
            raise ValueError(
                "Unsupported figure format: "
                f"{file_format}"
            )

        output_path = (
            output_dir
            / f"{safe_filename}.{file_format}"
        )

        save_kwargs = {
            "bbox_inches": "tight",
            "pad_inches": 0.04,
        }

        if file_format in {
            "png",
            "tiff",
            "tif",
        }:
            save_kwargs["dpi"] = dpi

        temporary_handle = tempfile.NamedTemporaryFile(
            dir=output_dir,
            prefix=f".{safe_filename}.",
            suffix=f".{file_format}",
            delete=False,
        )
        temporary_path = Path(temporary_handle.name)
        temporary_handle.close()

        try:
            fig.savefig(
                temporary_path,
                format=file_format,
                **save_kwargs,
            )

            with temporary_path.open("rb") as completed_file:
                os.fsync(completed_file.fileno())

            os.replace(
                temporary_path,
                output_path,
            )

        finally:
            temporary_path.unlink(
                missing_ok=True,
            )

        saved_files[file_format] = (
            output_path
        )

    if close:
        plt.close(fig)

    return saved_files


# ------------------------------------------------------------
# DataFrame validation
# ------------------------------------------------------------

def validate_plotting_columns(
    dataframe,
    required_columns,
):
    """
    Checks whether a DataFrame contains the columns required by a plot.
    """
    if dataframe is None:
        raise ValueError(
            "dataframe cannot be None."
        )

    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise KeyError(
            "Missing columns required for plotting: "
            + ", ".join(missing_columns)
        )

    return True


# ------------------------------------------------------------
# Validate configuration
# ------------------------------------------------------------

validate_plotting_configuration()

In [ ]:
def plot_pathway_shares_journal(
    results,
    y_upper=1.0,
    show_uncertainty=False,
    filename="figure_2_pathway_allocation",
):
    """
    Plot annual treated-flow shares by EoL pathway for each scenario.

    Pathway shares are calculated relative to total treated WTB flow in
    each simulation row. For Monte Carlo results, the median pathway
    share is displayed for each scenario-year combination.

    Notes
    -----
    Uncertainty bands are not shown because uncertainty intervals for
    individual components of a stacked composition are not directly
    interpretable and may violate the unit-sum constraint.

    Parameters
    ----------
    results : pandas.DataFrame
        Deterministic or Monte Carlo simulation results.

    y_upper : float
        Upper y-axis limit. For compositional pathway shares this should
        normally remain equal to 1.0.

    show_uncertainty : bool
        Retained for API compatibility. When True, a warning explains
        why uncertainty bands are not added to the stacked-area plot.

    filename : str
        Output filename without extension.

    Returns
    -------
    matplotlib.figure.Figure
        Generated figure.
    """
    import warnings

    if show_uncertainty:
        warnings.warn(
            "Uncertainty bands are not displayed for stacked pathway "
            "shares because separate quantile bands do not preserve the "
            "unit-sum compositional constraint. Median compositions are "
            "shown instead.",
            UserWarning,
            stacklevel=2,
        )

    if y_upper <= 0.0:
        raise ValueError(
            "y_upper must be strictly positive."
        )

    required_columns = [
        "scenario",
        "year",
        "total_treated",
        *PATHWAY_FLOW_COLUMNS.values(),
    ]

    validate_plotting_columns(
        dataframe=results,
        required_columns=required_columns,
    )

    if results.empty:
        raise ValueError(
            "results cannot be empty."
        )

    plotting_data = results.copy()

    # --------------------------------------------------------
    # Calculate pathway shares safely
    # --------------------------------------------------------

    treated_denominator = (
        plotting_data["total_treated"]
        .astype(float)
        .to_numpy()
    )

    valid_treatment = (
        np.isfinite(treated_denominator)
        & (treated_denominator > NUMERICAL_EPSILON)
    )

    for pathway in PATHWAY_ORDER:
        flow_column = PATHWAY_FLOW_COLUMNS[pathway]
        share_column = f"share_{pathway}"

        pathway_flow = (
            plotting_data[flow_column]
            .astype(float)
            .to_numpy()
        )

        pathway_share = np.divide(
            pathway_flow,
            treated_denominator,
            out=np.zeros_like(
                pathway_flow,
                dtype=float,
            ),
            where=valid_treatment,
        )

        plotting_data[share_column] = np.clip(
            np.nan_to_num(
                pathway_share,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            ),
            0.0,
            1.0,
        )

    # --------------------------------------------------------
    # Summarize each pathway by scenario and year
    # --------------------------------------------------------

    pathway_summaries = []

    for pathway in PATHWAY_ORDER:
        indicator = f"share_{pathway}"

        summary = summarize_indicator_by_scenario_year(
            results=plotting_data,
            indicator=indicator,
            quantiles=(0.05, 0.50, 0.95),
        )

        summary = summary[
            ["scenario", "year", "median"]
        ].rename(
            columns={
                "median": f"{indicator}_median",
            }
        )

        pathway_summaries.append(summary)

    merged_summary = pathway_summaries[0]

    for summary in pathway_summaries[1:]:
        merged_summary = pd.merge(
            merged_summary,
            summary,
            on=["scenario", "year"],
            how="outer",
            validate="one_to_one",
        )

    merged_summary = (
        merged_summary
        .sort_values(["scenario", "year"])
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Scenario ordering and figure layout
    # --------------------------------------------------------

    available_scenarios = [
        scenario
        for scenario in SCENARIO_ORDER
        if scenario in set(
            merged_summary["scenario"]
        )
    ]

    additional_scenarios = [
        scenario
        for scenario in merged_summary[
            "scenario"
        ].dropna().unique()
        if scenario not in available_scenarios
    ]

    available_scenarios.extend(
        additional_scenarios
    )

    if not available_scenarios:
        raise ValueError(
            "No scenarios are available for plotting."
        )

    n_scenarios = len(available_scenarios)

    figure_width = min(
        FIGSIZE_DOUBLE_COLUMN[0],
        max(
            FIGSIZE_SINGLE_COLUMN[0],
            2.35 * n_scenarios,
        ),
    )

    fig, axes = plt.subplots(
        nrows=1,
        ncols=n_scenarios,
        figsize=(
            figure_width,
            FIGSIZE_ONE_AND_HALF_COLUMN[1] * 1.20,
        ),
        sharex=True,
        sharey=True,
        squeeze=False,
    )

    axes = axes.ravel()

    start_year = int(
        plotting_data["year"].min()
    )
    end_year = int(
        plotting_data["year"].max()
    )

    # --------------------------------------------------------
    # Plot median pathway compositions
    # --------------------------------------------------------

    for panel_index, scenario in enumerate(
        available_scenarios
    ):
        ax = axes[panel_index]

        scenario_data = (
            merged_summary.loc[
                merged_summary["scenario"]
                == scenario
            ]
            .sort_values("year")
            .copy()
        )

        if scenario_data.empty:
            ax.set_visible(False)
            continue

        years = (
            scenario_data["year"]
            .astype(int)
            .to_numpy()
        )

        raw_shares = np.vstack([
            scenario_data[
                f"share_{pathway}_median"
            ]
            .fillna(0.0)
            .astype(float)
            .to_numpy()
            for pathway in PATHWAY_ORDER
        ])

        raw_shares = np.clip(
            np.nan_to_num(
                raw_shares,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            ),
            0.0,
            None,
        )

        composition_totals = raw_shares.sum(
            axis=0
        )

        normalized_shares = np.divide(
            raw_shares,
            composition_totals,
            out=np.zeros_like(
                raw_shares,
                dtype=float,
            ),
            where=(
                composition_totals
                > NUMERICAL_EPSILON
            ),
        )

        ax.stackplot(
            years,
            normalized_shares,
            colors=[
                PATHWAY_COLORS[pathway]
                for pathway in PATHWAY_ORDER
            ],
            labels=[
                PATHWAY_LABELS[pathway]
                for pathway in PATHWAY_ORDER
            ],
            alpha=0.85,
            linewidth=0.35,
        )

        scenario_label = SCENARIO_LABELS.get(
            scenario,
            scenario.replace("_", " ").title(),
        )

        ax.set_title(
            scenario_label,
            fontsize=10,
        )

        ax.set_xlabel("Year")

        ax.set_ylim(
            0.0,
            y_upper,
        )

        tick_step = (
            0.20
            if y_upper <= 1.0
            else y_upper / 5.0
        )

        ax.set_yticks(
            np.arange(
                0.0,
                y_upper + 0.5 * tick_step,
                tick_step,
            )
        )

        set_year_axis(
            ax,
            start_year=start_year,
            end_year=end_year,
        )

        clean_axis(
            ax,
            horizontal_grid=False,
            vertical_grid=False,
        )

        add_panel_label(
            ax,
            label=f"({chr(97 + panel_index)})",
        )

    axes[0].set_ylabel(
        "Share of total treated WTB flow"
    )

    # --------------------------------------------------------
    # Shared legend and export
    # --------------------------------------------------------

    handles, labels = (
        axes[0].get_legend_handles_labels()
    )

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=min(3, len(PATHWAY_ORDER)),
        frameon=False,
        bbox_to_anchor=(0.5, 0.01),
        handlelength=2.0,
        columnspacing=1.2,
    )

    fig.tight_layout(
        rect=(0.0, 0.14, 1.0, 1.0)
    )

    save_figure(
        fig,
        filename=filename,
        formats=("pdf", "svg", "png"),
        dpi=600,
        close=False,
    )

    return fig

In [ ]:
# ============================================================
# 12. Shared plotting utilities and Figure 2 generation
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Year-axis utility
# ------------------------------------------------------------

def set_year_axis(
    ax,
    start_year=START_YEAR,
    end_year=END_YEAR,
    preferred_ticks=None,
):
    """
    Apply a compact and consistent simulation-year axis.
    """
    start_year = int(start_year)
    end_year = int(end_year)

    if end_year < start_year:
        raise ValueError(
            "end_year must be equal to or greater than start_year."
        )

    if preferred_ticks is None:
        preferred_ticks = [
            2026,
            2030,
            2035,
            2040,
            2045,
            2050,
        ]

    ticks = [
        int(year)
        for year in preferred_ticks
        if start_year <= int(year) <= end_year
    ]

    if start_year not in ticks:
        ticks.insert(0, start_year)

    if end_year not in ticks:
        ticks.append(end_year)

    ticks = sorted(set(ticks))

    ax.set_xlim(
        start_year,
        end_year,
    )

    ax.set_xticks(ticks)

    return ax


# ------------------------------------------------------------
# Scenario-order utility
# ------------------------------------------------------------

def get_available_scenarios(results):
    """
    Return scenarios present in the results in manuscript order.
    """
    validate_plotting_columns(
        dataframe=results,
        required_columns=[
            "scenario",
            "year",
        ],
    )

    if results.empty:
        raise ValueError(
            "results cannot be empty."
        )

    observed_scenarios = set(
        results["scenario"]
        .dropna()
        .unique()
    )

    available_scenarios = [
        scenario
        for scenario in SCENARIO_ORDER
        if scenario in observed_scenarios
    ]

    if not available_scenarios:
        raise ValueError(
            "The results DataFrame contains no recognized scenarios."
        )

    return available_scenarios


# ------------------------------------------------------------
# Monte Carlo run-identifier utility
# ------------------------------------------------------------

def get_run_identifier_column(results):
    """
    Detect the column identifying Monte Carlo simulation runs.

    Returns
    -------
    str or None
        Run-identifier column, or None for deterministic results.
    """
    candidates = [
        "run_id",
        "run",
        "simulation_run",
        "mc_run",
        "iteration",
    ]

    for column in candidates:
        if column in results.columns:
            return column

    return None


# ------------------------------------------------------------
# Scenario-year summary utility
# ------------------------------------------------------------

def summarize_indicator_by_scenario_year(
    results,
    indicator,
    quantiles=(0.05, 0.50, 0.95),
):
    """
    Summarize an indicator by scenario and year.

    For Monte Carlo results, lower, median, and upper quantiles
    are returned. For deterministic results, the observed value
    is returned as the median and uncertainty bounds are missing.
    """
    validate_plotting_columns(
        dataframe=results,
        required_columns=[
            "scenario",
            "year",
            indicator,
        ],
    )

    if results.empty:
        raise ValueError(
            "results cannot be empty."
        )

    if len(quantiles) != 3:
        raise ValueError(
            "quantiles must contain exactly three values: "
            "lower, median, and upper."
        )

    lower_q, median_q, upper_q = quantiles

    if not (
        0.0
        <= lower_q
        <= median_q
        <= upper_q
        <= 1.0
    ):
        raise ValueError(
            "Quantiles must satisfy "
            "0 <= lower <= median <= upper <= 1."
        )

    data = results[
        [
            "scenario",
            "year",
            indicator,
        ]
    ].copy()

    data[indicator] = pd.to_numeric(
        data[indicator],
        errors="coerce",
    )

    run_column = get_run_identifier_column(
        results
    )

    # --------------------------------------------------------
    # Deterministic simulation
    # --------------------------------------------------------

    if run_column is None:
        duplicate_rows = data.duplicated(
            subset=["scenario", "year"],
            keep=False,
        )

        if duplicate_rows.any():
            raise ValueError(
                "Multiple observations exist for at least one "
                "scenario-year pair, but no Monte Carlo run identifier "
                "column was found."
            )

        data = data.rename(
            columns={
                indicator: "median",
            }
        )

        data["lower"] = np.nan
        data["upper"] = np.nan

        return (
            data[
                [
                    "scenario",
                    "year",
                    "lower",
                    "median",
                    "upper",
                ]
            ]
            .sort_values(["scenario", "year"])
            .reset_index(drop=True)
        )

    # --------------------------------------------------------
    # Monte Carlo simulation
    # --------------------------------------------------------

    summary = (
        data
        .groupby(
            [
                "scenario",
                "year",
            ],
            as_index=False,
            observed=True,
        )
        .agg(
            lower=(
                indicator,
                lambda values: values.quantile(
                    lower_q
                ),
            ),
            median=(
                indicator,
                lambda values: values.quantile(
                    median_q
                ),
            ),
            upper=(
                indicator,
                lambda values: values.quantile(
                    upper_q
                ),
            ),
        )
        .sort_values(["scenario", "year"])
        .reset_index(drop=True)
    )

    return summary


# ------------------------------------------------------------
# Scenario trajectory utility
# ------------------------------------------------------------

def plot_scenario_indicator(
    ax,
    summary,
    scenario,
    show_uncertainty=True,
    uncertainty_alpha=0.16,
    label=None,
):
    """
    Plot one scenario trajectory and its optional uncertainty band.
    """
    validate_plotting_columns(
        dataframe=summary,
        required_columns=[
            "scenario",
            "year",
            "lower",
            "median",
            "upper",
        ],
    )

    if scenario not in SCENARIO_ORDER:
        raise ValueError(
            f"Unknown scenario: {scenario}"
        )

    if not (
        0.0
        <= uncertainty_alpha
        <= 1.0
    ):
        raise ValueError(
            "uncertainty_alpha must be between 0 and 1."
        )

    scenario_data = (
        summary.loc[
            summary["scenario"] == scenario
        ]
        .sort_values("year")
        .copy()
    )

    if scenario_data.empty:
        return ax

    years = (
        scenario_data["year"]
        .astype(float)
        .to_numpy()
    )

    median_values = (
        scenario_data["median"]
        .astype(float)
        .to_numpy()
    )

    ax.plot(
        years,
        median_values,
        color=SCENARIO_COLORS[scenario],
        linestyle=SCENARIO_LINESTYLES[scenario],
        marker=SCENARIO_MARKERS[scenario],
        markevery=max(
            1,
            len(scenario_data) // 5,
        ),
        linewidth=1.8,
        label=(
            label
            if label is not None
            else SCENARIO_LABELS.get(
                scenario,
                scenario.replace("_", " ").title(),
            )
        ),
    )

    uncertainty_available = (
        scenario_data["lower"].notna().all()
        and scenario_data["upper"].notna().all()
    )

    if (
        show_uncertainty
        and uncertainty_available
    ):
        lower_values = (
            scenario_data["lower"]
            .astype(float)
            .to_numpy()
        )

        upper_values = (
            scenario_data["upper"]
            .astype(float)
            .to_numpy()
        )

        ax.fill_between(
            years,
            lower_values,
            upper_values,
            color=SCENARIO_COLORS[scenario],
            alpha=uncertainty_alpha,
            linewidth=0.0,
        )

    return ax


# ============================================================
# Figure 2 generation is deferred until the Monte Carlo sample
# ============================================================
# The deterministic results remain available for model checks, but
# the publication figure is generated from the median composition
# of all paired Monte Carlo runs in the final-figure cell.


In [ ]:
# ============================================================
# Celda 13. Shared plotting utilities - consistency check
# ============================================================
# Utility functions are defined in cells 11 and 12.
# This cell verifies that all required plotting objects
# are available before generating the remaining figures.

_required_plotting_objects = [
    # Configuration objects
    "FIG_DIR",
    "SCENARIO_LABELS",
    "SCENARIO_ORDER",
    "SCENARIO_COLORS",
    "SCENARIO_LINESTYLES",
    "SCENARIO_MARKERS",
    "PATHWAY_LABELS",
    "PATHWAY_ORDER",
    "PATHWAY_COLORS",
    "PATHWAY_FLOW_COLUMNS",

    # General plotting utilities
    "clean_axis",
    "add_panel_label",
    "add_reference_line",
    "save_figure",
    "validate_plotting_columns",

    # Shared utilities from Cell 12
    "set_year_axis",
    "get_available_scenarios",
    "get_run_identifier_column",
    "summarize_indicator_by_scenario_year",
    "plot_scenario_indicator",

    # Figure-specific function
    "plot_pathway_shares_journal",
]

_missing_plotting_objects = [
    object_name
    for object_name in _required_plotting_objects
    if object_name not in globals()
]

if _missing_plotting_objects:
    raise RuntimeError(
        "Run the journal-style plotting configuration "
        "and shared plotting utility cells first. "
        "Missing objects: "
        + ", ".join(_missing_plotting_objects)
    )

print(
    "Shared plotting utilities loaded successfully."
)

In [ ]:
# ============================================================
# Figure 3. Solvolysis diffusion across scenarios
# ============================================================

def plot_solvolysis_diffusion_journal(
    results,
    y_upper=None,
    show_uncertainty=True,
    filename="figure_3_solvolysis_diffusion",
):
    """
    Plots the diffusion of solvolysis across transition scenarios.

    Solvolysis diffusion is measured as the share of total treated
    WTB flow processed through solvolysis.

    This indicator is distinct from the closed-loop WTB flow share,
    because not all solvolysis-derived material is necessarily used
    in closed-loop applications.

    Parameters
    ----------
    results : pandas.DataFrame
        Deterministic or Monte Carlo simulation results.

    y_upper : float or None
        Upper limit of the y-axis. If None, it is determined
        automatically.

    show_uncertainty : bool
        Displays the 5th-95th percentile interval when Monte Carlo
        results are available.

    filename : str
        Output filename without extension.

    Returns
    -------
    matplotlib.figure.Figure
        Generated figure.
    """
    indicator = (
        "solvolysis_treatment_share"
    )

    validate_plotting_columns(
        dataframe=results,
        required_columns=[
            "scenario",
            "year",
            indicator,
        ],
    )

    summary = (
        summarize_indicator_by_scenario_year(
            results=results,
            indicator=indicator,
            quantiles=(
                0.05,
                0.50,
                0.95,
            ),
        )
    )

    available_scenarios = (
        get_available_scenarios(
            results
        )
    )

    fig, ax = plt.subplots(
        figsize=FIGSIZE_ONE_AND_HALF_COLUMN
    )

    for scenario in available_scenarios:
        plot_scenario_indicator(
            ax=ax,
            summary=summary,
            scenario=scenario,
            show_uncertainty=(
                show_uncertainty
            ),
            uncertainty_alpha=0.14,
        )

    # --------------------------------------------------------
    # Automatic y-axis limit
    # --------------------------------------------------------

    observed_upper = float(
        summary["median"].max()
    )

    if (
        show_uncertainty
        and summary["upper"].notna().any()
    ):
        observed_upper = max(
            observed_upper,
            float(
                summary["upper"].max()
            ),
        )

    if y_upper is None:
        y_upper = min(
            1.0,
            max(
                0.10,
                np.ceil(
                    observed_upper
                    * 20.0
                )
                / 20.0
                + 0.05,
            ),
        )

    if not (
        0.0
        < y_upper
        <= 1.0
    ):
        raise ValueError(
            "y_upper must lie between 0 and 1."
        )

    # --------------------------------------------------------
    # Axis formatting
    # --------------------------------------------------------

    ax.set_xlabel(
        "Year"
    )

    ax.set_ylabel(
        "Solvolysis share of treated WTB flow"
    )

    ax.set_ylim(
        0.0,
        y_upper,
    )

    # Use a finer step when y_upper is small so at least 2 ticks appear.
    ytick_step = 0.05 if y_upper <= 0.20 else 0.10 if y_upper <= 0.40 else 0.20
    ax.set_yticks(
        np.arange(
            0.0,
            y_upper + 1e-9,
            ytick_step,
        )
    )

    set_year_axis(
        ax=ax,
        start_year=int(
            results["year"].min()
        ),
        end_year=int(
            results["year"].max()
        ),
    )

    clean_axis(
        ax=ax,
        horizontal_grid=True,
        vertical_grid=False,
    )

    ax.legend(
        loc="upper left",
        frameon=False,
        handlelength=2.8,
    )

    fig.tight_layout()

    save_figure(
        fig=fig,
        filename=filename,
        formats=(
            "pdf",
            "svg",
            "png",
        ),
        dpi=600,
        close=False,
    )

    return fig


# ============================================================
# Generate deterministic figure
# ============================================================

# y_upper=1.0 for deterministic first pass; use y_upper=None for MC auto-scale.
fig_solvolysis_diffusion = (
    plot_solvolysis_diffusion_journal(
        results=results,
        y_upper=None,
        show_uncertainty=(
            get_run_identifier_column(results)
            is not None
        ),
        filename="figure_3_solvolysis_diffusion",
    )
)

plt.show()

In [ ]:
# ============================================================
# Figure 4. Open-loop versus closed-loop material utilization
# ============================================================

def prepare_open_closed_utilization_summary(
    results,
):
    """
    Prepares annual open-loop and closed-loop utilization shares.

    For Monte Carlo results, annual medians and 5th-95th percentile
    intervals are calculated by scenario.

    Open-loop and closed-loop utilization shares are calculated
    relative to the total amount of recovered material actually used.
    """
    required_columns = [
        "scenario",
        "year",
        "open_loop_utilization_share",
        "closed_loop_utilization_share",
    ]

    validate_plotting_columns(
        dataframe=results,
        required_columns=required_columns,
    )

    open_summary = (
        summarize_indicator_by_scenario_year(
            results=results,
            indicator=(
                "open_loop_utilization_share"
            ),
            quantiles=(
                0.05,
                0.50,
                0.95,
            ),
        )
        .rename(
            columns={
                "lower": "open_lower",
                "median": "open_median",
                "upper": "open_upper",
            }
        )
    )

    closed_summary = (
        summarize_indicator_by_scenario_year(
            results=results,
            indicator=(
                "closed_loop_utilization_share"
            ),
            quantiles=(
                0.05,
                0.50,
                0.95,
            ),
        )
        .rename(
            columns={
                "lower": "closed_lower",
                "median": "closed_median",
                "upper": "closed_upper",
            }
        )
    )

    summary = open_summary.merge(
        closed_summary,
        on=[
            "scenario",
            "year",
        ],
        how="outer",
        validate="one_to_one",
    )

    return summary


def plot_open_closed_loop_journal(
    results,
    show_uncertainty=False,
    filename=(
        "figure_4_open_closed_loop_utilization"
    ),
):
    """
    Plots open-loop and closed-loop utilization shares
    across transition scenarios.

    Parameters
    ----------
    results : pandas.DataFrame
        Deterministic or Monte Carlo simulation results.

    show_uncertainty : bool
        Displays 5th-95th percentile intervals when Monte Carlo
        results are available.

    filename : str
        Output filename without extension.

    Returns
    -------
    matplotlib.figure.Figure
        Generated figure.
    """
    summary = (
        prepare_open_closed_utilization_summary(
            results=results
        )
    )

    available_scenarios = (
        get_available_scenarios(
            results
        )
    )

    number_of_scenarios = len(
        available_scenarios
    )

    # Cap total width at the double-column journal limit (7.2 inches).
    panel_width = min(
        FIGSIZE_DOUBLE_COLUMN[0] / max(number_of_scenarios, 1),
        4.25,
    )
    fig, axes = plt.subplots(
        nrows=1,
        ncols=number_of_scenarios,
        figsize=(
            panel_width * number_of_scenarios,
            4.10,
        ),
        sharex=True,
        sharey=True,
    )

    axes = np.atleast_1d(
        axes
    )

    utilization_colors = {
        "open_loop": "#4477AA",
        "closed_loop": "#EE6677",
    }

    utilization_linestyles = {
        "open_loop": "--",
        "closed_loop": "-",
    }

    for panel_index, (
        ax,
        scenario,
    ) in enumerate(
        zip(
            axes,
            available_scenarios,
        )
    ):
        scenario_data = (
            summary.loc[
                summary["scenario"]
                == scenario
            ]
            .sort_values("year")
        )

        ax.plot(
            scenario_data["year"],
            scenario_data[
                "open_median"
            ],
            label=(
                "Open-loop utilization"
            ),
            color=(
                utilization_colors[
                    "open_loop"
                ]
            ),
            linestyle=(
                utilization_linestyles[
                    "open_loop"
                ]
            ),
            linewidth=1.9,
        )

        ax.plot(
            scenario_data["year"],
            scenario_data[
                "closed_median"
            ],
            label=(
                "Closed-loop utilization"
            ),
            color=(
                utilization_colors[
                    "closed_loop"
                ]
            ),
            linestyle=(
                utilization_linestyles[
                    "closed_loop"
                ]
            ),
            linewidth=1.9,
        )

        if (
            show_uncertainty
            and scenario_data[
                "open_lower"
            ].notna().any()
            and scenario_data[
                "open_upper"
            ].notna().any()
        ):
            ax.fill_between(
                scenario_data["year"],
                scenario_data[
                    "open_lower"
                ],
                scenario_data[
                    "open_upper"
                ],
                color=(
                    utilization_colors[
                        "open_loop"
                    ]
                ),
                alpha=0.10,
                linewidth=0.0,
            )

        if (
            show_uncertainty
            and scenario_data[
                "closed_lower"
            ].notna().any()
            and scenario_data[
                "closed_upper"
            ].notna().any()
        ):
            ax.fill_between(
                scenario_data["year"],
                scenario_data[
                    "closed_lower"
                ],
                scenario_data[
                    "closed_upper"
                ],
                color=(
                    utilization_colors[
                        "closed_loop"
                    ]
                ),
                alpha=0.10,
                linewidth=0.0,
            )

        ax.set_title(
            SCENARIO_LABELS[
                scenario
            ]
        )

        ax.set_xlabel(
            "Year"
        )

        ax.set_ylim(
            0.0,
            1.0,
        )

        ax.set_yticks(
            np.arange(
                0.0,
                1.01,
                0.20,
            )
        )

        set_year_axis(
            ax=ax,
            start_year=int(
                results[
                    "year"
                ].min()
            ),
            end_year=int(
                results[
                    "year"
                ].max()
            ),
        )

        clean_axis(
            ax=ax,
            horizontal_grid=True,
            vertical_grid=False,
        )

        add_panel_label(
            ax=ax,
            label=(
                f"({chr(97 + panel_index)})"
            ),
            x=-0.10,
            y=1.02,
        )

    axes[0].set_ylabel(
        "Share of utilized recovered material"
    )

    handles, labels = (
        axes[0]
        .get_legend_handles_labels()
    )

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(
            0.5,
            -0.01,
        ),
        handlelength=2.8,
        columnspacing=2.0,
    )

    fig.subplots_adjust(
        left=0.07,
        right=0.99,
        top=0.88,
        bottom=0.22,
        wspace=0.12,
    )

    save_figure(
        fig=fig,
        filename=filename,
        formats=(
            "pdf",
            "svg",
            "png",
        ),
        dpi=600,
        close=False,
    )

    return fig


# ============================================================
# Generate deterministic Figure 4
# ============================================================

fig_open_closed_utilization = (
    plot_open_closed_loop_journal(
        results=results,
        show_uncertainty=(
            get_run_identifier_column(results)
            is not None
        ),
        filename="figure_4_open_closed_loop_utilization",
    )
)

plt.show()

In [ ]:
# ============================================================
# Figure 5. Annual net untreated WTB stock increase
# ============================================================

def plot_annual_untreated_share_journal(
    results,
    y_upper=None,
    show_uncertainty=True,
    filename="figure_5_annual_untreated_share",
):
    """
    Plot the annual net increase in untreated WTB stock relative
    to annual decommissioned WTB inflow.

    The indicator is defined as:

        max(closing untreated stock - opening untreated stock, 0)
        --------------------------------------------------------
                 annual decommissioned WTB inflow

    A value of zero means that untreated stock did not increase
    during the year. A value of one means that the net increase
    in untreated stock was equal to the full annual inflow.

    Parameters
    ----------
    results : pandas.DataFrame
        Deterministic or Monte Carlo simulation results.

    y_upper : float or None
        Upper y-axis limit. When None, the limit is determined from
        the observed median and uncertainty interval.

    show_uncertainty : bool
        Display the 5th-95th percentile interval when Monte Carlo
        results are available.

    filename : str
        Output filename without extension.

    Returns
    -------
    matplotlib.figure.Figure
        Generated figure.
    """
    indicator = "net_untreated_stock_increase_share"

    validate_plotting_columns(
        dataframe=results,
        required_columns=[
            "scenario",
            "year",
            indicator,
        ],
    )

    if results.empty:
        raise ValueError(
            "results cannot be empty."
        )

    plotting_results = results.copy()

    plotting_results[indicator] = pd.to_numeric(
        plotting_results[indicator],
        errors="coerce",
    )

    invalid_values = (
        plotting_results[indicator].notna()
        & (
            (
                plotting_results[indicator]
                < -NUMERICAL_EPSILON
            )
            | (
                plotting_results[indicator]
                > 1.0 + NUMERICAL_EPSILON
            )
        )
    )

    if invalid_values.any():
        invalid_min = float(
            plotting_results.loc[
                invalid_values,
                indicator,
            ].min()
        )

        invalid_max = float(
            plotting_results.loc[
                invalid_values,
                indicator,
            ].max()
        )

        raise ValueError(
            f"{indicator} must lie between 0 and 1. "
            f"Observed invalid range: "
            f"{invalid_min:.6f} to {invalid_max:.6f}."
        )

    # Correct only negligible floating-point deviations.
    plotting_results[indicator] = (
        plotting_results[indicator]
        .clip(
            lower=0.0,
            upper=1.0,
        )
    )

    # --------------------------------------------------------
    # Summarize deterministic or Monte Carlo trajectories
    # --------------------------------------------------------

    summary = summarize_indicator_by_scenario_year(
        results=plotting_results,
        indicator=indicator,
        quantiles=(
            0.05,
            0.50,
            0.95,
        ),
    )

    available_scenarios = get_available_scenarios(
        plotting_results
    )

    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE_ONE_AND_HALF_COLUMN
    )

    for scenario in available_scenarios:
        plot_scenario_indicator(
            ax=ax,
            summary=summary,
            scenario=scenario,
            show_uncertainty=show_uncertainty,
            uncertainty_alpha=0.14,
        )

    # --------------------------------------------------------
    # Automatic y-axis limit
    # --------------------------------------------------------

    finite_medians = (
        summary["median"]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
    )

    observed_upper = (
        float(finite_medians.max())
        if not finite_medians.empty
        else 0.0
    )

    if (
        show_uncertainty
        and summary["upper"].notna().any()
    ):
        finite_upper = (
            summary["upper"]
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna()
        )

        if not finite_upper.empty:
            observed_upper = max(
                observed_upper,
                float(finite_upper.max()),
            )

    if y_upper is None:
        padded_upper = (
            observed_upper * 1.10
        )

        y_upper = min(
            1.0,
            max(
                0.10,
                np.ceil(
                    padded_upper / 0.05
                ) * 0.05,
            ),
        )

    y_upper = float(y_upper)

    if not (
        0.0
        < y_upper
        <= 1.0
    ):
        raise ValueError(
            "y_upper must lie between 0 and 1."
        )

    # --------------------------------------------------------
    # Axis formatting
    # --------------------------------------------------------

    ax.set_xlabel("Year")

    ax.set_ylabel(
        "Share of annual WTB inflow added\n"
        "to untreated stock"
    )

    ax.set_ylim(
        0.0,
        y_upper,
    )

    if y_upper <= 0.20:
        ytick_step = 0.05
    elif y_upper <= 0.50:
        ytick_step = 0.10
    else:
        ytick_step = 0.20

    y_ticks = np.arange(
        0.0,
        y_upper + 0.5 * ytick_step,
        ytick_step,
    )

    if not np.isclose(
        y_ticks[-1],
        y_upper,
    ):
        y_ticks = np.append(
            y_ticks[y_ticks < y_upper],
            y_upper,
        )

    ax.set_yticks(
        np.unique(
            np.round(
                y_ticks,
                10,
            )
        )
    )

    set_year_axis(
        ax=ax,
        start_year=int(
            plotting_results["year"].min()
        ),
        end_year=int(
            plotting_results["year"].max()
        ),
    )

    clean_axis(
        ax=ax,
        horizontal_grid=True,
        vertical_grid=False,
    )

    ax.legend(
        loc="upper left",
        frameon=False,
        handlelength=2.8,
    )

    fig.tight_layout()

    # --------------------------------------------------------
    # Save figure
    # --------------------------------------------------------

    save_figure(
        fig=fig,
        filename=filename,
        formats=(
            "pdf",
            "svg",
            "png",
        ),
        dpi=600,
        close=False,
    )

    return fig


# ============================================================
# Legacy annual-flow figure generation is disabled
# ============================================================
# The final output set uses untreated stock share as Figure 5
# and the annual net increase as Supplementary Figure S1.


In [ ]:
# ============================================================
# Figure 6. Economic outcome
# ============================================================

def add_y_margin(
    ax,
    values,
    margin=0.08,
    force_zero=False,
):
    """Set y-axis limits with a proportional margin."""
    if margin < 0.0:
        raise ValueError("margin must be non-negative.")

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return ax

    y_min = float(np.min(values))
    y_max = float(np.max(values))
    span = y_max - y_min

    if span <= NUMERICAL_EPSILON:
        span = max(abs(y_min), abs(y_max), 1.0)

    lower_limit = y_min - margin * span
    upper_limit = y_max + margin * span

    if force_zero and y_min >= -NUMERICAL_EPSILON:
        lower_limit = 0.0

    if upper_limit <= lower_limit:
        upper_limit = lower_limit + 1.0

    ax.set_ylim(lower_limit, upper_limit)
    return ax


def plot_processing_cost_journal(
    results,
    show_uncertainty=True,
    filename="figure_6_processing_cost",
):
    """
    Plot annual average processing cost.

    Monetary inputs are interpreted consistently as 2019 USD per treated
    whole-blade mass unit. The previous environmental proxy is not reported as
    greenhouse-gas performance because it is not a physical LCA inventory.
    """
    required_columns = [
        "scenario",
        "year",
        "average_processing_cost",
    ]

    validate_plotting_columns(
        dataframe=results,
        required_columns=required_columns,
    )

    if results.empty:
        raise ValueError("results cannot be empty.")

    plotting_data = results.copy()
    plotting_data["average_processing_cost"] = pd.to_numeric(
        plotting_data["average_processing_cost"],
        errors="coerce",
    )

    invalid_cost = (
        plotting_data["average_processing_cost"].notna()
        & (
            plotting_data["average_processing_cost"]
            < -NUMERICAL_EPSILON
        )
    )

    if invalid_cost.any():
        raise ValueError("average_processing_cost cannot be negative.")

    plotting_data["average_processing_cost"] = plotting_data[
        "average_processing_cost"
    ].clip(lower=0.0)

    available_scenarios = get_available_scenarios(plotting_data)
    cost_summary = summarize_indicator_by_scenario_year(
        results=plotting_data,
        indicator="average_processing_cost",
        quantiles=(0.05, 0.50, 0.95),
    )

    fig, ax = plt.subplots(
        figsize=FIGSIZE_ONE_AND_HALF_COLUMN,
    )

    for scenario in available_scenarios:
        plot_scenario_indicator(
            ax=ax,
            summary=cost_summary,
            scenario=scenario,
            show_uncertainty=show_uncertainty,
            uncertainty_alpha=0.14,
        )

    start_year = int(plotting_data["year"].min())
    end_year = int(plotting_data["year"].max())

    ax.set_title("Average processing cost")
    ax.set_xlabel("Year")
    ax.set_ylabel("2019 USD per treated WTB mass unit")
    set_year_axis(
        ax=ax,
        start_year=start_year,
        end_year=end_year,
    )

    axis_values = (
        cost_summary["median"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .tolist()
    )

    if (
        show_uncertainty
        and cost_summary["lower"].notna().all()
        and cost_summary["upper"].notna().all()
    ):
        axis_values.extend(
            cost_summary["lower"]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .tolist()
        )
        axis_values.extend(
            cost_summary["upper"]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .tolist()
        )

    add_y_margin(
        ax=ax,
        values=axis_values,
        margin=0.10,
        force_zero=False,
    )
    clean_axis(
        ax=ax,
        horizontal_grid=True,
        vertical_grid=False,
    )

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles,
        labels,
        loc="best",
        frameon=False,
    )
    fig.tight_layout()

    save_figure(
        fig=fig,
        filename=filename,
        formats=("pdf", "svg", "png"),
        dpi=600,
        close=False,
    )
    return fig


fig_processing_cost = plot_processing_cost_journal(
    results=results,
    show_uncertainty=(
        get_run_identifier_column(results) is not None
    ),
    filename="figure_6_processing_cost",
)

plt.show()


In [ ]:
# ============================================================
# Figure 7. Solvolysis technology maturity trajectory
# ============================================================

def plot_solvolysis_maturity_journal(
    results,
    y_lower=5.0,
    y_upper=9.0,
    show_uncertainty=True,
    filename="figure_7_solvolysis_maturity",
):
    """
    Plots the opening-period solvolysis technology maturity
    trajectory across transition scenarios.

    Technology maturity is represented by the stylized
    technology-readiness-level trajectory observed by agents
    at the beginning of each simulation year.

    The horizontal reference line indicates the maturity
    threshold above which solvolysis becomes available for
    commercial investment.

    Parameters
    ----------
    results : pandas.DataFrame
        Deterministic or Monte Carlo simulation results.

    y_lower : float
        Lower y-axis limit.

    y_upper : float
        Upper y-axis limit.

    show_uncertainty : bool
        Displays the 5th-95th percentile interval when Monte Carlo
        uncertainty affects the maturity trajectory.

    filename : str
        Output filename without extension.

    Returns
    -------
    matplotlib.figure.Figure
        Generated figure.
    """
    indicator = (
        "opening_trl_solvolysis"
    )

    validate_plotting_columns(
        dataframe=results,
        required_columns=[
            "scenario",
            "year",
            indicator,
        ],
    )

    if y_upper <= y_lower:
        raise ValueError(
            "y_upper must be greater than y_lower."
        )

    if y_lower < 1.0 or y_upper > 9.0:
        raise ValueError(
            "TRL axis limits must remain within the interval [1, 9]."
        )

    # --------------------------------------------------------
    # Summarize trajectories
    # --------------------------------------------------------

    summary = (
        summarize_indicator_by_scenario_year(
            results=results,
            indicator=indicator,
            quantiles=(
                0.05,
                0.50,
                0.95,
            ),
        )
    )

    available_scenarios = (
        get_available_scenarios(
            results
        )
    )

    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE_ONE_AND_HALF_COLUMN
    )

    for scenario in available_scenarios:
        plot_scenario_indicator(
            ax=ax,
            summary=summary,
            scenario=scenario,
            show_uncertainty=(
                show_uncertainty
            ),
            uncertainty_alpha=0.12,
        )

    # --------------------------------------------------------
    # Commercial maturity threshold
    # --------------------------------------------------------

    add_reference_line(
        ax=ax,
        value=TRL_THRESHOLD_SOLVOLYSIS,
        axis="y",
        label=(
            "Commercial maturity threshold"
        ),
        linestyle=":",
        linewidth=1.1,
        alpha=0.80,
    )

    # --------------------------------------------------------
    # Axis formatting
    # --------------------------------------------------------

    ax.set_xlabel(
        "Year"
    )

    ax.set_ylabel(
        "Solvolysis technology readiness level"
    )

    ax.set_ylim(
        y_lower,
        y_upper,
    )

    ax.set_yticks(
        np.arange(
            np.ceil(y_lower),
            np.floor(y_upper) + 1,
            1,
        )
    )

    set_year_axis(
        ax=ax,
        start_year=int(
            results["year"].min()
        ),
        end_year=int(
            results["year"].max()
        ),
    )

    clean_axis(
        ax=ax,
        horizontal_grid=True,
        vertical_grid=False,
    )

    ax.legend(
        loc="lower right",
        frameon=False,
        handlelength=2.8,
    )

    fig.tight_layout()

    # --------------------------------------------------------
    # Save figure
    # --------------------------------------------------------

    save_figure(
        fig=fig,
        filename=filename,
        formats=(
            "pdf",
            "svg",
            "png",
        ),
        dpi=600,
        close=False,
    )

    return fig


# ============================================================
# Generate deterministic Figure 7
# ============================================================

fig_solvolysis_maturity = (
    plot_solvolysis_maturity_journal(
        results=results,
        y_lower=5.0,
        y_upper=9.0,
        show_uncertainty=(
            get_run_identifier_column(results)
            is not None
        ),
        filename="figure_7_solvolysis_maturity",
    )
)

plt.show()

In [ ]:
# ============================================================
# 10. Monte Carlo uncertainty analysis
#     Parsimonious final specification
# ============================================================

import copy
import multiprocessing as mp
import warnings

import numpy as np
import pandas as pd


# ============================================================
# 10.1 Monte Carlo settings
# ============================================================

# QUICK_MODE=True is for code checks and ordinary editing.
# Set QUICK_MODE=False only for the final 500-run paper analysis.
QUICK_MODE = True
N_RUNS = 500 if QUICK_MODE else 500

RANDOM_SEED = 123

FINAL_ANALYSIS_MINIMUM_RUNS = 500

# Independent processes accelerate the simulation without changing
# parameter draws, run seeds, scenario pairing, or result ordering.
# The implementation falls back to one process if 'fork' is absent.
MONTE_CARLO_N_JOBS = 1 if QUICK_MODE else 4
MONTE_CARLO_START_METHOD = "fork"


# ============================================================
# 10.2 Uncertainty specification
# ============================================================
# The uncertainty set is deliberately parsimonious.
#
# It covers:
#
# 1. processing costs for solvolysis and the three incumbent treatments;
# 2. solvolysis recovered-material quality;
# 3. mandatory closed-loop demand activation;
# 4. closed-loop offtake realization;
# 5. adjustment of recycler sales expectations;
# 6. cost of holding unsold recovered material;
# 7. deterioration of solvolysis-derived inventory;
# 8. open-loop market absorption conditions.
#
# Solvolysis glass-fibre recovery efficiency is fixed at 0.85 and
# is not treated as a Monte Carlo distribution.
#
# Policy configurations themselves remain deterministic scenario
# definitions. Uncertainty is applied equally across all scenarios
# within a Monte Carlo run.


OPEN_LOOP_MARKET_CASES_MC = {
    "restricted": {
        "mechanical_recycling": 0.25,
        "pyrolysis": 0.08,
        "solvolysis": 0.04,
    },
    "central": {
        "mechanical_recycling": 0.45,
        "pyrolysis": 0.20,
        "solvolysis": 0.10,
    },
    "expansive": {
        "mechanical_recycling": 0.55,
        "pyrolysis": 0.30,
        "solvolysis": 0.15,
    },
}

OPEN_LOOP_MARKET_CASE_PROBABILITIES = {
    "restricted": 0.25,
    "central": 0.50,
    "expansive": 0.25,
}


# ------------------------------------------------------------
# Validate open-loop market cases
# ------------------------------------------------------------

_required_material_pathways = set(
    MATERIAL_RECOVERY_PATHWAYS
)

for (
    _market_case_name,
    _market_case_values,
) in OPEN_LOOP_MARKET_CASES_MC.items():

    _missing_pathways = (
        _required_material_pathways
        - set(_market_case_values)
    )

    _extra_pathways = (
        set(_market_case_values)
        - _required_material_pathways
    )

    if _missing_pathways:
        raise ValueError(
            f"Open-loop market case "
            f"'{_market_case_name}' is missing pathways: "
            f"{sorted(_missing_pathways)}"
        )

    if _extra_pathways:
        raise ValueError(
            f"Open-loop market case "
            f"'{_market_case_name}' contains unknown pathways: "
            f"{sorted(_extra_pathways)}"
        )

    if any(
        not 0.0 <= float(value) <= 1.0
        for value in _market_case_values.values()
    ):
        raise ValueError(
            f"All shares in market case "
            f"'{_market_case_name}' must lie between 0 and 1."
        )

    if (
        sum(
            float(value)
            for value in _market_case_values.values()
        )
        > 1.0 + NUMERICAL_EPSILON
    ):
        raise ValueError(
            f"Shares in market case "
            f"'{_market_case_name}' cannot sum to more than 1."
        )


if not np.isclose(
    sum(
        OPEN_LOOP_MARKET_CASE_PROBABILITIES.values()
    ),
    1.0,
):
    raise ValueError(
        "OPEN_LOOP_MARKET_CASE_PROBABILITIES must sum to 1."
    )

if (
    set(OPEN_LOOP_MARKET_CASE_PROBABILITIES)
    != set(OPEN_LOOP_MARKET_CASES_MC)
):
    raise ValueError(
        "Market-case probabilities and definitions "
        "must contain the same case names."
    )


# ============================================================
# 10.3 Baseline parameter backup
# ============================================================

BASE_PARAMETER_BACKUP = {
    "INITIAL_PROCESSING_COST": copy.deepcopy(
        INITIAL_PROCESSING_COST
    ),

    "RECOVERY_EFFICIENCY": copy.deepcopy(
        RECOVERY_EFFICIENCY
    ),

    "MATERIAL_QUALITY": copy.deepcopy(
        MATERIAL_QUALITY
    ),

    "OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY": copy.deepcopy(
        OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY
    ),

    "RECOVERED_STOCK_DECAY_RATE": copy.deepcopy(
        RECOVERED_STOCK_DECAY_RATE
    ),

    "CLOSED_LOOP_MANDATE_DEMAND_FACTOR": float(
        CLOSED_LOOP_MANDATE_DEMAND_FACTOR
    ),

    "CLOSED_LOOP_OFFTAKE_REALIZATION_RATE": float(
        CLOSED_LOOP_OFFTAKE_REALIZATION_RATE
    ),

    "MATERIAL_SALES_RATIO_SMOOTHING": float(
        MATERIAL_SALES_RATIO_SMOOTHING
    ),

    "INVENTORY_HOLDING_COST_RATE": float(
        INVENTORY_HOLDING_COST_RATE
    ),
}


# ============================================================
# 10.4 Restore deterministic baseline
# ============================================================

def restore_baseline_parameters():
    """
    Restore all parameters modified by the Monte Carlo analysis.

    Dictionary parameters are restored in place so that references
    already used by model functions and classes remain valid.
    """
    global CLOSED_LOOP_MANDATE_DEMAND_FACTOR
    global CLOSED_LOOP_OFFTAKE_REALIZATION_RATE
    global MATERIAL_SALES_RATIO_SMOOTHING
    global INVENTORY_HOLDING_COST_RATE

    dictionary_parameters = [
        "INITIAL_PROCESSING_COST",
        "RECOVERY_EFFICIENCY",
        "MATERIAL_QUALITY",
        "OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY",
        "RECOVERED_STOCK_DECAY_RATE",
    ]

    for parameter_name in dictionary_parameters:

        target_dictionary = globals()[
            parameter_name
        ]

        target_dictionary.clear()

        target_dictionary.update(
            copy.deepcopy(
                BASE_PARAMETER_BACKUP[
                    parameter_name
                ]
            )
        )

    CLOSED_LOOP_MANDATE_DEMAND_FACTOR = float(
        BASE_PARAMETER_BACKUP[
            "CLOSED_LOOP_MANDATE_DEMAND_FACTOR"
        ]
    )

    CLOSED_LOOP_OFFTAKE_REALIZATION_RATE = float(
        BASE_PARAMETER_BACKUP[
            "CLOSED_LOOP_OFFTAKE_REALIZATION_RATE"
        ]
    )

    MATERIAL_SALES_RATIO_SMOOTHING = float(
        BASE_PARAMETER_BACKUP[
            "MATERIAL_SALES_RATIO_SMOOTHING"
        ]
    )

    INVENTORY_HOLDING_COST_RATE = float(
        BASE_PARAMETER_BACKUP[
            "INVENTORY_HOLDING_COST_RATE"
        ]
    )


# ============================================================
# 10.5 Sample uncertain parameters
# ============================================================

def sample_uncertain_parameters(rng):
    """
    Sample one coherent Monte Carlo parameter vector.

    The same sampled vector is applied to all scenarios within one
    run, allowing direct scenario comparison under common external
    uncertainty.

    Parameters
    ----------
    rng : numpy.random.Generator
        Random-number generator used for parameter sampling.

    Returns
    -------
    dict
        Sampled uncertainty vector.
    """

    market_case_names = list(
        OPEN_LOOP_MARKET_CASES_MC
    )

    market_case_probabilities = np.asarray(
        [
            OPEN_LOOP_MARKET_CASE_PROBABILITIES[
                case_name
            ]
            for case_name in market_case_names
        ],
        dtype=float,
    )

    sampled_market_case = str(
        rng.choice(
            market_case_names,
            p=market_case_probabilities,
        )
    )

    return {
        # ----------------------------------------------------
        # Technology performance
        # ----------------------------------------------------

        "solvolysis_processing_cost": float(
            rng.triangular(
                left=0.85,
                mode=1.00,
                right=1.20,
            )
        ),

        "mechanical_processing_cost": float(
            rng.triangular(0.80, 1.00, 1.20)
        ),
        "pyrolysis_processing_cost": float(
            rng.triangular(0.80, 1.00, 1.20)
        ),
        "recovery_processing_cost": float(
            rng.triangular(0.80, 1.00, 1.20)
        ),

        # Fixed conservative model assumption; not sampled.
        "solvolysis_recovery_efficiency": float(
            BASE_PARAMETER_BACKUP[
                "RECOVERY_EFFICIENCY"
            ]["solvolysis"]
        ),

        "solvolysis_material_quality": float(
            rng.triangular(
                left=0.75,
                mode=0.85,
                right=0.95,
            )
        ),

        # ----------------------------------------------------
        # Closed-loop market formation
        # ----------------------------------------------------

        "closed_loop_mandate_demand_factor": float(
            rng.triangular(
                left=0.05,
                mode=0.10,
                right=0.15,
            )
        ),

        "closed_loop_offtake_realization_rate": float(
            rng.triangular(
                left=0.60,
                mode=0.85,
                right=1.00,
            )
        ),

        # ----------------------------------------------------
        # Recycler expectations and inventory economics
        # ----------------------------------------------------

        "material_sales_ratio_smoothing": float(
            rng.triangular(
                left=0.20,
                mode=0.40,
                right=0.60,
            )
        ),

        "inventory_holding_cost_rate": float(
            rng.triangular(
                left=0.04,
                mode=0.08,
                right=0.12,
            )
        ),

        "solvolysis_stock_decay_rate": float(
            rng.triangular(
                left=0.01,
                mode=0.03,
                right=0.06,
            )
        ),

        # ----------------------------------------------------
        # Open-loop market absorption regime
        # ----------------------------------------------------

        "open_loop_market_case": (
            sampled_market_case
        ),
    }


# ============================================================
# 10.6 Apply one sampled parameter vector
# ============================================================

def apply_uncertain_parameters(sampled):
    """
    Apply one Monte Carlo parameter vector to the active model.

    This function must be called only after restoring deterministic
    baseline values and before creating model objects for the run.
    """
    global CLOSED_LOOP_MANDATE_DEMAND_FACTOR
    global CLOSED_LOOP_OFFTAKE_REALIZATION_RATE
    global MATERIAL_SALES_RATIO_SMOOTHING
    global INVENTORY_HOLDING_COST_RATE

    required_sampled_parameters = {
        "solvolysis_processing_cost",
        "mechanical_processing_cost",
        "pyrolysis_processing_cost",
        "recovery_processing_cost",
        "solvolysis_recovery_efficiency",
        "solvolysis_material_quality",
        "closed_loop_mandate_demand_factor",
        "closed_loop_offtake_realization_rate",
        "material_sales_ratio_smoothing",
        "inventory_holding_cost_rate",
        "solvolysis_stock_decay_rate",
        "open_loop_market_case",
    }

    missing_parameters = (
        required_sampled_parameters
        - set(sampled)
    )

    if missing_parameters:
        raise KeyError(
            "The sampled parameter vector is missing: "
            + ", ".join(
                sorted(missing_parameters)
            )
        )

    # --------------------------------------------------------
    # Solvolysis processing cost
    # --------------------------------------------------------

    INITIAL_PROCESSING_COST[
        "solvolysis"
    ] = float(
        BASE_PARAMETER_BACKUP[
            "INITIAL_PROCESSING_COST"
        ]["solvolysis"]
        * sampled[
            "solvolysis_processing_cost"
        ]
    )

    for pathway, sampled_name in {
        "mechanical_recycling": "mechanical_processing_cost",
        "pyrolysis": "pyrolysis_processing_cost",
        "recovery": "recovery_processing_cost",
    }.items():
        INITIAL_PROCESSING_COST[pathway] = float(
            BASE_PARAMETER_BACKUP["INITIAL_PROCESSING_COST"][pathway]
            * sampled[sampled_name]
        )

    # --------------------------------------------------------
    # Solvolysis technical performance
    # --------------------------------------------------------

    RECOVERY_EFFICIENCY[
        "solvolysis"
    ] = float(
        np.clip(
            sampled[
                "solvolysis_recovery_efficiency"
            ],
            0.0,
            1.0,
        )
    )

    MATERIAL_QUALITY[
        "solvolysis"
    ] = float(
        np.clip(
            sampled[
                "solvolysis_material_quality"
            ],
            0.0,
            1.0,
        )
    )

    # --------------------------------------------------------
    # Mandatory closed-loop demand
    # --------------------------------------------------------

    CLOSED_LOOP_MANDATE_DEMAND_FACTOR = float(
        np.clip(
            sampled[
                "closed_loop_mandate_demand_factor"
            ],
            0.0,
            1.0,
        )
    )

    CLOSED_LOOP_OFFTAKE_REALIZATION_RATE = float(
        np.clip(
            sampled[
                "closed_loop_offtake_realization_rate"
            ],
            0.0,
            1.0,
        )
    )

    # --------------------------------------------------------
    # Sales expectation adjustment
    # --------------------------------------------------------

    MATERIAL_SALES_RATIO_SMOOTHING = float(
        np.clip(
            sampled[
                "material_sales_ratio_smoothing"
            ],
            0.0,
            1.0,
        )
    )

    # --------------------------------------------------------
    # Inventory economics
    # --------------------------------------------------------

    INVENTORY_HOLDING_COST_RATE = float(
        max(
            0.0,
            sampled[
                "inventory_holding_cost_rate"
            ],
        )
    )

    RECOVERED_STOCK_DECAY_RATE[
        "solvolysis"
    ] = float(
        np.clip(
            sampled[
                "solvolysis_stock_decay_rate"
            ],
            0.0,
            1.0,
        )
    )

    # --------------------------------------------------------
    # Open-loop market regime
    # --------------------------------------------------------

    market_case = str(
        sampled[
            "open_loop_market_case"
        ]
    )

    if market_case not in OPEN_LOOP_MARKET_CASES_MC:
        raise KeyError(
            f"Unknown open-loop market case: {market_case}"
        )

    OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY.clear()

    OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY.update(
        copy.deepcopy(
            OPEN_LOOP_MARKET_CASES_MC[
                market_case
            ]
        )
    )


# ============================================================
# 10.7 Parameter traceability
# ============================================================

def build_parameter_record(
    sampled,
    run_id,
):
    """
    Create a flat parameter record for one Monte Carlo run.

    Numeric parameters are stored directly. The categorical
    open-loop market regime is stored as text.
    """
    record = {
        "run_id": int(run_id),
    }

    for (
        parameter_name,
        parameter_value,
    ) in sampled.items():

        output_name = (
            f"sampled_{parameter_name}"
        )

        if isinstance(
            parameter_value,
            (str, np.str_),
        ):
            record[output_name] = str(
                parameter_value
            )

        else:
            record[output_name] = float(
                parameter_value
            )

    market_case = str(
        sampled[
            "open_loop_market_case"
        ]
    )

    market_values = (
        OPEN_LOOP_MARKET_CASES_MC[
            market_case
        ]
    )

    for pathway in MATERIAL_RECOVERY_PATHWAYS:
        record[
            f"sampled_open_loop_share_{pathway}"
        ] = float(
            market_values[pathway]
        )

    record["sampled_solvolysis_whole_wtb_output_yield"] = float(
        RECOVERABLE_FEEDSTOCK_FRACTION["solvolysis"]
        * sampled["solvolysis_recovery_efficiency"]
    )

    return record


# ============================================================
# 10.8 Run a scenario family for one parameter draw
# ============================================================

def run_all_scenarios_mc(
    sampled,
    run_id,
    run_seed,
    n_operators=N_OPERATORS,
    n_recyclers=N_RECYCLERS,
    n_manufacturers=N_MANUFACTURERS,
    scenario_configs=None,
    scenario_order=None,
):
    """
    Run one explicitly supplied scenario family under one draw.

    The same random seed and parameter vector are used across all
    scenarios in the run. This common-random-number design improves
    the precision of paired scenario and policy-ablation comparisons.

    When no scenario family is supplied, the function uses the three
    principal transition scenarios, preserving the original analysis.
    """

    if scenario_configs is None:
        scenario_configs = SCENARIOS

    if scenario_order is None:
        scenario_order = SCENARIO_ORDER

    scenario_order = tuple(scenario_order)

    if not scenario_order:
        raise ValueError("scenario_order cannot be empty.")

    scenario_results = []

    parameter_record = build_parameter_record(
        sampled=sampled,
        run_id=run_id,
    )

    for scenario_name in scenario_order:

        if scenario_name not in scenario_configs:
            raise KeyError(
                f"Scenario '{scenario_name}' is not defined in "
                "the supplied scenario family."
            )

        scenario_config = scenario_configs[scenario_name]

        if scenario_config.name != scenario_name:
            raise ValueError(
                f"Scenario key '{scenario_name}' does not match "
                f"ScenarioConfig.name '{scenario_config.name}'."
            )

        model = WTBHybridModel(
            scenario_config=scenario_config,
            start_year=START_YEAR,
            end_year=END_YEAR,
            seed=run_seed,
            n_operators=n_operators,
            n_recyclers=n_recyclers,
            n_manufacturers=n_manufacturers,
        )

        scenario_results_df = model.run()

        scenario_results_df["run_id"] = int(run_id)
        scenario_results_df["run_seed"] = int(run_seed)

        for parameter_name, parameter_value in parameter_record.items():

            if parameter_name == "run_id":
                continue

            scenario_results_df[parameter_name] = parameter_value

        scenario_results.append(scenario_results_df)

    if not scenario_results:
        return pd.DataFrame()

    return pd.concat(
        scenario_results,
        ignore_index=True,
    )


# ============================================================
# 10.9 Main Monte Carlo runner
# ============================================================

def _run_single_monte_carlo_task(task):
    """Execute one isolated draw; safe for sequential or forked use."""

    (
        sampled_parameters,
        run_id,
        run_seed,
        n_operators,
        n_recyclers,
        n_manufacturers,
        scenario_configs,
        scenario_order,
    ) = task

    restore_baseline_parameters()

    try:
        apply_uncertain_parameters(sampled_parameters)

        run_results = run_all_scenarios_mc(
            sampled=sampled_parameters,
            run_id=run_id,
            run_seed=run_seed,
            n_operators=n_operators,
            n_recyclers=n_recyclers,
            n_manufacturers=n_manufacturers,
            scenario_configs=scenario_configs,
            scenario_order=scenario_order,
        )

    finally:
        restore_baseline_parameters()

    return run_results


def _prepare_monte_carlo_tasks(
    n_runs,
    seed,
    n_operators,
    n_recyclers,
    n_manufacturers,
    scenario_configs,
    scenario_order,
):
    """Prepare draws and seeds once, preserving the original sequence."""

    master_seed_sequence = np.random.SeedSequence(seed)
    child_seed_sequences = master_seed_sequence.spawn(n_runs + 1)
    parameter_rng = np.random.default_rng(child_seed_sequences[0])
    run_seed_sequences = child_seed_sequences[1:]

    tasks = []

    for run_id in range(n_runs):
        sampled_parameters = sample_uncertain_parameters(parameter_rng)

        run_seed = int(
            run_seed_sequences[run_id].generate_state(1)[0]
        )

        tasks.append(
            (
                sampled_parameters,
                run_id,
                run_seed,
                n_operators,
                n_recyclers,
                n_manufacturers,
                scenario_configs,
                scenario_order,
            )
        )

    return tasks


def run_monte_carlo(
    n_runs=N_RUNS,
    seed=RANDOM_SEED,
    n_operators=N_OPERATORS,
    n_recyclers=N_RECYCLERS,
    n_manufacturers=N_MANUFACTURERS,
    progress_interval=25,
    scenario_configs=None,
    scenario_order=None,
    n_jobs=MONTE_CARLO_N_JOBS,
):
    """
    Execute Monte Carlo analysis for one scenario family.

    Parameters
    ----------
    n_runs : int
        Number of Monte Carlo parameter draws.

    seed : int
        Master random seed.

    n_operators : int
        Number of operator agents.

    n_recyclers : int
        Initial number of recycler agents.

    n_manufacturers : int
        Number of manufacturer agents.

    progress_interval : int or None
        Number of completed draws between progress messages.

    scenario_configs : mapping or None
        ScenarioConfig objects keyed by scenario name. None uses
        the principal SCENARIOS mapping.

    scenario_order : sequence or None
        Ordered scenario names. None uses SCENARIO_ORDER.

    n_jobs : int
        Independent worker processes. A value of one preserves the
        sequential path. Parallel execution requires the fork method.

    Returns
    -------
    pandas.DataFrame
        Complete scenario-year-run result table in run/scenario/year order.
    """

    n_runs = int(n_runs)
    n_jobs = int(n_jobs)

    if n_runs <= 0:
        raise ValueError("n_runs must be strictly positive.")

    if n_jobs <= 0:
        raise ValueError("n_jobs must be strictly positive.")

    if scenario_configs is None:
        scenario_configs = SCENARIOS

    if scenario_order is None:
        scenario_order = SCENARIO_ORDER

    scenario_order = tuple(scenario_order)

    if not scenario_order:
        raise ValueError("scenario_order cannot be empty.")

    available_start_methods = mp.get_all_start_methods()

    if (
        n_jobs > 1
        and MONTE_CARLO_START_METHOD not in available_start_methods
    ):
        warnings.warn(
            "Process parallelism requires the fork start method; "
            "falling back to sequential execution.",
            UserWarning,
            stacklevel=1,
        )
        n_jobs = 1

    tasks = _prepare_monte_carlo_tasks(
        n_runs=n_runs,
        seed=seed,
        n_operators=n_operators,
        n_recyclers=n_recyclers,
        n_manufacturers=n_manufacturers,
        scenario_configs=scenario_configs,
        scenario_order=scenario_order,
    )

    monte_carlo_results = []

    try:
        if n_jobs == 1:
            iterator = map(_run_single_monte_carlo_task, tasks)

            for completed_runs, run_results in enumerate(iterator, start=1):
                if run_results.empty:
                    raise RuntimeError(
                        f"Monte Carlo run {completed_runs - 1} returned "
                        "an empty result table."
                    )

                monte_carlo_results.append(run_results)

                if (
                    progress_interval is not None
                    and progress_interval > 0
                    and completed_runs % progress_interval == 0
                ):
                    print(
                        f"Completed {completed_runs}/{n_runs} "
                        "Monte Carlo runs"
                    )

        else:
            process_context = mp.get_context(MONTE_CARLO_START_METHOD)

            with process_context.Pool(processes=n_jobs) as pool:
                iterator = pool.imap(
                    _run_single_monte_carlo_task,
                    tasks,
                    chunksize=1,
                )

                for completed_runs, run_results in enumerate(
                    iterator,
                    start=1,
                ):
                    if run_results.empty:
                        raise RuntimeError(
                            f"Monte Carlo run {completed_runs - 1} "
                            "returned an empty result table."
                        )

                    monte_carlo_results.append(run_results)

                    if (
                        progress_interval is not None
                        and progress_interval > 0
                        and completed_runs % progress_interval == 0
                    ):
                        print(
                            f"Completed {completed_runs}/{n_runs} "
                            "Monte Carlo runs"
                        )

    finally:
        restore_baseline_parameters()

    if not monte_carlo_results:
        return pd.DataFrame()

    return pd.concat(
        monte_carlo_results,
        ignore_index=True,
    )


# ============================================================
# 10.10 Execute Monte Carlo
# ============================================================

if N_RUNS < FINAL_ANALYSIS_MINIMUM_RUNS:
    warnings.warn(
        f"N_RUNS={N_RUNS} is below the "
        f"{FINAL_ANALYSIS_MINIMUM_RUNS}-run minimum "
        "recommended for the final analysis. "
        "These results are suitable for debugging only.",
        UserWarning,
        stacklevel=1,
    )

mc_results = run_monte_carlo(
    n_runs=N_RUNS,
    seed=RANDOM_SEED,
    n_operators=N_OPERATORS,
    n_recyclers=N_RECYCLERS,
    n_manufacturers=N_MANUFACTURERS,
    progress_interval=25,
)


# ============================================================
# 10.11 Execution checks
# ============================================================

if mc_results.empty:
    raise RuntimeError(
        "The Monte Carlo result table is empty."
    )

expected_mc_rows = (
    N_RUNS
    * len(SCENARIO_ORDER)
    * len(YEARS)
)

print(
    "Monte Carlo rows:",
    len(mc_results),
)

print(
    "Expected rows:",
    expected_mc_rows,
)

print(
    "Monte Carlo runs:",
    mc_results["run_id"].nunique(),
)

print(
    "Scenarios:",
    mc_results["scenario"].nunique(),
)

print(
    "Years:",
    mc_results["year"].min(),
    "-",
    mc_results["year"].max(),
)

print(
    "Open-loop market cases sampled:",
    mc_results[
        "sampled_open_loop_market_case"
    ].value_counts().to_dict(),
)

assert (
    len(mc_results)
    == expected_mc_rows
)

assert (
    mc_results["run_id"].nunique()
    == N_RUNS
)

assert (
    mc_results["scenario"].nunique()
    == len(SCENARIO_ORDER)
)

assert not mc_results.duplicated(
    subset=[
        "run_id",
        "scenario",
        "year",
    ]
).any()


# ============================================================
# 10.12 Parameter sampling summary
# ============================================================

sampled_parameter_columns = [
    "sampled_solvolysis_processing_cost",
    "sampled_mechanical_processing_cost",
    "sampled_pyrolysis_processing_cost",
    "sampled_recovery_processing_cost",
    "sampled_solvolysis_recovery_efficiency",
    "sampled_solvolysis_material_quality",
    "sampled_closed_loop_mandate_demand_factor",
    "sampled_closed_loop_offtake_realization_rate",
    "sampled_material_sales_ratio_smoothing",
    "sampled_inventory_holding_cost_rate",
    "sampled_solvolysis_stock_decay_rate",
]

parameter_sample_summary = (
    mc_results
    .drop_duplicates(
        subset=["run_id"]
    )
    [
        [
            "run_id",
            "sampled_open_loop_market_case",
            *sampled_parameter_columns,
        ]
    ]
    .describe(
        include="all"
    )
)

display(
    parameter_sample_summary
)


# ============================================================
# 10.13 Monte Carlo convergence check
# ============================================================

convergence_indicators = [
    "solvolysis_treatment_share",
    "closed_loop_wtb_flow_share",
    "untreated_stock_share",
    "unutilized_material_share",
    "aggregate_capacity_shortage",
    "average_processing_cost",
    "virgin_material_displacement",
]

required_convergence_columns = [
    "run_id",
    "scenario",
    "year",
    *convergence_indicators,
]

missing_convergence_columns = [
    column
    for column in required_convergence_columns
    if column not in mc_results.columns
]

if missing_convergence_columns:
    raise KeyError(
        "Missing columns required for convergence checks: "
        + ", ".join(
            missing_convergence_columns
        )
    )


final_year_data = (
    mc_results.loc[
        mc_results["year"] == END_YEAR
    ]
    .copy()
)

if final_year_data.empty:
    raise ValueError(
        f"No final-year data were found at {END_YEAR}."
    )


recent_window_runs = max(
    20,
    N_RUNS // 5,
)

print(
    "Convergence window:",
    recent_window_runs,
    "runs",
)


if recent_window_runs >= N_RUNS:

    print(
        "More Monte Carlo runs are required for "
        "a meaningful convergence assessment."
    )

    convergence_check = pd.DataFrame()

    max_relative_delta = np.nan

    convergence_pass_rate = np.nan

else:

    recent_start_run_id = (
        N_RUNS
        - recent_window_runs
    )

    recent_final_year_data = (
        final_year_data.loc[
            final_year_data["run_id"]
            >= recent_start_run_id
        ]
        .copy()
    )

    if recent_final_year_data.empty:
        raise ValueError(
            "The recent convergence window is empty."
        )

    def summarize_final_year_distribution(
        dataframe,
        indicator_name,
    ):
        grouped = (
            dataframe
            .groupby(
                "scenario"
            )[indicator_name]
        )

        return pd.DataFrame({
            "median": grouped.median(),
            "p05": grouped.quantile(0.05),
            "p95": grouped.quantile(0.95),
        })

    convergence_records = []

    RELATIVE_DELTA_TOLERANCE = 0.05

    ABSOLUTE_DELTA_TOLERANCE = 0.01

    for indicator_name in convergence_indicators:

        full_summary = (
            summarize_final_year_distribution(
                final_year_data,
                indicator_name,
            )
        )

        recent_summary = (
            summarize_final_year_distribution(
                recent_final_year_data,
                indicator_name,
            )
        )

        shared_scenarios = (
            full_summary.index.intersection(
                recent_summary.index
            )
        )

        for scenario_name in shared_scenarios:

            for statistic_name in [
                "median",
                "p05",
                "p95",
            ]:

                full_value = float(
                    full_summary.loc[
                        scenario_name,
                        statistic_name,
                    ]
                )

                recent_value = float(
                    recent_summary.loc[
                        scenario_name,
                        statistic_name,
                    ]
                )

                absolute_delta = abs(
                    recent_value
                    - full_value
                )

                scale = max(
                    abs(full_value),
                    NUMERICAL_EPSILON,
                )

                relative_delta = (
                    absolute_delta
                    / scale
                )

                within_tolerance = (
                    absolute_delta
                    <= (
                        ABSOLUTE_DELTA_TOLERANCE
                        + RELATIVE_DELTA_TOLERANCE
                        * abs(full_value)
                    )
                )

                convergence_records.append({
                    "indicator": indicator_name,
                    "scenario": scenario_name,
                    "statistic": statistic_name,
                    "full_sample_value": full_value,
                    "recent_window_value": recent_value,
                    "absolute_delta": absolute_delta,
                    "relative_delta": relative_delta,
                    "within_tolerance": (
                        within_tolerance
                    ),
                })

    convergence_check = pd.DataFrame(
        convergence_records
    )

    max_relative_delta = float(
        convergence_check[
            "relative_delta"
        ].max()
    )

    convergence_pass_rate = float(
        convergence_check[
            "within_tolerance"
        ].mean()
    )


print(
    "Convergence maximum relative delta:",
    max_relative_delta,
)

print(
    "Convergence pass rate:",
    convergence_pass_rate,
)

display(
    convergence_check.head(20)
)

display(
    mc_results.head()
)


# ============================================================
# 10.14 Final parameter restoration check
# ============================================================

print(
    "Active open-loop market parameters restored:",
    OPEN_LOOP_DEMAND_SHARE_BY_PATHWAY,
)

print(
    "Active mandatory-demand factor restored:",
    CLOSED_LOOP_MANDATE_DEMAND_FACTOR,
)

print(
    "Active material-sales smoothing restored:",
    MATERIAL_SALES_RATIO_SMOOTHING,
)

print(
    "Active inventory holding-cost rate restored:",
    INVENTORY_HOLDING_COST_RATE,
)

print(
    "Monte Carlo analysis completed successfully."
)

In [ ]:
# ============================================================
# 11. Annual WTB flow indicators and consistency checks
# ============================================================
# Revised:
# - accepts either total_capacity_shortage or
#   aggregate_capacity_shortage;
# - validates recovered-material utilization shares;
# - separates treatment-capacity constraints from downstream
#   closed-loop demand gaps.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 11.1 Validate input dataset
# ============================================================

if "mc_results" not in globals():
    raise NameError(
        "mc_results is not defined. Run the Monte Carlo cell first."
    )

if not isinstance(mc_results, pd.DataFrame):
    raise TypeError(
        "mc_results must be a pandas DataFrame."
    )

if mc_results.empty:
    raise ValueError(
        "mc_results cannot be empty."
    )


# ============================================================
# 11.2 Required model outputs
# ============================================================

required_columns = [
    # Identification
    "scenario",
    "run_id",
    "year",

    # Annual WTB stocks and flows
    "decommissioned",
    "desired_total_flow",
    "total_allocatable_flow",
    "opening_untreated_stock",
    "untreated_stock",
    "total_treated",
    "stock_drawdown",
    "net_untreated_stock_increase",
    "net_untreated_stock_increase_share",

    # Annual inflow ratios
    "annual_treatment_to_inflow_ratio",
    "annual_direct_reuse_to_inflow_ratio",
    "annual_open_loop_material_to_inflow_ratio",
    "annual_energy_recovery_to_inflow_ratio",
    "annual_solvolysis_to_inflow_ratio",

    # Treatment composition
    "direct_reuse_share",
    "incumbent_open_loop_material_treatment_share",
    "energy_recovery_treatment_share",
    "solvolysis_treatment_share",

    # Closed-loop outcome and solvolysis physical conversion
    "closed_loop_wtb_flow_share",
    "flow_solvolysis",
    "recovered_generation_solvolysis",
    "solvolysis_recoverable_glass_fibre_input",
    "solvolysis_recovery_efficiency",
    "solvolysis_whole_wtb_output_yield",
    "sampled_solvolysis_recovery_efficiency",
    "sampled_solvolysis_whole_wtb_output_yield",

    # Recovered-material utilization
    "recovered_material_utilization_share",
    "unutilized_material_share",
    "open_loop_utilization_share",
    "closed_loop_utilization_share",
]


missing_columns = [
    column
    for column in required_columns
    if column not in mc_results.columns
]


if missing_columns:
    raise KeyError(
        "Missing columns required for annual WTB-flow analysis: "
        + ", ".join(missing_columns)
    )


# ============================================================
# 11.3 Pathway-flow validation
# ============================================================

pathway_flow_columns = [
    f"flow_{pathway}"
    for pathway in PATHWAYS
]


missing_pathway_columns = [
    column
    for column in pathway_flow_columns
    if column not in mc_results.columns
]


if missing_pathway_columns:
    raise KeyError(
        "Missing pathway-flow columns: "
        + ", ".join(missing_pathway_columns)
    )


# ============================================================
# 11.4 Numeric conversion
# ============================================================

numeric_columns = list(
    dict.fromkeys(
        [
            "year",
            "decommissioned",
            "desired_total_flow",
            "total_allocatable_flow",
            "opening_untreated_stock",
            "untreated_stock",
            "total_treated",
            "stock_drawdown",
            "net_untreated_stock_increase",
            "net_untreated_stock_increase_share",
            "annual_treatment_to_inflow_ratio",
            "annual_direct_reuse_to_inflow_ratio",
            "annual_open_loop_material_to_inflow_ratio",
            "annual_energy_recovery_to_inflow_ratio",
            "annual_solvolysis_to_inflow_ratio",
            "direct_reuse_share",
            "incumbent_open_loop_material_treatment_share",
            "energy_recovery_treatment_share",
            "solvolysis_treatment_share",
            "closed_loop_wtb_flow_share",
            "flow_solvolysis",
            "recovered_generation_solvolysis",
            "solvolysis_recoverable_glass_fibre_input",
            "solvolysis_recovery_efficiency",
            "solvolysis_whole_wtb_output_yield",
            "sampled_solvolysis_recovery_efficiency",
            "sampled_solvolysis_whole_wtb_output_yield",
            "recovered_material_utilization_share",
            "unutilized_material_share",
            "open_loop_utilization_share",
            "closed_loop_utilization_share",
        ]
        + pathway_flow_columns
    )
)


for column in numeric_columns:
    mc_results[column] = pd.to_numeric(
        mc_results[column],
        errors="coerce",
    )


missing_numeric_values = [
    column
    for column in numeric_columns
    if mc_results[column].isna().any()
]


if missing_numeric_values:
    raise ValueError(
        "Missing or non-numeric values found in: "
        + ", ".join(missing_numeric_values)
    )


# ============================================================
# 11.5 Recalculate total treatment from pathway flows
# ============================================================

mc_results[
    "total_treated_from_pathways"
] = (
    mc_results[
        pathway_flow_columns
    ]
    .sum(axis=1)
)


mc_results[
    "pathway_flow_balance_error"
] = (
    mc_results[
        "total_treated_from_pathways"
    ]
    - mc_results[
        "total_treated"
    ]
).abs()


# ============================================================
# 11.6 WTB stock-flow balance
# ============================================================

mc_results[
    "calculated_closing_untreated_stock"
] = (
    mc_results[
        "opening_untreated_stock"
    ]
    + mc_results[
        "decommissioned"
    ]
    - mc_results[
        "total_treated"
    ]
).clip(
    lower=0.0
)


mc_results[
    "wtb_stock_balance_error"
] = (
    mc_results[
        "untreated_stock"
    ]
    - mc_results[
        "calculated_closing_untreated_stock"
    ]
).abs()




# ============================================================
# 11.6a Solvolysis whole-blade-to-glass fibre output balance
# ============================================================

mc_results = mc_results.copy()

mc_results["expected_solvolysis_recoverable_glass_fibre_input"] = (
    mc_results["flow_solvolysis"]
    * RECOVERABLE_FEEDSTOCK_FRACTION["solvolysis"]
)

mc_results["expected_recovered_generation_solvolysis"] = (
    mc_results["expected_solvolysis_recoverable_glass_fibre_input"]
    * mc_results["sampled_solvolysis_recovery_efficiency"]
)

mc_results["solvolysis_glass_fibre_input_balance_error"] = (
    mc_results["solvolysis_recoverable_glass_fibre_input"]
    - mc_results["expected_solvolysis_recoverable_glass_fibre_input"]
).abs()

mc_results["solvolysis_output_balance_error"] = (
    mc_results["recovered_generation_solvolysis"]
    - mc_results["expected_recovered_generation_solvolysis"]
).abs()

mc_results["solvolysis_efficiency_trace_error"] = (
    mc_results["solvolysis_recovery_efficiency"]
    - mc_results["sampled_solvolysis_recovery_efficiency"]
).abs()

mc_results["solvolysis_yield_trace_error"] = (
    mc_results["solvolysis_whole_wtb_output_yield"]
    - mc_results["sampled_solvolysis_whole_wtb_output_yield"]
).abs()


# ============================================================
# 11.7 Inflow-based decomposition
# ============================================================
# These ratios are not required to sum to one when accumulated
# untreated stock is processed during the same year.
# ============================================================

mc_results[
    "annual_reuse_to_inflow_ratio"
] = (
    mc_results[
        "annual_direct_reuse_to_inflow_ratio"
    ]
)


mc_results[
    "annual_incumbent_open_loop_material_to_inflow_ratio"
] = (
    mc_results[
        "annual_open_loop_material_to_inflow_ratio"
    ]
)

mc_results[
    "annual_energy_recovery_to_inflow_ratio_checked"
] = (
    mc_results[
        "annual_energy_recovery_to_inflow_ratio"
    ]
)


mc_results[
    "annual_net_untreated_increase_to_inflow_ratio"
] = (
    mc_results[
        "net_untreated_stock_increase_share"
    ]
)


# ============================================================
# 11.8 Treatment-composition check
# ============================================================
# Direct reuse/repurposing, incumbent open-loop material
# recycling, energy recovery, and solvolysis are mutually
# exclusive components of total treated WTB flow.
# ============================================================

mc_results[
    "treatment_composition_sum"
] = (
    mc_results[
        "direct_reuse_share"
    ]
    + mc_results[
        "incumbent_open_loop_material_treatment_share"
    ]
    + mc_results[
        "energy_recovery_treatment_share"
    ]
    + mc_results[
        "solvolysis_treatment_share"
    ]
)


mc_results[
    "treatment_composition_error"
] = (
    mc_results[
        "treatment_composition_sum"
    ]
    - 1.0
).abs()


# ============================================================
# 11.9 Recovered-material utilization composition
# ============================================================
# Open-loop and closed-loop utilization shares are calculated
# relative to total utilized recovered material and should sum
# to one whenever recovered material is actually used.
# ============================================================

mc_results[
    "material_utilization_composition_sum"
] = (
    mc_results[
        "open_loop_utilization_share"
    ]
    + mc_results[
        "closed_loop_utilization_share"
    ]
)


mc_results[
    "material_utilization_composition_error"
] = (
    mc_results[
        "material_utilization_composition_sum"
    ]
    - 1.0
).abs()


mc_results[
    "recovered_material_balance_share_sum"
] = (
    mc_results[
        "recovered_material_utilization_share"
    ]
    + mc_results[
        "unutilized_material_share"
    ]
)


mc_results[
    "recovered_material_balance_share_error"
] = (
    mc_results[
        "recovered_material_balance_share_sum"
    ]
    - 1.0
).abs()


# ============================================================
# 11.10 Resolve capacity-shortage column name
# ============================================================
# Earlier model versions used aggregate_capacity_shortage.
# Newer versions may use total_capacity_shortage.
# Both are standardized to total_capacity_shortage.
# ============================================================

if "total_capacity_shortage" in mc_results.columns:

    capacity_shortage_source_column = (
        "total_capacity_shortage"
    )

elif "aggregate_capacity_shortage" in mc_results.columns:

    capacity_shortage_source_column = (
        "aggregate_capacity_shortage"
    )

else:

    raise KeyError(
        "Missing capacity-shortage diagnostic. "
        "Expected either 'total_capacity_shortage' or "
        "'aggregate_capacity_shortage'."
    )


mc_results[
    "total_capacity_shortage"
] = pd.to_numeric(
    mc_results[
        capacity_shortage_source_column
    ],
    errors="coerce",
)


if mc_results[
    "total_capacity_shortage"
].isna().any():
    raise ValueError(
        "Missing or non-numeric values found in "
        f"'{capacity_shortage_source_column}'."
    )


print(
    "Capacity-shortage source column used:",
    capacity_shortage_source_column,
)


# ============================================================
# 11.11 Unmet-flow bottleneck decomposition
# ============================================================
# Downstream closed-loop demand gaps are deliberately kept
# separate from WTB treatment-capacity constraints.
# ============================================================

required_bottleneck_columns = [
    "unmet_flow",
    "capacity_allocation_mismatch",
    "initial_rejected_flow",
    "reallocated_flow",
    "unused_capacity_after_rerouting",
]


missing_bottleneck_columns = [
    column
    for column in required_bottleneck_columns
    if column not in mc_results.columns
]


if missing_bottleneck_columns:
    raise KeyError(
        "Missing causal bottleneck diagnostics: "
        + ", ".join(missing_bottleneck_columns)
    )


numeric_bottleneck_columns = (
    required_bottleneck_columns
    + [
        "total_capacity_shortage",
    ]
)


for column in numeric_bottleneck_columns:
    mc_results[column] = pd.to_numeric(
        mc_results[column],
        errors="coerce",
    )


if mc_results[
    numeric_bottleneck_columns
].isna().any().any():
    raise ValueError(
        "Missing or non-numeric values found in "
        "bottleneck diagnostics."
    )


mc_results[
    "unmet_due_total_capacity_shortage"
] = (
    mc_results[
        "total_capacity_shortage"
    ]
)


mc_results[
    "unmet_due_allocation_mismatch"
] = (
    mc_results[
        "capacity_allocation_mismatch"
    ]
)


# ============================================================
# 11.11a Normalized treatment-capacity diagnostics
# ============================================================
# All bottleneck rates use the same denominator: the total annual WTB
# flow submitted to pathway allocation (new inflow plus the share of
# historical backlog reconsidered in that year). This avoids comparing
# absolute shortages across scenarios with different attempted flows.
# ============================================================

mc_results = mc_results.copy()


positive_desired_flow = (
    mc_results["desired_total_flow"]
    > NUMERICAL_EPSILON
)


mc_results[
    "capacity_shortage_rate"
] = np.divide(
    mc_results[
        "total_capacity_shortage"
    ].to_numpy(dtype=float),
    mc_results[
        "desired_total_flow"
    ].to_numpy(dtype=float),
    out=np.zeros(len(mc_results), dtype=float),
    where=positive_desired_flow.to_numpy(),
)


mc_results[
    "capacity_allocation_mismatch_rate"
] = np.divide(
    mc_results[
        "capacity_allocation_mismatch"
    ].to_numpy(dtype=float),
    mc_results[
        "desired_total_flow"
    ].to_numpy(dtype=float),
    out=np.zeros(len(mc_results), dtype=float),
    where=positive_desired_flow.to_numpy(),
)


mc_results[
    "unmet_flow_rate"
] = np.divide(
    mc_results[
        "unmet_flow"
    ].to_numpy(dtype=float),
    mc_results[
        "desired_total_flow"
    ].to_numpy(dtype=float),
    out=np.zeros(len(mc_results), dtype=float),
    where=positive_desired_flow.to_numpy(),
)


mc_results[
    "treatment_coverage_rate"
] = np.divide(
    mc_results[
        "total_treated"
    ].to_numpy(dtype=float),
    mc_results[
        "desired_total_flow"
    ].to_numpy(dtype=float),
    out=np.zeros(len(mc_results), dtype=float),
    where=positive_desired_flow.to_numpy(),
)


mc_results[
    "desired_allocatable_flow_error"
] = (
    mc_results[
        "desired_total_flow"
    ]
    - mc_results[
        "total_allocatable_flow"
    ]
).abs()


mc_results[
    "normalized_bottleneck_balance_error"
] = (
    mc_results[
        "unmet_flow_rate"
    ]
    - mc_results[
        "capacity_shortage_rate"
    ]
    - mc_results[
        "capacity_allocation_mismatch_rate"
    ]
).abs()


mc_results[
    "normalized_treatment_balance_error"
] = np.where(
    positive_desired_flow,
    (
        mc_results[
            "treatment_coverage_rate"
        ]
        + mc_results[
            "unmet_flow_rate"
        ]
        - 1.0
    ).abs(),
    0.0,
)


# Retained only for backward compatibility.
# A downstream material-demand gap is not a causal component
# of untreated WTB flow in this model.
mc_results[
    "unmet_due_closed_loop_demand_insufficiency"
] = 0.0


mc_results[
    "unmet_flow_diagnostic_balance_error"
] = (
    mc_results[
        "unmet_flow"
    ]
    - (
        mc_results[
            "total_capacity_shortage"
        ]
        + mc_results[
            "capacity_allocation_mismatch"
        ]
    )
).abs()


mc_results[
    "rerouting_diagnostic_balance_error"
] = (
    mc_results[
        "initial_rejected_flow"
    ]
    - mc_results[
        "reallocated_flow"
    ]
    - mc_results[
        "unmet_flow"
    ]
).abs()


# ============================================================
# 11.12 Closed-loop demand and utilization diagnostics
# ============================================================

if (
    "effective_closed_loop_demand"
    in mc_results.columns
    and
    "total_closed_loop_utilization"
    in mc_results.columns
):

    mc_results[
        "effective_closed_loop_demand"
    ] = pd.to_numeric(
        mc_results[
            "effective_closed_loop_demand"
        ],
        errors="coerce",
    )

    mc_results[
        "total_closed_loop_utilization"
    ] = pd.to_numeric(
        mc_results[
            "total_closed_loop_utilization"
        ],
        errors="coerce",
    )

    if mc_results[
        [
            "effective_closed_loop_demand",
            "total_closed_loop_utilization",
        ]
    ].isna().any().any():
        raise ValueError(
            "Missing or non-numeric values found in "
            "closed-loop demand diagnostics."
        )

    mc_results[
        "closed_loop_demand_gap"
    ] = (
        mc_results[
            "effective_closed_loop_demand"
        ]
        - mc_results[
            "total_closed_loop_utilization"
        ]
    ).clip(
        lower=0.0
    )

else:

    mc_results[
        "closed_loop_demand_gap"
    ] = np.nan


if (
    "high_quality_recovered_material"
    in mc_results.columns
    and
    "total_closed_loop_utilization"
    in mc_results.columns
):

    mc_results[
        "high_quality_recovered_material"
    ] = pd.to_numeric(
        mc_results[
            "high_quality_recovered_material"
        ],
        errors="coerce",
    )

    if mc_results[
        "high_quality_recovered_material"
    ].isna().any():
        raise ValueError(
            "Missing or non-numeric values found in "
            "high_quality_recovered_material."
        )

    mc_results[
        "high_quality_material_not_used_closed_loop"
    ] = (
        mc_results[
            "high_quality_recovered_material"
        ]
        - mc_results[
            "total_closed_loop_utilization"
        ]
    ).clip(
        lower=0.0
    )

else:

    mc_results[
        "high_quality_material_not_used_closed_loop"
    ] = np.nan


# ============================================================
# 11.13 Numerical tolerances
# ============================================================

FLOW_TOLERANCE = 1e-10

SHARE_TOLERANCE = 1e-8


# ============================================================
# 11.14 Physical non-negativity checks
# ============================================================

nonnegative_flow_columns = (
    [
        "decommissioned",
        "desired_total_flow",
        "total_allocatable_flow",
        "opening_untreated_stock",
        "untreated_stock",
        "total_treated",
        "stock_drawdown",
        "unmet_flow",
        "total_capacity_shortage",
        "capacity_allocation_mismatch",
        "initial_rejected_flow",
        "reallocated_flow",
        "unused_capacity_after_rerouting",
    ]
    + pathway_flow_columns
)


for column in nonnegative_flow_columns:

    minimum_value = float(
        mc_results[
            column
        ].min()
    )

    assert (
        minimum_value
        >= -FLOW_TOLERANCE
    ), (
        f"{column} contains negative physical values. "
        f"Minimum observed value: {minimum_value}"
    )


# ============================================================
# 11.15 Share-bound checks
# ============================================================

share_columns = [
    "direct_reuse_share",
    "incumbent_open_loop_material_treatment_share",
    "energy_recovery_treatment_share",
    "solvolysis_treatment_share",
    "closed_loop_wtb_flow_share",
    "recovered_material_utilization_share",
    "unutilized_material_share",
    "open_loop_utilization_share",
    "closed_loop_utilization_share",
    "capacity_shortage_rate",
    "capacity_allocation_mismatch_rate",
    "unmet_flow_rate",
    "treatment_coverage_rate",
]


for column in share_columns:

    minimum_value = float(
        mc_results[
            column
        ].min()
    )

    maximum_value = float(
        mc_results[
            column
        ].max()
    )

    assert (
        minimum_value
        >= -SHARE_TOLERANCE
    ), (
        f"{column} contains values below zero. "
        f"Minimum observed value: {minimum_value}"
    )

    assert (
        maximum_value
        <= 1.0 + SHARE_TOLERANCE
    ), (
        f"{column} contains values above one. "
        f"Maximum observed value: {maximum_value}"
    )


# ============================================================
# 11.16 Balance assertions
# ============================================================

maximum_pathway_flow_balance_error = float(
    mc_results[
        "pathway_flow_balance_error"
    ].max()
)


assert (
    maximum_pathway_flow_balance_error
    <= FLOW_TOLERANCE
), (
    "Pathway flows do not match total treated flow. "
    f"Maximum error: "
    f"{maximum_pathway_flow_balance_error}"
)


maximum_wtb_stock_balance_error = float(
    mc_results[
        "wtb_stock_balance_error"
    ].max()
)


assert (
    maximum_wtb_stock_balance_error
    <= SHARE_TOLERANCE
), (
    "The untreated WTB stock balance is inconsistent. "
    f"Maximum error: "
    f"{maximum_wtb_stock_balance_error}"
)


treated_rows = (
    mc_results[
        "total_treated"
    ]
    > NUMERICAL_EPSILON
)


if treated_rows.any():

    maximum_treatment_composition_error = float(
        mc_results.loc[
            treated_rows,
            "treatment_composition_error",
        ].max()
    )

    assert (
        maximum_treatment_composition_error
        <= SHARE_TOLERANCE
    ), (
        "Treatment shares do not sum to one. "
        f"Maximum error: "
        f"{maximum_treatment_composition_error}"
    )

else:

    maximum_treatment_composition_error = np.nan


material_use_rows = (
    (
        mc_results[
            "open_loop_utilization_share"
        ]
        + mc_results[
            "closed_loop_utilization_share"
        ]
    )
    > NUMERICAL_EPSILON
)


if material_use_rows.any():

    maximum_material_utilization_composition_error = float(
        mc_results.loc[
            material_use_rows,
            "material_utilization_composition_error",
        ].max()
    )

    assert (
        maximum_material_utilization_composition_error
        <= SHARE_TOLERANCE
    ), (
        "Open-loop and closed-loop utilization shares "
        "do not sum to one. Maximum error: "
        f"{maximum_material_utilization_composition_error}"
    )

else:

    maximum_material_utilization_composition_error = np.nan


available_material_rows = (
    (
        mc_results[
            "recovered_material_utilization_share"
        ]
        + mc_results[
            "unutilized_material_share"
        ]
    )
    > NUMERICAL_EPSILON
)


if available_material_rows.any():

    maximum_recovered_material_balance_share_error = float(
        mc_results.loc[
            available_material_rows,
            "recovered_material_balance_share_error",
        ].max()
    )

    assert (
        maximum_recovered_material_balance_share_error
        <= SHARE_TOLERANCE
    ), (
        "Recovered-material utilization and unutilized "
        "shares do not sum to one. Maximum error: "
        f"{maximum_recovered_material_balance_share_error}"
    )

else:

    maximum_recovered_material_balance_share_error = np.nan




maximum_solvolysis_glass_fibre_input_balance_error = float(
    mc_results["solvolysis_glass_fibre_input_balance_error"].max()
)

maximum_solvolysis_output_balance_error = float(
    mc_results["solvolysis_output_balance_error"].max()
)

maximum_solvolysis_efficiency_trace_error = float(
    mc_results["solvolysis_efficiency_trace_error"].max()
)

maximum_solvolysis_yield_trace_error = float(
    mc_results["solvolysis_yield_trace_error"].max()
)

for balance_name, balance_value in {
    "glass fibre input": maximum_solvolysis_glass_fibre_input_balance_error,
    "recovered output": maximum_solvolysis_output_balance_error,
    "efficiency trace": maximum_solvolysis_efficiency_trace_error,
    "whole-WTB yield trace": maximum_solvolysis_yield_trace_error,
}.items():
    assert balance_value <= FLOW_TOLERANCE, (
        f"Solvolysis {balance_name} balance is inconsistent. "
        f"Maximum error: {balance_value}"
    )


maximum_unmet_flow_diagnostic_error = float(
    mc_results[
        "unmet_flow_diagnostic_balance_error"
    ].max()
)


assert (
    maximum_unmet_flow_diagnostic_error
    <= FLOW_TOLERANCE
), (
    "Unmet-flow bottleneck decomposition is inconsistent. "
    f"Maximum error: "
    f"{maximum_unmet_flow_diagnostic_error}"
)


maximum_rerouting_diagnostic_error = float(
    mc_results[
        "rerouting_diagnostic_balance_error"
    ].max()
)


assert (
    maximum_rerouting_diagnostic_error
    <= FLOW_TOLERANCE
), (
    "Initial rejection, rerouting, and final unmet flow "
    "are inconsistent. "
    f"Maximum error: "
    f"{maximum_rerouting_diagnostic_error}"
)


maximum_desired_allocatable_flow_error = float(
    mc_results[
        "desired_allocatable_flow_error"
    ].max()
)


assert (
    maximum_desired_allocatable_flow_error
    <= FLOW_TOLERANCE
), (
    "Desired and allocatable treatment flows are inconsistent. "
    f"Maximum error: {maximum_desired_allocatable_flow_error}"
)


maximum_normalized_bottleneck_balance_error = float(
    mc_results[
        "normalized_bottleneck_balance_error"
    ].max()
)


assert (
    maximum_normalized_bottleneck_balance_error
    <= SHARE_TOLERANCE
), (
    "Normalized shortage and allocation-mismatch rates do not "
    "sum to the final unmet-flow rate. Maximum error: "
    f"{maximum_normalized_bottleneck_balance_error}"
)


maximum_normalized_treatment_balance_error = float(
    mc_results[
        "normalized_treatment_balance_error"
    ].max()
)


assert (
    maximum_normalized_treatment_balance_error
    <= SHARE_TOLERANCE
), (
    "Treatment coverage and unmet-flow rates do not sum to one. "
    f"Maximum error: {maximum_normalized_treatment_balance_error}"
)


# ============================================================
# 11.17 Closed-loop consistency checks
# ============================================================

if (
    "effective_closed_loop_demand"
    in mc_results.columns
    and
    "total_closed_loop_utilization"
    in mc_results.columns
):

    demand_utilization_difference = (
        mc_results[
            "effective_closed_loop_demand"
        ]
        - mc_results[
            "total_closed_loop_utilization"
        ]
    )

    assert (
        demand_utilization_difference.min()
        >= -FLOW_TOLERANCE
    ), (
        "Closed-loop utilization exceeds effective "
        "closed-loop demand. "
        "Minimum demand-minus-utilization difference: "
        f"{demand_utilization_difference.min()}"
    )


if (
    "high_quality_recovered_material"
    in mc_results.columns
    and
    "total_closed_loop_utilization"
    in mc_results.columns
):

    material_utilization_difference = (
        mc_results[
            "high_quality_recovered_material"
        ]
        - mc_results[
            "total_closed_loop_utilization"
        ]
    )

    assert (
        material_utilization_difference.min()
        >= -FLOW_TOLERANCE
    ), (
        "Closed-loop utilization exceeds available "
        "high-quality material. "
        "Minimum material-minus-utilization difference: "
        f"{material_utilization_difference.min()}"
    )


# ============================================================
# 11.18 SD-ABM capacity synchronization
# ============================================================

if "capacity_sync_error" in mc_results.columns:

    mc_results[
        "capacity_sync_error"
    ] = pd.to_numeric(
        mc_results[
            "capacity_sync_error"
        ],
        errors="coerce",
    )

    if mc_results[
        "capacity_sync_error"
    ].isna().any():
        raise ValueError(
            "Missing or non-numeric values found in "
            "capacity_sync_error."
        )

    if "CAPACITY_SYNC_TOLERANCE" not in globals():
        CAPACITY_SYNC_TOLERANCE = 1e-8

    maximum_capacity_sync_error = float(
        mc_results[
            "capacity_sync_error"
        ].max()
    )

    assert (
        maximum_capacity_sync_error
        <= CAPACITY_SYNC_TOLERANCE
    ), (
        "SD and ABM capacity ledgers are inconsistent. "
        f"Maximum error: "
        f"{maximum_capacity_sync_error}"
    )

else:

    maximum_capacity_sync_error = np.nan


# ============================================================
# 11.19 Inspect annual indicators
# ============================================================

display_columns = [
    "scenario",
    "run_id",
    "year",

    # Annual inflow ratios
    "annual_direct_reuse_to_inflow_ratio",
    "annual_open_loop_material_to_inflow_ratio",
    "annual_energy_recovery_to_inflow_ratio",
    "annual_solvolysis_to_inflow_ratio",
    "annual_treatment_to_inflow_ratio",

    # WTB stocks
    "net_untreated_stock_increase_share",
    "stock_drawdown",

    # Treatment shares
    "direct_reuse_share",
    "incumbent_open_loop_material_treatment_share",
    "energy_recovery_treatment_share",
    "solvolysis_treatment_share",
    "closed_loop_wtb_flow_share",

    # Recovered-material shares
    "recovered_material_utilization_share",
    "unutilized_material_share",
    "open_loop_utilization_share",
    "closed_loop_utilization_share",

    # Balances
    "pathway_flow_balance_error",
    "wtb_stock_balance_error",
    "treatment_composition_error",
    "material_utilization_composition_error",
    "recovered_material_balance_share_error",

    # Capacity bottlenecks
    "total_capacity_shortage",
    "capacity_shortage_rate",
    "capacity_allocation_mismatch",
    "capacity_allocation_mismatch_rate",
    "unmet_flow_rate",
    "treatment_coverage_rate",
    "unused_capacity_after_rerouting",

    # Market diagnostics
    "closed_loop_demand_gap",
    "high_quality_material_not_used_closed_loop",

    # Causal decomposition
    "unmet_due_total_capacity_shortage",
    "unmet_due_allocation_mismatch",
    "unmet_due_closed_loop_demand_insufficiency",
    "unmet_flow_diagnostic_balance_error",
    "rerouting_diagnostic_balance_error",
]


if "capacity_sync_error" in mc_results.columns:
    display_columns.append(
        "capacity_sync_error"
    )


# ============================================================
# 11.20 Validation summary
# ============================================================

print(
    "Maximum pathway-flow balance error:",
    maximum_pathway_flow_balance_error,
)


print(
    "Maximum WTB stock-balance error:",
    maximum_wtb_stock_balance_error,
)


print(
    "Maximum treatment-composition error:",
    maximum_treatment_composition_error,
)


print(
    "Maximum material-utilization composition error:",
    maximum_material_utilization_composition_error,
)


print(
    "Maximum recovered-material balance-share error:",
    maximum_recovered_material_balance_share_error,
)


print(
    "Maximum solvolysis output-balance error:",
    maximum_solvolysis_output_balance_error,
)


print(
    "Maximum unmet-flow diagnostic error:",
    maximum_unmet_flow_diagnostic_error,
)


print(
    "Maximum rerouting diagnostic error:",
    maximum_rerouting_diagnostic_error,
)
print(
    "Maximum normalized bottleneck-balance error:",
    maximum_normalized_bottleneck_balance_error,
)
print(
    "Maximum normalized treatment-balance error:",
    maximum_normalized_treatment_balance_error,
)


if "capacity_sync_error" in mc_results.columns:
    print(
        "Maximum SD-ABM capacity synchronization error:",
        maximum_capacity_sync_error,
    )


display(
    mc_results[
        display_columns
    ].tail()
)


print(
    "Annual WTB-flow and consistency checks "
    "completed successfully."
)

In [ ]:
# ============================================================
# Cell 22. Emergent-configuration classification
# Revised: two emergent configurations only
# Self-contained: builds mc_final from mc_results
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Validate Monte Carlo results
# ------------------------------------------------------------

if "mc_results" not in globals():
    raise NameError(
        "mc_results is not defined. Run the Monte Carlo "
        "simulation cell before executing this cell."
    )


if mc_results.empty:
    raise ValueError(
        "mc_results cannot be empty."
    )


# ------------------------------------------------------------
# Build final-year Monte Carlo dataset
# ------------------------------------------------------------

required_final_columns = [
    "run_id",
    "scenario",
    "year",
]


missing_final_columns = [
    column
    for column in required_final_columns
    if column not in mc_results.columns
]


if missing_final_columns:
    raise KeyError(
        "Missing columns required to build mc_final: "
        + ", ".join(
            missing_final_columns
        )
    )


mc_results[
    "year"
] = pd.to_numeric(
    mc_results[
        "year"
    ],
    errors="coerce",
)


if mc_results[
    "year"
].isna().any():
    raise ValueError(
        "Column 'year' contains missing or non-numeric values."
    )


if "END_YEAR" in globals():
    final_year = int(
        END_YEAR
    )
else:
    final_year = int(
        mc_results[
            "year"
        ].max()
    )

    print(
        "END_YEAR is not defined. "
        f"Using the maximum year in mc_results: {final_year}."
    )


mc_final = (
    mc_results.loc[
        mc_results[
            "year"
        ]
        == final_year
    ]
    .copy()
)


if mc_final.empty:
    available_years = sorted(
        mc_results[
            "year"
        ]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    raise ValueError(
        f"No final-year observations were found for "
        f"year={final_year}. Available years: "
        + ", ".join(
            map(
                str,
                available_years,
            )
        )
    )


print(
    f"Final-year classification dataset created for {final_year}: "
    f"{len(mc_final)} observations."
)


# ------------------------------------------------------------
# Classification thresholds
# ------------------------------------------------------------
# These thresholds operationalize the two theoretical
# configurations and should later be tested through
# threshold-sensitivity analysis.

SOLVOLYSIS_DIFFUSION_THRESHOLD = 0.08

# Minimum proportion of solvolysis-treated WTB material that
# must actually enter closed-loop WTB manufacturing.
CLOSED_LOOP_REALIZATION_RATIO_THRESHOLD = 0.60

# Diagnostic threshold only. This does not define an additional
# emergent configuration.
HIGH_UNTREATED_STOCK_THRESHOLD = 0.20


# ------------------------------------------------------------
# Configuration labels and reporting order
# ------------------------------------------------------------

CONFIGURATION_LABELS = {
    "open_loop_lock_in": (
        "Open-loop lock-in"
    ),
    "closed_loop_market_formation": (
        "Closed-loop market formation"
    ),
}


CONFIGURATION_ORDER = [
    "open_loop_lock_in",
    "closed_loop_market_formation",
]


# ------------------------------------------------------------
# Validate final-year classification inputs
# ------------------------------------------------------------

required_classification_columns = [
    "run_id",
    "scenario",
    "year",
    "solvolysis_treatment_share",
    "closed_loop_wtb_flow_share",
    "untreated_stock_share",
]


missing_columns = [
    column
    for column in required_classification_columns
    if column not in mc_final.columns
]


if missing_columns:
    raise KeyError(
        "Missing columns required for emergent-configuration "
        "classification: "
        + ", ".join(
            missing_columns
        )
    )


# ------------------------------------------------------------
# Prepare final-year classification dataset
# ------------------------------------------------------------

mc_final_classified = mc_final.copy()


numeric_classification_columns = [
    "solvolysis_treatment_share",
    "closed_loop_wtb_flow_share",
    "untreated_stock_share",
]


for column in numeric_classification_columns:
    mc_final_classified[
        column
    ] = pd.to_numeric(
        mc_final_classified[
            column
        ],
        errors="coerce",
    )


if mc_final_classified[
    numeric_classification_columns
].isna().any().any():
    raise ValueError(
        "Missing or non-numeric values were found in the "
        "classification variables."
    )


# ------------------------------------------------------------
# Basic range validation
# ------------------------------------------------------------

RANGE_TOLERANCE = 1e-10


for column in numeric_classification_columns:

    invalid_values = (
        (
            mc_final_classified[
                column
            ]
            < -RANGE_TOLERANCE
        )
        | (
            mc_final_classified[
                column
            ]
            > 1.0 + RANGE_TOLERANCE
        )
    )

    if invalid_values.any():

        minimum_value = float(
            mc_final_classified.loc[
                invalid_values,
                column,
            ].min()
        )

        maximum_value = float(
            mc_final_classified.loc[
                invalid_values,
                column,
            ].max()
        )

        raise ValueError(
            f"Column '{column}' contains values outside "
            f"the valid share range [0, 1]. "
            f"Observed invalid range: "
            f"{minimum_value:.6g} to {maximum_value:.6g}."
        )


# Clip only negligible floating-point deviations.
mc_final_classified[
    numeric_classification_columns
] = mc_final_classified[
    numeric_classification_columns
].clip(
    lower=0.0,
    upper=1.0,
)


# ------------------------------------------------------------
# Closed-loop realization ratio
# ------------------------------------------------------------
# This measures the proportion of solvolysis-treated WTB flow
# that is actually absorbed by closed-loop WTB manufacturing.

solvolysis_positive = (
    mc_final_classified[
        "solvolysis_treatment_share"
    ]
    > 1e-12
)


raw_realization_ratio = np.where(
    solvolysis_positive,
    (
        mc_final_classified[
            "closed_loop_wtb_flow_share"
        ]
        / mc_final_classified[
            "solvolysis_treatment_share"
        ]
    ),
    0.0,
)


invalid_realization_ratio = (
    (
        raw_realization_ratio
        < -RANGE_TOLERANCE
    )
    | (
        raw_realization_ratio
        > 1.0 + RANGE_TOLERANCE
    )
)


if invalid_realization_ratio.any():
    raise ValueError(
        "Closed-loop realization ratio falls outside [0, 1]. "
        "This implies that closed-loop WTB flow exceeds "
        "solvolysis-treated WTB flow in at least one observation."
    )


mc_final_classified[
    "closed_loop_realization_ratio"
] = np.clip(
    raw_realization_ratio,
    0.0,
    1.0,
)


# ------------------------------------------------------------
# Diagnostic flags
# ------------------------------------------------------------
# These variables explain why an observation remains in
# open-loop lock-in. They are not additional configurations.

mc_final_classified[
    "high_untreated_stock_flag"
] = (
    mc_final_classified[
        "untreated_stock_share"
    ]
    >= HIGH_UNTREATED_STOCK_THRESHOLD
)


mc_final_classified[
    "solvolysis_diffusion_flag"
] = (
    mc_final_classified[
        "solvolysis_treatment_share"
    ]
    >= SOLVOLYSIS_DIFFUSION_THRESHOLD
)


mc_final_classified[
    "sufficient_closed_loop_realization_flag"
] = (
    mc_final_classified[
        "closed_loop_realization_ratio"
    ]
    >= CLOSED_LOOP_REALIZATION_RATIO_THRESHOLD
)


mc_final_classified[
    "supply_demand_decoupling_flag"
] = (
    mc_final_classified[
        "solvolysis_diffusion_flag"
    ]
    & ~mc_final_classified[
        "sufficient_closed_loop_realization_flag"
    ]
)


# ------------------------------------------------------------
# Diagnostic: compare solvolysis and closed-loop shares
# ------------------------------------------------------------

share_difference = (
    mc_final_classified[
        "solvolysis_treatment_share"
    ]
    - mc_final_classified[
        "closed_loop_wtb_flow_share"
    ]
).abs()


identical_share_fraction = float(
    np.isclose(
        share_difference,
        0.0,
        atol=1e-10,
    ).mean()
)


print(
    "Fraction of final-year observations where "
    "solvolysis treatment share equals closed-loop WTB "
    f"flow share: {identical_share_fraction:.1%}"
)


if np.isclose(
    identical_share_fraction,
    1.0,
    atol=1e-10,
):
    print(
        "WARNING: The two indicators are identical in every "
        "final-year observation. The supply-demand decoupling "
        "diagnostic cannot emerge unless the upstream model "
        "allows solvolysis-treated material not to enter "
        "closed-loop WTB manufacturing."
    )


# ------------------------------------------------------------
# Emergent-configuration classifier
# ------------------------------------------------------------

def classify_emergent_configuration(row):
    """
    Classify one final-year Monte Carlo outcome into one of the
    two theoretical emergent configurations:

    1. open-loop lock-in;
    2. closed-loop market formation.

    Closed-loop market formation requires both:
    - sufficient solvolysis diffusion; and
    - sufficient realization of solvolysis-derived material in
      closed-loop WTB manufacturing.

    All other outcomes remain classified as open-loop lock-in.
    High untreated stock and supply-demand decoupling are retained
    as diagnostic flags, not separate configurations.
    """

    solvolysis_share = float(
        row[
            "solvolysis_treatment_share"
        ]
    )

    realization_ratio = float(
        row[
            "closed_loop_realization_ratio"
        ]
    )

    if (
        solvolysis_share
        >= SOLVOLYSIS_DIFFUSION_THRESHOLD
        and realization_ratio
        >= CLOSED_LOOP_REALIZATION_RATIO_THRESHOLD
    ):
        return "closed_loop_market_formation"

    return "open_loop_lock_in"


# ------------------------------------------------------------
# Classify every final-year Monte Carlo outcome
# ------------------------------------------------------------

mc_final_classified[
    "configuration"
] = mc_final_classified.apply(
    classify_emergent_configuration,
    axis=1,
)


mc_final_classified[
    "configuration_label"
] = mc_final_classified[
    "configuration"
].map(
    CONFIGURATION_LABELS
)


if mc_final_classified[
    "configuration_label"
].isna().any():
    raise ValueError(
        "At least one configuration could not be mapped "
        "to a readable label."
    )


# ------------------------------------------------------------
# Validate one observation per run and scenario
# ------------------------------------------------------------

duplicate_rows = (
    mc_final_classified
    .duplicated(
        subset=[
            "run_id",
            "scenario",
        ]
    )
)


if duplicate_rows.any():
    raise ValueError(
        "Some run-scenario combinations contain more than "
        "one final-year observation."
    )


# ------------------------------------------------------------
# Validate scenario definitions
# ------------------------------------------------------------

if "SCENARIO_ORDER" not in globals():
    SCENARIO_ORDER = (
        mc_final_classified[
            "scenario"
        ]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    print(
        "SCENARIO_ORDER is not defined. "
        "Using scenario order from mc_final: "
        + ", ".join(
            map(
                str,
                SCENARIO_ORDER,
            )
        )
    )


if "SCENARIO_LABELS" not in globals():
    SCENARIO_LABELS = {
        scenario_name: str(
            scenario_name
        ).replace(
            "_",
            " ",
        ).title()
        for scenario_name in SCENARIO_ORDER
    }

    print(
        "SCENARIO_LABELS is not defined. "
        "Readable labels were generated automatically."
    )


unexpected_scenarios = sorted(
    set(
        mc_final_classified[
            "scenario"
        ]
        .dropna()
        .unique()
    )
    - set(
        SCENARIO_ORDER
    )
)


if unexpected_scenarios:
    raise ValueError(
        "The following scenarios are not included in "
        "SCENARIO_ORDER: "
        + ", ".join(
            unexpected_scenarios
        )
    )


missing_scenarios = [
    scenario_name
    for scenario_name in SCENARIO_ORDER
    if scenario_name
    not in set(
        mc_final_classified[
            "scenario"
        ].unique()
    )
]


if missing_scenarios:
    raise ValueError(
        "No final-year observations were found for: "
        + ", ".join(
            missing_scenarios
        )
    )


# ------------------------------------------------------------
# Count configurations by scenario
# ------------------------------------------------------------

configuration_counts = (
    mc_final_classified
    .groupby(
        [
            "scenario",
            "configuration",
        ],
        observed=True,
    )
    .size()
    .rename(
        "count"
    )
    .reset_index()
)


# ------------------------------------------------------------
# Number of runs available per scenario
# ------------------------------------------------------------

scenario_run_counts = (
    mc_final_classified
    .groupby(
        "scenario",
        observed=True,
    )[
        "run_id"
    ]
    .nunique()
    .rename(
        "n_runs"
    )
    .reset_index()
)


# ------------------------------------------------------------
# Complete all scenario-configuration combinations
# ------------------------------------------------------------

complete_index = pd.MultiIndex.from_product(
    [
        SCENARIO_ORDER,
        CONFIGURATION_ORDER,
    ],
    names=[
        "scenario",
        "configuration",
    ],
)


configuration_probabilities = (
    configuration_counts
    .set_index(
        [
            "scenario",
            "configuration",
        ]
    )
    .reindex(
        complete_index,
        fill_value=0,
    )
    .reset_index()
)


# ------------------------------------------------------------
# Attach run counts and calculate probabilities
# ------------------------------------------------------------

configuration_probabilities = (
    configuration_probabilities
    .merge(
        scenario_run_counts,
        on="scenario",
        how="left",
        validate="many_to_one",
    )
)


if configuration_probabilities[
    "n_runs"
].isna().any():

    missing_run_scenarios = (
        configuration_probabilities.loc[
            configuration_probabilities[
                "n_runs"
            ].isna(),
            "scenario",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    raise ValueError(
        "No Monte Carlo runs were found for the following "
        "scenarios: "
        + ", ".join(
            missing_run_scenarios
        )
    )


if (
    configuration_probabilities[
        "n_runs"
    ]
    <= 0
).any():
    raise ValueError(
        "Every scenario must contain at least one "
        "Monte Carlo run."
    )


configuration_probabilities[
    "probability"
] = (
    configuration_probabilities[
        "count"
    ]
    / configuration_probabilities[
        "n_runs"
    ]
)


# ------------------------------------------------------------
# Add readable labels
# ------------------------------------------------------------

configuration_probabilities[
    "scenario_label"
] = configuration_probabilities[
    "scenario"
].map(
    SCENARIO_LABELS
)


configuration_probabilities[
    "configuration_label"
] = configuration_probabilities[
    "configuration"
].map(
    CONFIGURATION_LABELS
)


if configuration_probabilities[
    [
        "scenario_label",
        "configuration_label",
    ]
].isna().any().any():
    raise ValueError(
        "At least one scenario or configuration label is missing."
    )


# ------------------------------------------------------------
# Preserve reporting order
# ------------------------------------------------------------

configuration_probabilities[
    "scenario"
] = pd.Categorical(
    configuration_probabilities[
        "scenario"
    ],
    categories=SCENARIO_ORDER,
    ordered=True,
)


configuration_probabilities[
    "configuration"
] = pd.Categorical(
    configuration_probabilities[
        "configuration"
    ],
    categories=CONFIGURATION_ORDER,
    ordered=True,
)


configuration_probabilities = (
    configuration_probabilities
    .sort_values(
        [
            "scenario",
            "configuration",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Probability consistency check
# ------------------------------------------------------------

probability_check = (
    configuration_probabilities
    .groupby(
        "scenario",
        observed=True,
    )[
        "probability"
    ]
    .sum()
)


if not np.allclose(
    probability_check.to_numpy(),
    1.0,
    atol=1e-10,
):
    raise ValueError(
        "Configuration probabilities do not sum to one "
        "for every scenario."
    )


# ------------------------------------------------------------
# Diagnostic frequencies by scenario
# ------------------------------------------------------------

diagnostic_probabilities = (
    mc_final_classified
    .groupby(
        "scenario",
        observed=True,
    )
    .agg(
        n_runs=(
            "run_id",
            "nunique",
        ),
        high_untreated_stock_probability=(
            "high_untreated_stock_flag",
            "mean",
        ),
        solvolysis_diffusion_probability=(
            "solvolysis_diffusion_flag",
            "mean",
        ),
        sufficient_realization_probability=(
            "sufficient_closed_loop_realization_flag",
            "mean",
        ),
        supply_demand_decoupling_probability=(
            "supply_demand_decoupling_flag",
            "mean",
        ),
    )
    .reset_index()
)


diagnostic_probabilities[
    "scenario_label"
] = diagnostic_probabilities[
    "scenario"
].map(
    SCENARIO_LABELS
)


diagnostic_probabilities = (
    diagnostic_probabilities
    .set_index(
        "scenario"
    )
    .reindex(
        SCENARIO_ORDER
    )
    .reset_index()
)


# ------------------------------------------------------------
# Additional diagnostic summary
# ------------------------------------------------------------

classification_diagnostics = (
    mc_final_classified
    .groupby(
        "scenario",
        observed=True,
    )
    .agg(
        n_runs=(
            "run_id",
            "nunique",
        ),
        solvolysis_share_median=(
            "solvolysis_treatment_share",
            "median",
        ),
        closed_loop_share_median=(
            "closed_loop_wtb_flow_share",
            "median",
        ),
        realization_ratio_median=(
            "closed_loop_realization_ratio",
            "median",
        ),
        untreated_stock_share_median=(
            "untreated_stock_share",
            "median",
        ),
    )
    .reset_index()
)


classification_diagnostics[
    "scenario_label"
] = classification_diagnostics[
    "scenario"
].map(
    SCENARIO_LABELS
)


classification_diagnostics = (
    classification_diagnostics
    .set_index(
        "scenario"
    )
    .reindex(
        SCENARIO_ORDER
    )
    .reset_index()
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    "Emergent configurations:",
    CONFIGURATION_ORDER,
)


display(
    configuration_probabilities[
        [
            "scenario_label",
            "configuration_label",
            "count",
            "n_runs",
            "probability",
        ]
    ]
)


display(
    diagnostic_probabilities[
        [
            "scenario_label",
            "n_runs",
            "high_untreated_stock_probability",
            "solvolysis_diffusion_probability",
            "sufficient_realization_probability",
            "supply_demand_decoupling_probability",
        ]
    ]
)


display(
    classification_diagnostics[
        [
            "scenario_label",
            "n_runs",
            "solvolysis_share_median",
            "closed_loop_share_median",
            "realization_ratio_median",
            "untreated_stock_share_median",
        ]
    ]
)


In [ ]:
# ============================================================
# Monte Carlo shaded uncertainty plots
# Revised and robust version
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Validate Monte Carlo plotting dataset
# ------------------------------------------------------------

if "mc_results" not in globals():
    raise NameError(
        "mc_results is not defined. Run the Monte Carlo "
        "simulation cell before executing this plotting cell."
    )


if mc_results.empty:
    raise ValueError(
        "mc_results cannot be empty."
    )


required_mc_columns = [
    "run_id",
    "scenario",
    "year",
]


validate_plotting_columns(
    dataframe=mc_results,
    required_columns=required_mc_columns,
)


mc_plot_results = mc_results.copy()


# Ensure year is numeric.
mc_plot_results[
    "year"
] = pd.to_numeric(
    mc_plot_results[
        "year"
    ],
    errors="coerce",
)


if mc_plot_results[
    "year"
].isna().any():
    raise ValueError(
        "Column 'year' contains missing or non-numeric values."
    )


# ------------------------------------------------------------
# Monte Carlo summary over time
# ------------------------------------------------------------

def summarize_mc_over_time(
    mc_results,
    variable,
    lower_quantile=0.05,
    upper_quantile=0.95,
):
    """
    Compute annual Monte Carlo medians and percentile intervals
    by scenario.

    Parameters
    ----------
    mc_results : pandas.DataFrame
        Full Monte Carlo results containing one or more runs.

    variable : str
        Indicator to summarize.

    lower_quantile : float
        Lower uncertainty quantile.

    upper_quantile : float
        Upper uncertainty quantile.

    Returns
    -------
    pandas.DataFrame
        Scenario-year median and uncertainty interval.
    """

    validate_plotting_columns(
        dataframe=mc_results,
        required_columns=[
            "run_id",
            "scenario",
            "year",
            variable,
        ],
    )


    if not (
        0.0
        <= lower_quantile
        < 0.50
        < upper_quantile
        <= 1.0
    ):
        raise ValueError(
            "Quantiles must satisfy "
            "0 <= lower_quantile < 0.5 "
            "< upper_quantile <= 1."
        )


    plotting_data = mc_results[
        [
            "run_id",
            "scenario",
            "year",
            variable,
        ]
    ].copy()


    plotting_data[
        "year"
    ] = pd.to_numeric(
        plotting_data[
            "year"
        ],
        errors="coerce",
    )


    plotting_data[
        variable
    ] = pd.to_numeric(
        plotting_data[
            variable
        ],
        errors="coerce",
    )


    if plotting_data[
        [
            "year",
            variable,
        ]
    ].isna().any().any():
        raise ValueError(
            f"Columns 'year' and '{variable}' must contain "
            "only numeric, non-missing values."
        )


    duplicate_run_years = plotting_data.duplicated(
        subset=[
            "run_id",
            "scenario",
            "year",
        ]
    )


    if duplicate_run_years.any():
        raise ValueError(
            "Some run-scenario-year combinations occur more "
            "than once. Monte Carlo trajectories must contain "
            "one observation per run, scenario, and year."
        )


    summary = (
        plotting_data
        .groupby(
            [
                "scenario",
                "year",
            ],
            as_index=False,
            observed=True,
        )[
            variable
        ]
        .agg(
            median=lambda values: float(
                np.nanmedian(
                    values.to_numpy(
                        dtype=float
                    )
                )
            ),
            lower=lambda values: float(
                np.nanquantile(
                    values.to_numpy(
                        dtype=float
                    ),
                    lower_quantile,
                )
            ),
            upper=lambda values: float(
                np.nanquantile(
                    values.to_numpy(
                        dtype=float
                    ),
                    upper_quantile,
                )
            ),
            n_runs=lambda values: int(
                values.notna().sum()
            ),
        )
    )


    if summary.empty:
        raise ValueError(
            f"No Monte Carlo summary could be calculated "
            f"for '{variable}'."
        )


    invalid_interval = (
        summary[
            "lower"
        ]
        > summary[
            "median"
        ]
    ) | (
        summary[
            "median"
        ]
        > summary[
            "upper"
        ]
    )


    if invalid_interval.any():
        raise ValueError(
            f"Invalid uncertainty interval detected for "
            f"'{variable}': lower <= median <= upper "
            "does not hold."
        )


    return summary


# ------------------------------------------------------------
# Generic Monte Carlo uncertainty-band figure
# ------------------------------------------------------------

def plot_mc_uncertainty_band(
    mc_results,
    variable,
    ylabel,
    filename,
    y_lower=None,
    y_upper=None,
    legend_loc="best",
    show_title=False,
    title=None,
    uncertainty_alpha=0.15,
    lower_quantile=0.05,
    upper_quantile=0.95,
):
    """
    Plot Monte Carlo median trajectories and percentile
    uncertainty intervals across scenarios.

    Parameters
    ----------
    mc_results : pandas.DataFrame
        Full Monte Carlo simulation results.

    variable : str
        Indicator to plot.

    ylabel : str
        Y-axis label.

    filename : str
        Output filename without extension.

    y_lower : float or None
        Optional lower y-axis limit.

    y_upper : float or None
        Optional upper y-axis limit.

    legend_loc : str
        Matplotlib legend location.

    show_title : bool
        Whether to display a title inside the figure.

    title : str or None
        Optional figure title.

    uncertainty_alpha : float
        Transparency of percentile bands.

    lower_quantile : float
        Lower uncertainty quantile.

    upper_quantile : float
        Upper uncertainty quantile.

    Returns
    -------
    matplotlib.figure.Figure
        Generated figure.
    """

    if not (
        0.0
        <= uncertainty_alpha
        <= 1.0
    ):
        raise ValueError(
            "uncertainty_alpha must lie between 0 and 1."
        )


    summary = summarize_mc_over_time(
        mc_results=mc_results,
        variable=variable,
        lower_quantile=lower_quantile,
        upper_quantile=upper_quantile,
    )


    available_scenarios = get_available_scenarios(
        mc_results
    )


    if not available_scenarios:
        raise ValueError(
            "No valid scenarios are available for plotting."
        )


    missing_scenario_labels = [
        scenario
        for scenario in available_scenarios
        if scenario not in SCENARIO_LABELS
    ]


    missing_scenario_colors = [
        scenario
        for scenario in available_scenarios
        if scenario not in SCENARIO_COLORS
    ]


    missing_scenario_linestyles = [
        scenario
        for scenario in available_scenarios
        if scenario not in SCENARIO_LINESTYLES
    ]


    if missing_scenario_labels:
        raise KeyError(
            "Missing scenario labels for: "
            + ", ".join(
                map(
                    str,
                    missing_scenario_labels,
                )
            )
        )


    if missing_scenario_colors:
        raise KeyError(
            "Missing scenario colors for: "
            + ", ".join(
                map(
                    str,
                    missing_scenario_colors,
                )
            )
        )


    if missing_scenario_linestyles:
        raise KeyError(
            "Missing scenario line styles for: "
            + ", ".join(
                map(
                    str,
                    missing_scenario_linestyles,
                )
            )
        )


    fig, ax = plt.subplots(
        figsize=FIGSIZE_ONE_AND_HALF_COLUMN
    )


    plotted_scenarios = 0


    for scenario in available_scenarios:

        scenario_data = (
            summary.loc[
                summary[
                    "scenario"
                ]
                == scenario
            ]
            .sort_values(
                "year"
            )
            .copy()
        )


        if scenario_data.empty:
            continue


        years = scenario_data[
            "year"
        ].to_numpy(
            dtype=float
        )


        median = scenario_data[
            "median"
        ].to_numpy(
            dtype=float
        )


        lower = scenario_data[
            "lower"
        ].to_numpy(
            dtype=float
        )


        upper = scenario_data[
            "upper"
        ].to_numpy(
            dtype=float
        )


        ax.plot(
            years,
            median,
            label=SCENARIO_LABELS[
                scenario
            ],
            color=SCENARIO_COLORS[
                scenario
            ],
            linestyle=SCENARIO_LINESTYLES[
                scenario
            ],
            linewidth=1.9,
        )


        ax.fill_between(
            years,
            lower,
            upper,
            color=SCENARIO_COLORS[
                scenario
            ],
            alpha=uncertainty_alpha,
            linewidth=0.0,
        )


        plotted_scenarios += 1


    if plotted_scenarios == 0:
        plt.close(
            fig
        )

        raise ValueError(
            f"No scenario trajectories could be plotted "
            f"for '{variable}'."
        )


    if show_title and title:
        ax.set_title(
            title
        )


    ax.set_xlabel(
        "Year"
    )


    ax.set_ylabel(
        ylabel
    )


    set_year_axis(
        ax=ax,
        start_year=int(
            summary[
                "year"
            ].min()
        ),
        end_year=int(
            summary[
                "year"
            ].max()
        ),
    )


    if (
        y_lower is not None
        or y_upper is not None
    ):

        current_lower, current_upper = (
            ax.get_ylim()
        )


        final_lower = (
            current_lower
            if y_lower is None
            else float(
                y_lower
            )
        )


        final_upper = (
            current_upper
            if y_upper is None
            else float(
                y_upper
            )
        )


        if final_upper <= final_lower:
            plt.close(
                fig
            )

            raise ValueError(
                "y_upper must be greater than y_lower."
            )


        ax.set_ylim(
            final_lower,
            final_upper,
        )


    clean_axis(
        ax=ax,
        horizontal_grid=True,
        vertical_grid=False,
    )


    ax.legend(
        frameon=False,
        loc=legend_loc,
        handlelength=2.8,
    )


    fig.tight_layout()


    save_figure(
        fig=fig,
        filename=filename,
        formats=(
            "pdf",
            "svg",
            "png",
        ),
        dpi=600,
        close=False,
    )


    return fig


In [ ]:
# ============================================================
# Generate final Monte Carlo uncertainty figures
# Revised and publication-ready version
# ============================================================

import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Validate Monte Carlo results
# ------------------------------------------------------------

if "mc_results" not in globals():
    raise NameError(
        "mc_results is not defined. Run the Monte Carlo "
        "simulation cell before generating the figures."
    )


if mc_results.empty:
    raise ValueError(
        "mc_results cannot be empty."
    )


required_mc_plot_columns = [
    "scenario",
    "year",
    "run_id",
    "solvolysis_treatment_share",
    "closed_loop_wtb_flow_share",
    "untreated_stock_share",
    "average_processing_cost",
    "opening_trl_solvolysis",
    "capacity_shortage_rate",
    "total_treated",
    *PATHWAY_FLOW_COLUMNS.values(),
]


missing_mc_plot_columns = [
    column
    for column in required_mc_plot_columns
    if column not in mc_results.columns
]


if missing_mc_plot_columns:
    raise KeyError(
        "Missing columns required for final Monte Carlo figures: "
        + ", ".join(
            missing_mc_plot_columns
        )
    )


# ------------------------------------------------------------
# Validate required plotting functions
# ------------------------------------------------------------

required_plotting_functions = [
    "plot_solvolysis_diffusion_journal",
    "plot_open_closed_loop_journal",
    "plot_mc_uncertainty_band",
    "plot_processing_cost_journal",
    "plot_solvolysis_maturity_journal",
]


missing_plotting_functions = [
    function_name
    for function_name in required_plotting_functions
    if function_name not in globals()
]


if missing_plotting_functions:
    raise NameError(
        "The following plotting functions are not defined: "
        + ", ".join(
            missing_plotting_functions
        )
        + ". Run the plotting-function cells first."
    )


# ------------------------------------------------------------
# Use complete Monte Carlo result table
# ------------------------------------------------------------

mc_plot_results = mc_results.copy()


# ============================================================
# Figure 2. Median pathway allocation across Monte Carlo runs
# ============================================================

fig_pathway_allocation = (
    plot_pathway_shares_journal(
        results=mc_plot_results,
        y_upper=1.0,
        show_uncertainty=False,
        filename=(
            "figure_2_pathway_allocation"
        ),
    )
)


# ============================================================
# Figure 3. Solvolysis diffusion
# ============================================================

fig_solvolysis_diffusion = (
    plot_solvolysis_diffusion_journal(
        results=mc_plot_results,
        show_uncertainty=True,
        y_upper=None,
        filename=(
            "figure_3_solvolysis_diffusion"
        ),
    )
)


# ============================================================
# Figure 4. Open-loop versus closed-loop utilization
# ============================================================

fig_open_closed_utilization = (
    plot_open_closed_loop_journal(
        results=mc_plot_results,
        show_uncertainty=True,
        filename=(
            "figure_4_open_closed_loop_utilization"
        ),
    )
)


# ============================================================
# Figure 5. Untreated WTB stock share
# ============================================================
# This is the preferred main-text stock indicator.
# The annual net untreated-stock increase is generated below
# as a supplementary flow indicator.

fig_untreated_stock = (
    plot_mc_uncertainty_band(
        mc_results=mc_plot_results,
        variable="untreated_stock_share",
        ylabel="Untreated WTB stock share",
        filename=(
            "figure_5_untreated_stock_share"
        ),
        y_lower=0.0,
        y_upper=1.0,
        legend_loc="upper left",
        lower_quantile=0.05,
        upper_quantile=0.95,
    )
)


# ============================================================
# Figure 6. Economic outcome
# ============================================================

fig_processing_cost = (
    plot_processing_cost_journal(
        results=mc_plot_results,
        show_uncertainty=True,
        filename=(
            "figure_6_processing_cost"
        ),
    )
)


# ============================================================
# Figure 7. Solvolysis technology maturity
# ============================================================

fig_solvolysis_maturity = (
    plot_solvolysis_maturity_journal(
        results=mc_plot_results,
        y_lower=5.0,
        y_upper=9.0,
        show_uncertainty=True,
        filename=(
            "figure_7_solvolysis_maturity"
        ),
    )
)


# ============================================================
# Supplementary Figure S1.
# Annual net untreated-stock increase
# ============================================================

fig_net_untreated_increase = None


if (
    "net_untreated_stock_increase_share"
    in mc_plot_results.columns
):

    fig_net_untreated_increase = (
        plot_mc_uncertainty_band(
            mc_results=mc_plot_results,
            variable=(
                "net_untreated_stock_increase_share"
            ),
            ylabel=(
                "Net untreated-stock increase "
                "relative to annual inflow"
            ),
            filename=(
                "figure_s1_net_untreated_stock_increase"
            ),
            y_lower=0.0,
            y_upper=None,
            legend_loc="upper left",
            lower_quantile=0.05,
            upper_quantile=0.95,
        )
    )

else:

    print(
        "Supplementary Figure S1 was not generated because "
        "'net_untreated_stock_increase_share' is not available."
    )


# ============================================================
# Supplementary Figure S2.
# Normalized aggregate capacity shortage
# ============================================================

fig_capacity_shortage_rate = (
    plot_mc_uncertainty_band(
        mc_results=mc_plot_results,
        variable="capacity_shortage_rate",
        ylabel=(
            "Capacity-shortage rate\n"
            "(share of desired treatment flow)"
        ),
        filename=(
            "figure_s2_capacity_shortage_rate"
        ),
        y_lower=0.0,
        y_upper=None,
        legend_loc="upper left",
        lower_quantile=0.05,
        upper_quantile=0.95,
    )
)


# ------------------------------------------------------------
# Figure-generation summary
# ------------------------------------------------------------

generated_figures = {
    "Figure 2": fig_pathway_allocation,
    "Figure 3": fig_solvolysis_diffusion,
    "Figure 4": fig_open_closed_utilization,
    "Figure 5": fig_untreated_stock,
    "Figure 6": fig_processing_cost,
    "Figure 7": fig_solvolysis_maturity,
}


if fig_net_untreated_increase is not None:
    generated_figures[
        "Figure S1"
    ] = fig_net_untreated_increase

generated_figures[
    "Figure S2"
] = fig_capacity_shortage_rate


print(
    "Generated Monte Carlo figures:"
)


for figure_name in generated_figures:
    print(
        f"- {figure_name}"
    )


# ------------------------------------------------------------
# Show all generated figures
# ------------------------------------------------------------

plt.show()


In [ ]:
# ============================================================
# 12. Monte Carlo summary of final-year outcomes
# Creates mc_summary_2050 from mc_results
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 12.1 Validate Monte Carlo results
# ============================================================

if "mc_results" not in globals():
    raise NameError(
        "mc_results is not defined. Run the Monte Carlo "
        "simulation cell before creating the final-year summary."
    )


if not isinstance(mc_results, pd.DataFrame):
    raise TypeError(
        "mc_results must be a pandas DataFrame."
    )


if mc_results.empty:
    raise ValueError(
        "mc_results cannot be empty."
    )


# ============================================================
# 12.2 Indicators included in the final-year summary
# ============================================================

summary_indicators = [
    "solvolysis_treatment_share",
    "closed_loop_wtb_flow_share",
    "closed_loop_utilization_share",
    "untreated_stock_share",
    "unutilized_material_share",
    "capacity_shortage_rate",
    "capacity_allocation_mismatch_rate",
    "unmet_flow_rate",
    "treatment_coverage_rate",
    "closed_loop_supply_coverage_ratio",
]


required_columns = [
    "scenario",
    "run_id",
    "year",
    *summary_indicators,
]


missing_columns = [
    column
    for column in required_columns
    if column not in mc_results.columns
]


if missing_columns:
    raise KeyError(
        "Missing columns required to create mc_summary_2050: "
        + ", ".join(
            missing_columns
        )
    )


# ============================================================
# 12.3 Prepare numeric data
# ============================================================

summary_source = (
    mc_results[
        required_columns
    ]
    .copy()
)


numeric_columns = [
    "year",
    *summary_indicators,
]


for column in numeric_columns:
    summary_source[column] = pd.to_numeric(
        summary_source[column],
        errors="coerce",
    )


invalid_numeric_columns = [
    column
    for column in numeric_columns
    if summary_source[column].isna().any()
]


if invalid_numeric_columns:
    raise ValueError(
        "Missing or non-numeric values found in: "
        + ", ".join(
            invalid_numeric_columns
        )
    )


# ============================================================
# 12.4 Determine final simulation year
# ============================================================

if "END_YEAR" in globals():
    summary_year = int(
        END_YEAR
    )

else:
    summary_year = int(
        summary_source[
            "year"
        ].max()
    )

    print(
        "END_YEAR is not defined. "
        f"Using the maximum year in mc_results: {summary_year}."
    )


final_year_results = (
    summary_source.loc[
        summary_source[
            "year"
        ]
        == summary_year
    ]
    .copy()
)


if final_year_results.empty:

    available_years = sorted(
        summary_source[
            "year"
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    raise ValueError(
        f"No observations were found for year {summary_year}. "
        "Available years: "
        + ", ".join(
            map(
                str,
                available_years,
            )
        )
    )


# ============================================================
# 12.5 Validate one observation per run and scenario
# ============================================================

duplicate_rows = final_year_results.duplicated(
    subset=[
        "scenario",
        "run_id",
        "year",
    ],
    keep=False,
)


if duplicate_rows.any():

    duplicated_combinations = (
        final_year_results.loc[
            duplicate_rows,
            [
                "scenario",
                "run_id",
                "year",
            ],
        ]
        .drop_duplicates()
    )

    raise ValueError(
        "Some run-scenario combinations contain more than one "
        f"observation for year {summary_year}. "
        f"Duplicated combinations: "
        f"{duplicated_combinations.to_dict(orient='records')}"
    )


# ============================================================
# 12.6 Validate run coverage by scenario
# ============================================================

run_coverage_by_scenario = (
    final_year_results
    .groupby(
        "scenario",
        observed=True,
    )["run_id"]
    .nunique()
)


if run_coverage_by_scenario.empty:
    raise ValueError(
        "No scenario-level Monte Carlo run coverage was found."
    )


if run_coverage_by_scenario.nunique() != 1:
    raise ValueError(
        "The number of Monte Carlo runs is not consistent "
        "across scenarios. Run counts: "
        + str(
            run_coverage_by_scenario.to_dict()
        )
    )


expected_run_count = int(
    run_coverage_by_scenario.iloc[0]
)


if "N_RUNS" in globals():
    if expected_run_count != int(N_RUNS):
        raise ValueError(
            "The final-year Monte Carlo run count does not match N_RUNS. "
            f"Observed: {expected_run_count}; expected: {int(N_RUNS)}."
        )


# ============================================================
# 12.7 Monte Carlo summary function
# ============================================================

def summarize_final_year_indicator(
    dataframe,
    indicator,
    lower_quantile=0.05,
    upper_quantile=0.95,
):
    """
    Summarize one final-year indicator by scenario.

    Returns the median, 5th percentile, 95th percentile,
    minimum, maximum, mean, standard deviation, and valid
    Monte Carlo run count.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Final-year Monte Carlo observations.

    indicator : str
        Indicator to summarize.

    lower_quantile : float
        Lower uncertainty quantile.

    upper_quantile : float
        Upper uncertainty quantile.

    Returns
    -------
    pandas.DataFrame
        Scenario-level summary.
    """

    if indicator not in dataframe.columns:
        raise KeyError(
            f"Indicator '{indicator}' is missing from the dataframe."
        )

    if not (
        0.0
        <= lower_quantile
        < 0.50
        < upper_quantile
        <= 1.0
    ):
        raise ValueError(
            "Quantiles must satisfy "
            "0 <= lower < 0.50 < upper <= 1."
        )

    grouped = (
        dataframe
        .groupby(
            "scenario",
            observed=True,
        )[indicator]
    )

    summary = (
        grouped
        .agg(
            median=lambda values: float(
                np.nanmedian(
                    values.to_numpy(
                        dtype=float
                    )
                )
            ),

            p05=lambda values: float(
                np.nanquantile(
                    values.to_numpy(
                        dtype=float
                    ),
                    lower_quantile,
                )
            ),

            p95=lambda values: float(
                np.nanquantile(
                    values.to_numpy(
                        dtype=float
                    ),
                    upper_quantile,
                )
            ),

            mean=lambda values: float(
                np.nanmean(
                    values.to_numpy(
                        dtype=float
                    )
                )
            ),

            std=lambda values: float(
                np.nanstd(
                    values.to_numpy(
                        dtype=float
                    ),
                    ddof=1,
                )
                if values.notna().sum() > 1
                else 0.0
            ),

            minimum=lambda values: float(
                np.nanmin(
                    values.to_numpy(
                        dtype=float
                    )
                )
            ),

            maximum=lambda values: float(
                np.nanmax(
                    values.to_numpy(
                        dtype=float
                    )
                )
            ),

            n_runs=lambda values: int(
                values.notna().sum()
            ),
        )
        .reset_index()
    )


    invalid_percentile_order = (
        (
            summary["p05"]
            > summary["median"]
        )
        |
        (
            summary["median"]
            > summary["p95"]
        )
    )


    if invalid_percentile_order.any():
        invalid_scenarios = (
            summary.loc[
                invalid_percentile_order,
                "scenario",
            ]
            .astype(str)
            .tolist()
        )

        raise ValueError(
            f"Invalid percentile ordering detected for "
            f"'{indicator}' in scenarios: "
            + ", ".join(
                invalid_scenarios
            )
        )


    invalid_minimum_order = (
        summary["minimum"]
        > summary["p05"]
    )


    invalid_maximum_order = (
        summary["p95"]
        > summary["maximum"]
    )


    if (
        invalid_minimum_order.any()
        or invalid_maximum_order.any()
    ):
        raise ValueError(
            f"Invalid minimum/maximum ordering detected "
            f"for '{indicator}'."
        )


    return summary


# ============================================================
# 12.8 Summarize all indicators
# ============================================================

summary_tables = []


for indicator in summary_indicators:

    indicator_summary = (
        summarize_final_year_indicator(
            dataframe=final_year_results,
            indicator=indicator,
            lower_quantile=0.05,
            upper_quantile=0.95,
        )
    )


    indicator_summary = indicator_summary.rename(
        columns={
            "median": f"{indicator}_median",
            "p05": f"{indicator}_p05",
            "p95": f"{indicator}_p95",
            "mean": f"{indicator}_mean",
            "std": f"{indicator}_std",
            "minimum": f"{indicator}_min",
            "maximum": f"{indicator}_max",
            "n_runs": f"{indicator}_n_runs",
        }
    )


    summary_tables.append(
        indicator_summary
    )


if not summary_tables:
    raise RuntimeError(
        "No Monte Carlo indicator summaries were generated."
    )


# ============================================================
# 12.9 Merge indicator summaries
# ============================================================

mc_summary_2050 = (
    summary_tables[0]
    .copy()
)


for indicator_summary in summary_tables[1:]:

    mc_summary_2050 = (
        mc_summary_2050
        .merge(
            indicator_summary,
            on="scenario",
            how="outer",
            validate="one_to_one",
        )
    )


# ============================================================
# 12.10 Validate consistent run counts
# ============================================================

run_count_columns = [
    f"{indicator}_n_runs"
    for indicator in summary_indicators
]


run_count_matrix = (
    mc_summary_2050[
        run_count_columns
    ]
    .to_numpy(
        dtype=int
    )
)


if not np.all(
    run_count_matrix
    == run_count_matrix[
        :,
        [0],
    ]
):
    raise ValueError(
        "The number of valid Monte Carlo runs is not consistent "
        "across the summarized indicators."
    )


mc_summary_2050[
    "n_runs"
] = (
    mc_summary_2050[
        run_count_columns[0]
    ]
    .astype(int)
)


if not (
    mc_summary_2050["n_runs"]
    == expected_run_count
).all():
    raise ValueError(
        "One or more scenarios have an unexpected number "
        "of valid Monte Carlo runs."
    )


mc_summary_2050 = (
    mc_summary_2050
    .drop(
        columns=run_count_columns
    )
)


# ============================================================
# 12.11 Add readable scenario labels
# ============================================================

if "SCENARIO_LABELS" in globals():

    mc_summary_2050[
        "scenario_label"
    ] = (
        mc_summary_2050[
            "scenario"
        ]
        .astype(str)
        .map(
            SCENARIO_LABELS
        )
    )


    unknown_scenarios = (
        mc_summary_2050.loc[
            mc_summary_2050[
                "scenario_label"
            ].isna(),
            "scenario",
        ]
        .astype(str)
        .unique()
        .tolist()
    )


    if unknown_scenarios:
        raise KeyError(
            "Readable labels are missing for scenarios: "
            + ", ".join(
                unknown_scenarios
            )
        )

else:

    mc_summary_2050[
        "scenario_label"
    ] = (
        mc_summary_2050[
            "scenario"
        ]
        .astype(str)
        .str.replace(
            "_",
            " ",
            regex=False,
        )
        .str.title()
    )


# ============================================================
# 12.12 Preserve scenario order
# ============================================================

available_scenarios = set(
    mc_summary_2050[
        "scenario"
    ]
    .astype(str)
)


if "SCENARIO_ORDER" in globals():

    ordered_available_scenarios = [
        scenario
        for scenario in SCENARIO_ORDER
        if scenario in available_scenarios
    ]


    additional_scenarios = sorted(
        available_scenarios
        - set(
            ordered_available_scenarios
        )
    )


    final_scenario_order = (
        ordered_available_scenarios
        + additional_scenarios
    )

else:

    final_scenario_order = sorted(
        available_scenarios
    )


mc_summary_2050[
    "scenario"
] = pd.Categorical(
    mc_summary_2050[
        "scenario"
    ]
    .astype(str),
    categories=final_scenario_order,
    ordered=True,
)


mc_summary_2050 = (
    mc_summary_2050
    .sort_values(
        "scenario"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 12.13 Add summary metadata
# ============================================================

mc_summary_2050[
    "summary_year"
] = int(
    summary_year
)


mc_summary_2050[
    "lower_quantile"
] = 0.05


mc_summary_2050[
    "upper_quantile"
] = 0.95


# ============================================================
# 12.14 Create compact reporting view
# ============================================================

compact_summary_columns = [
    "scenario",
    "scenario_label",
    "summary_year",
    "n_runs",
]


for indicator in summary_indicators:

    compact_summary_columns.extend([
        f"{indicator}_median",
        f"{indicator}_p05",
        f"{indicator}_p95",
    ])


mc_summary_2050_compact = (
    mc_summary_2050[
        compact_summary_columns
    ]
    .copy()
)


# ============================================================
# 12.15 Validation checks
# ============================================================

expected_scenarios = (
    len(SCENARIO_ORDER)
    if "SCENARIO_ORDER" in globals()
    else mc_summary_2050[
        "scenario"
    ].nunique()
)


if (
    mc_summary_2050[
        "scenario"
    ].nunique()
    != expected_scenarios
):
    raise ValueError(
        "The number of scenarios in mc_summary_2050 "
        "does not match the expected number."
    )


if not (
    mc_summary_2050[
        "summary_year"
    ]
    == summary_year
).all():
    raise AssertionError(
        "The summary year is not consistent across scenarios."
    )


if not (
    mc_summary_2050[
        "n_runs"
    ]
    == expected_run_count
).all():
    raise AssertionError(
        "Monte Carlo run counts are not consistent across scenarios."
    )


# ============================================================
# 12.16 Display results
# ============================================================

print(
    f"mc_summary_2050 created from "
    f"{summary_year} outcomes."
)


print(
    "Monte Carlo runs per scenario:",
    expected_run_count,
)


print(
    "Indicators summarized:",
    summary_indicators,
)


display(
    mc_summary_2050_compact
)

In [ ]:
# ============================================================
# Table 1. Monte Carlo summary of 2050 transition outcomes
# ============================================================

import numpy as np
import pandas as pd


if "mc_summary_2050" not in globals():
    raise NameError(
        "mc_summary_2050 is not defined. Run the 2050 Monte Carlo "
        "summary cell before generating Table 1."
    )


def format_median_interval(
    row,
    median_column,
    lower_column,
    upper_column,
    decimals=1,
    multiplier=1.0,
    suffix="",
):
    """Format a median and its 5th--95th percentile interval."""
    median = float(row[median_column]) * multiplier
    lower = float(row[lower_column]) * multiplier
    upper = float(row[upper_column]) * multiplier
    return (
        f"{median:.{decimals}f}{suffix} "
        f"[{lower:.{decimals}f}{suffix}--"
        f"{upper:.{decimals}f}{suffix}]"
    )


reported_indicators = {
    "solvolysis_treatment_share": "Solvolysis treatment share",
    "closed_loop_utilization_share": (
        "Solvolysis-derived material supplied to new blades "
        "(% of utilized recovered material)"
    ),
    "untreated_stock_share": "Untreated stock share",
    "capacity_shortage_rate": "Capacity-shortage rate",
}


required_columns = ["scenario"]
for indicator in reported_indicators:
    required_columns.extend(
        [
            f"{indicator}_median",
            f"{indicator}_p05",
            f"{indicator}_p95",
        ]
    )


missing_columns = [
    column
    for column in required_columns
    if column not in mc_summary_2050.columns
]
if missing_columns:
    raise KeyError(
        "Missing columns required for Table 1: "
        + ", ".join(missing_columns)
    )


table_1 = mc_summary_2050[required_columns].copy()
table_1["scenario"] = table_1["scenario"].astype(str)
table_1["Scenario"] = table_1["scenario"].map(SCENARIO_LABELS)

if table_1["Scenario"].isna().any():
    raise KeyError("Missing readable scenario labels in Table 1.")

table_1["scenario"] = pd.Categorical(
    table_1["scenario"],
    categories=SCENARIO_ORDER,
    ordered=True,
)
table_1 = table_1.sort_values("scenario").reset_index(drop=True)


for indicator, label in reported_indicators.items():
    lower = f"{indicator}_p05"
    median = f"{indicator}_median"
    upper = f"{indicator}_p95"

    invalid_order = (
        (table_1[lower] > table_1[median])
        | (table_1[median] > table_1[upper])
    )
    if invalid_order.any():
        raise ValueError(
            f"Invalid percentile ordering for '{indicator}'."
        )

    table_1[label] = table_1.apply(
        lambda row, lo=lower, med=median, hi=upper: (
            format_median_interval(
                row=row,
                median_column=med,
                lower_column=lo,
                upper_column=hi,
                decimals=1,
                multiplier=100.0,
                suffix="%",
            )
        ),
        axis=1,
    )


table_1_clean = table_1[
    ["Scenario", *reported_indicators.values()]
].copy()

display(table_1_clean)


In [ ]:
# ============================================================
# Table 2. Median change compared with the post-2025 baseline
# ============================================================

import numpy as np
import pandas as pd


if "mc_summary_2050" not in globals():
    raise NameError(
        "mc_summary_2050 is not defined. Run the 2050 Monte Carlo "
        "summary cell before generating Table 2."
    )


BASELINE_SCENARIO = "post_2025_baseline"
change_indicators = {
    "solvolysis_treatment_share_median": "Solvolysis treatment share",
    "closed_loop_utilization_share_median": (
        "Solvolysis-derived material supplied to new blades "
        "(% of utilized recovered material)"
    ),
    "untreated_stock_share_median": "Untreated stock share",
    "capacity_shortage_rate_median": "Capacity-shortage rate",
}


required_columns = ["scenario", *change_indicators]
missing_columns = [
    column
    for column in required_columns
    if column not in mc_summary_2050.columns
]
if missing_columns:
    raise KeyError(
        "Missing columns required for Table 2: "
        + ", ".join(missing_columns)
    )


summary_table = mc_summary_2050[required_columns].copy()
summary_table["scenario"] = summary_table["scenario"].astype(str)

baseline_rows = summary_table.loc[
    summary_table["scenario"] == BASELINE_SCENARIO
]
if len(baseline_rows) != 1:
    raise ValueError("Exactly one post-2025 baseline row is required.")

baseline = baseline_rows.iloc[0]
relative_rows = []

for scenario in SCENARIO_ORDER:
    if scenario == BASELINE_SCENARIO:
        continue

    scenario_rows = summary_table.loc[
        summary_table["scenario"] == scenario
    ]
    if len(scenario_rows) != 1:
        raise ValueError(
            f"Exactly one summary row is required for '{scenario}'."
        )

    row = scenario_rows.iloc[0]
    record = {
        "scenario": scenario,
        "Scenario": SCENARIO_LABELS[scenario],
    }

    for indicator, label in change_indicators.items():
        record[label] = 100.0 * (
            float(row[indicator]) - float(baseline[indicator])
        )

    relative_rows.append(record)


table_2_numeric = pd.DataFrame(relative_rows)
non_baseline_order = [
    scenario
    for scenario in SCENARIO_ORDER
    if scenario != BASELINE_SCENARIO
]
table_2_numeric["scenario"] = pd.Categorical(
    table_2_numeric["scenario"],
    categories=non_baseline_order,
    ordered=True,
)
table_2_numeric = table_2_numeric.sort_values(
    "scenario"
).reset_index(drop=True)


table_2 = table_2_numeric.copy()
for label in change_indicators.values():
    table_2[label] = table_2[label].map(
        lambda value: f"{float(value):+.1f} pp"
    )

table_2 = table_2[
    ["Scenario", *change_indicators.values()]
].copy()

display(table_2)
